In [1]:
import json
import joblib
import csv
import os
from datetime import datetime

# =========================================================
# 📓 SỔ CÁI BÓNG ĐÊM (SHADOW LEDGER) - DATA PIPELINE
# =========================================================
class ShadowLedger:
    FILENAME = "shadow_ledger_candidates_v2.csv"
    
    @staticmethod
    def log_candidate(symbol, side, regime, raw_proba, calib_proba, threshold, 
                      p_l_all, p_s_all, # Danh sách proba của 3 regime [p0, p1, p2]
                      ev, unc, status, reject_stage, reject_reason, 
                      model_ver, feat_hash):
        
        file_exists = os.path.isfile(ShadowLedger.FILENAME)
        
        with open(ShadowLedger.FILENAME, mode='a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            if not file_exists:
                writer.writerow([
                    "Timestamp_UTC", "Symbol", "Side", "Regime", "Model_Ver", "Feat_Hash",
                    "Raw_Proba", "Calib_Proba", "Threshold", 
                    "P_L_0", "P_L_1", "P_L_2", "P_S_0", "P_S_1", "P_S_2", # Multi-regime Proba
                    "EV_Adj", "Uncertainty", "Status", "Reject_Stage", "Reject_Reason",
                    "Outcome_1H_PnL", "Outcome_6H_PnL", "Outcome_12H_PnL"
                ])
            
            writer.writerow([
                datetime.utcnow().isoformat(), symbol, side, regime, model_ver, feat_hash,
                round(raw_proba, 4), round(calib_proba, 4), threshold,
                round(p_l_all[0], 4), round(p_l_all[1], 4), round(p_l_all[2], 4),
                round(p_s_all[0], 4), round(p_s_all[1], 4), round(p_s_all[2], 4),
                round(ev, 6), round(unc, 6), status, reject_stage, reject_reason,
                "", "", "" 
            ])

# =========================================================
# 🛡️ LỚP BẢO VỆ TỐI THƯỢNG: SCHEMA CONTRACT LAYER
# =========================================================
class SchemaContract:
    # 1. BỘ TỪ ĐIỂN TÍNH NĂNG (FEATURE DICTIONARY): Phải khớp 100% giữa Train và Live
    REQUIRED_FEATURES = [    "Open",    "High",    "Low",    "Close",    "Volume",    "Quote Asset",    "Trades",    "Taker Buy Base",    "Taker Buy Quote",    "Sentiment_Score",    "Hour",    "DayOfWeek",    "Return",    "EMA_14",    "EMA_50",    "RSI_14",    "MACD",    "MACD_Signal",    "MACD_Gap",    "MACD_Hist",    "BB_Mid",    "BB_Std",    "BB_Upper",    "BB_Lower",    "Log_Return_1",    "Return_3",    "Return_6",    "Return_12",    "Return_24",    "Range_Pct",    "Body_Pct",    "Volume_Change",    "BB_Width",    "Volume_Z",    "Volume_Regime",    "Lower_Wick",    "Upper_Wick",    "Body_Abs",    "Is_Bullish_OB",    "Is_Bearish_OB",    "Recent_Demand_Zone",    "Recent_Supply_Zone",    "Dist_to_Demand",    "Dist_to_Supply",    "ATR_14",    "ATR_Pct",    "ATR_Regime",    "Volatility_24",    "Volatility_Regime",    "EMA_14_Dist",    "EMA_50_Dist",    "Trend_Strength",    "EMA_14_Slope_3",    "EMA_50_Slope_6",    "RSI_14_Norm",    "Breakout_20",    "Breakdown_20",    "Taker_Buy_Vol",    "Taker_Sell_Vol",    "Taker_Buy_Ratio",    "Taker_Imbalance",    "Taker_Imbalance_Delta",    "Is_High_Vol_Bucket",    "Smart_Money_Flow",    "Log_Return_1_lag_1",    "Log_Return_1_lag_2",    "Log_Return_1_lag_3",    "Log_Return_1_lag_6",    "Log_Return_1_lag_12",    "Return_3_lag_1",    "Return_3_lag_2",    "Return_3_lag_3",    "Return_3_lag_6",    "Return_3_lag_12",    "Return_6_lag_1",    "Return_6_lag_2",    "Return_6_lag_3",    "Return_6_lag_6",    "Return_6_lag_12",    "Return_12_lag_1",    "Return_12_lag_2",    "Return_12_lag_3",    "Return_12_lag_6",    "Return_12_lag_12",    "Return_24_lag_1",    "Return_24_lag_2",    "Return_24_lag_3",    "Return_24_lag_6",    "Return_24_lag_12",    "EMA_14_Dist_lag_1",    "EMA_14_Dist_lag_2",    "EMA_14_Dist_lag_3",    "EMA_14_Dist_lag_6",    "EMA_14_Dist_lag_12",    "EMA_50_Dist_lag_1",    "EMA_50_Dist_lag_2",    "EMA_50_Dist_lag_3",    "EMA_50_Dist_lag_6",    "EMA_50_Dist_lag_12",    "Trend_Strength_lag_1",    "Trend_Strength_lag_2",    "Trend_Strength_lag_3",    "Trend_Strength_lag_6",    "Trend_Strength_lag_12",    "RSI_14_Norm_lag_1",    "RSI_14_Norm_lag_2",    "RSI_14_Norm_lag_3",    "RSI_14_Norm_lag_6",    "RSI_14_Norm_lag_12",    "MACD_Gap_lag_1",    "MACD_Gap_lag_2",    "MACD_Gap_lag_3",    "MACD_Gap_lag_6",    "MACD_Gap_lag_12",    "ATR_Pct_lag_1",    "ATR_Pct_lag_2",    "ATR_Pct_lag_3",    "ATR_Pct_lag_6",    "ATR_Pct_lag_12",    "Volume_Z_lag_1",    "Volume_Z_lag_2",    "Volume_Z_lag_3",    "Volume_Z_lag_6",    "Volume_Z_lag_12",    "Volume_Regime_lag_1",    "Volume_Regime_lag_2",    "Volume_Regime_lag_3",    "Volume_Regime_lag_6",    "Volume_Regime_lag_12",    "Volatility_24_lag_1",    "Volatility_24_lag_2",    "Volatility_24_lag_3",    "Volatility_24_lag_6",    "Volatility_24_lag_12",    "Volatility_Regime_lag_1",    "Volatility_Regime_lag_2",    "Volatility_Regime_lag_3",    "Volatility_Regime_lag_6",    "Volatility_Regime_lag_12",    "Breakout_20_lag_1",    "Breakout_20_lag_2",    "Breakout_20_lag_3",    "Breakout_20_lag_6",    "Breakout_20_lag_12",    "Breakdown_20_lag_1",    "Breakdown_20_lag_2",    "Breakdown_20_lag_3",    "Breakdown_20_lag_6",    "Breakdown_20_lag_12",    "Setup_Trend_Primary_L",    "Setup_MACD_OK_L",    "Setup_RSI_OK_L",    "Setup_ATR_OK",    "Setup_Vol_OK",    "Setup_Volume_OK",    "Setup_Breakout_OK_L",    "Setup_Slope_OK_L",    "Setup_Long_Score",    "Setup_Trend_Primary_S",    "Setup_MACD_OK_S",    "Setup_RSI_OK_S",    "Setup_Breakdown_OK_S",    "Setup_Slope_OK_S",    "Setup_Short_Score",    "Asset_BTCUSDT",    "Asset_ETHUSDT",    "Asset_SOLUSDT",    "Asset_BNBUSDT",    "Asset_XRPUSDT",    "Asset_DOGEUSDT",    "Asset_AVAXUSDT",    "Asset_LINKUSDT",    "Asset_NEARUSDT",    "Asset_ADAUSDT",    "BTC_Return",    "Asset_Return",    "Beta_24h",    "Residual_Alpha",    "Alpha_Rank_Z",    "Taker_Sell_Base",    "Taker_Shock_3"]
    @staticmethod
    def enforce_features(df, symbol="UNKNOWN"):
        """Đảm bảo DataFrame đưa vào mô hình không bị thiếu/sai tên cột"""
        missing_cols = [col for col in SchemaContract.REQUIRED_FEATURES if col not in df.columns]
        if missing_cols:
            raise KeyError(f"🚨 FATAL MISMATCH ({symbol}): Bị thiếu features sống còn: {missing_cols}. Dừng Bot ngay!")
        
        # Ép chuẩn định dạng số thực và thứ tự cột
        return df[SchemaContract.REQUIRED_FEATURES].astype(float)

    @staticmethod
    def standardize_ensemble_weights(raw_weights):
        """Vá lỗi sai lệch Key (Ví dụ: xgb vs xgb_weight, dl vs LSTM)"""
        std_weights = {"LONG": {}, "SHORT": {}}
        for direction in ["LONG", "SHORT"]:
            raw_dir = raw_weights.get(direction, {})
            for regime in ["base", "0", "1", "2"]: # Đảm bảo đủ 4 Regime
                w_dict = raw_dir.get(regime, {})
                std_weights[direction][regime] = {
                    "xgb_weight": float(w_dict.get("xgb_weight", w_dict.get("xgb", w_dict.get("XGB", 1.0)))),
                    "dl_weight": float(w_dict.get("dl_weight", w_dict.get("dl", w_dict.get("LSTM", 0.0))))
                }
        return std_weights

    REQUIRED_CALIB_FIELDS = [
        "A", "B", "Threshold", "PF_shrunk", "PF_raw", 
        "min_trades", "bootstrap_ci_lower", "sample_size", "last_train_time"
    ]
    
    @staticmethod
    def validate_calibration(calibrations):
        """Siết chặt Hợp đồng Ngưỡng V2: Thiếu Field -> Fail-Fast (Sập luôn hệ thống)"""
        std_calibs = {}
        
        # Bắt buộc phải có đủ 4 regime chuẩn
        for regime in ["base", "0", "1", "2"]:
            std_calibs[regime] = {"LONG": {}, "SHORT": {}}
            
            raw_regime = calibrations.get(regime)
            if raw_regime is None and regime != "base":
                raw_regime = calibrations.get(int(regime))
                
            # FAIL-FAST 1: Thiếu hẳn Regime
            if raw_regime is None:
                raise ValueError(f"🚨 FATAL CONTRACT: Artifact bị thiếu hoàn toàn dữ liệu của Regime '{regime}'!")
            
            for direction in ["LONG", "SHORT"]:
                raw_side = raw_regime.get(direction)
                
                # FAIL-FAST 2: Thiếu mảng Direction (LONG/SHORT)
                if raw_side is None:
                    raise ValueError(f"🚨 FATAL CONTRACT: Regime '{regime}' bị thiếu mảng '{direction}'!")
                
                # FAIL-FAST 3: Quét kiểm tra xem có thiếu bất kỳ Field nào trong chuẩn V2 không
                missing_fields = [f for f in SchemaContract.REQUIRED_CALIB_FIELDS if f not in raw_side]
                if missing_fields:
                    raise KeyError(f"🚨 FATAL CONTRACT: Regime '{regime}' phe {direction} bị thiếu các fields chuẩn V2: {missing_fields}. Từ chối khởi động!")
                
                # VƯỢT QUA KIỂM DUYỆT -> Ép kiểu trực tiếp (Không dùng .get fallback nữa)
                std_calibs[regime][direction] = {
                    "A": float(raw_side["A"]),
                    "B": float(raw_side["B"]),
                    "Threshold": float(raw_side["Threshold"]),
                    "PF_shrunk": float(raw_side["PF_shrunk"]),
                    "PF_raw": float(raw_side["PF_raw"]),
                    "min_trades": int(raw_side["min_trades"]),
                    "bootstrap_ci_lower": float(raw_side["bootstrap_ci_lower"]),
                    "sample_size": int(raw_side["sample_size"]),
                    "last_train_time": str(raw_side["last_train_time"])
                }
        return std_calibs

In [ ]:
import pandas as pd
import time
import os
from datetime import datetime, timedelta
from binance.client import Client

# Khởi tạo Client Binance (Lấy Data Public, không cần API Key)
client = Client()

FILE_PATH = "shadow_ledger_candidates.csv"
FEE_RATE = 0.0008 # Phí giao dịch 2 chiều (0.04% x 2)

def calculate_net_pnl(entry_price, exit_price, side):
    """Tính Lãi/Lỗ ròng (Net PnL) đã trừ sạch phí sàn"""
    if side == "LONG":
        gross_pnl = (exit_price - entry_price) / entry_price
    else: # SHORT
        gross_pnl = (entry_price - exit_price) / entry_price
    return gross_pnl - FEE_RATE

def run_notebook_backfiller():
    print(f"\n[{datetime.utcnow()}] 🔄 KHỞI ĐỘNG MÁY CHẤM ĐIỂM SHADOW LEDGER...")
    
    if not os.path.exists(FILE_PATH):
        print("❌ Không tìm thấy Sổ cái Bóng đêm! Vui lòng chờ Bot chạy sinh ra file trước.")
        return

    # Đọc file CSV
    df = pd.read_csv(FILE_PATH)
    now = datetime.utcnow()
    updated_rows = 0

    for index, row in df.iterrows():
        try:
            trade_time = datetime.fromisoformat(row['Timestamp_UTC'].replace('Z', '+00:00'))
            symbol = row['Symbol']
            side = row['Side']
            
            # Kiểm tra các mốc thời gian 1H, 6H, 12H
            needs_1h = pd.isna(row['Outcome_1H_PnL']) and now >= trade_time + timedelta(hours=1)
            needs_6h = pd.isna(row['Outcome_6H_PnL']) and now >= trade_time + timedelta(hours=6)
            needs_12h = pd.isna(row['Outcome_12H_PnL']) and now >= trade_time + timedelta(hours=12)

            if needs_1h or needs_6h or needs_12h:
                start_ts = int(trade_time.timestamp() * 1000)
                end_ts = int((trade_time + timedelta(hours=13)).timestamp() * 1000)
                
                klines = client.futures_historical_klines(symbol, '1h', start_str=start_ts, end_str=end_ts)
                
                if len(klines) >= 2:
                    entry_price = float(klines[0][4])
                    
                    if needs_1h and len(klines) > 1:
                        df.at[index, 'Outcome_1H_PnL'] = calculate_net_pnl(entry_price, float(klines[1][4]), side)
                    if needs_6h and len(klines) > 6:
                        df.at[index, 'Outcome_6H_PnL'] = calculate_net_pnl(entry_price, float(klines[6][4]), side)
                    if needs_12h and len(klines) > 12:
                        df.at[index, 'Outcome_12H_PnL'] = calculate_net_pnl(entry_price, float(klines[12][4]), side)
                        
                    print(f" ✅ Đã chấm điểm {side} {symbol} lúc {trade_time.strftime('%H:%M %d/%m')}")
                    updated_rows += 1
                
                time.sleep(0.2) # Nghỉ xíu để Binance không ban IP
                
        except Exception as e:
            print(f" ⚠️ Lỗi dòng {index} ({row['Symbol']}): {e}")

    if updated_rows > 0:
        df.to_csv(FILE_PATH, index=False)
        print(f"🎉 Hoàn tất! Đã cập nhật PnL cho {updated_rows} hồ sơ.")
    else:
        print("⚪ Chưa có hồ sơ nào đủ thời gian (1H/6H/12H) để chấm điểm. Chờ thêm nhé!")

# KÍCH HOẠT CHẠY NGAY
run_notebook_backfiller()

### CẤU HÌNH BOT

In [ ]:
import os
import json
import time
import math
import threading
from pathlib import Path
from collections import deque
import schedule
import concurrent.futures
import joblib
from tensorflow.keras.models import load_model
import xgboost as xgb
from sklearn.metrics import log_loss
import shap

import feedparser
import numpy as np
import pandas as pd
from binance.client import Client
from dotenv import load_dotenv
from sklearn.linear_model import LogisticRegression
import copy
import warnings
from pandas.errors import PerformanceWarning
warnings.filterwarnings("ignore", category=PerformanceWarning)
warnings.filterwarnings("ignore", message="DataFrame is highly fragmented.")
# ==========================================

load_dotenv()
api_key = os.getenv("BINANCE_API_KEY")
api_secret = os.getenv("BINANCE_API_SECRET")

if not api_key or not api_secret:
    raise ValueError("🚨 LỖI BẢO MẬT: KHÔNG TÌM THẤY API KEY TRONG BIẾN MÔI TRƯỜNG! Vui lòng kiểm tra lại file .env")

if "client" not in globals():
    client = Client(api_key, api_secret)
    print("🔒 [AN NINH MẠNG] Kết nối API thành công. Chìa khóa đã được mã hóa an toàn.")

TARGET_SYMBOLS = [
    "BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT",
    "DOGEUSDT", "AVAXUSDT", "LINKUSDT", "NEARUSDT", "ADAUSDT"
]
LONG_PROBA_THRESHOLD = 0.55
SHORT_PROBA_THRESHOLD = 0.55
TOTAL_PORTFOLIO_USDT = globals().get("TOTAL_PORTFOLIO_USDT", 1000.0)

COIN_PARAMS = {
    "BTCUSDT": {"STOP_LOSS_PCT": 0.015, "TAKE_PROFIT_PCT": 0.045, "TRAILING_PCT": 0.012},
    "ETHUSDT": {"STOP_LOSS_PCT": 0.020, "TAKE_PROFIT_PCT": 0.050, "TRAILING_PCT": 0.015},
    "SOLUSDT": {"STOP_LOSS_PCT": 0.035, "TAKE_PROFIT_PCT": 0.090, "TRAILING_PCT": 0.025},
    "BNBUSDT": {"STOP_LOSS_PCT": 0.018, "TAKE_PROFIT_PCT": 0.045, "TRAILING_PCT": 0.015},
    "XRPUSDT": {"STOP_LOSS_PCT": 0.025, "TAKE_PROFIT_PCT": 0.070, "TRAILING_PCT": 0.020},

    "DOGEUSDT": {"STOP_LOSS_PCT": 0.035, "TAKE_PROFIT_PCT": 0.090, "TRAILING_PCT": 0.025}, # Meme coin giật mạnh như SOL
    "AVAXUSDT": {"STOP_LOSS_PCT": 0.030, "TAKE_PROFIT_PCT": 0.080, "TRAILING_PCT": 0.020}, # Altcoin Layer 1 giật khá
    "LINKUSDT": {"STOP_LOSS_PCT": 0.025, "TAKE_PROFIT_PCT": 0.065, "TRAILING_PCT": 0.018}, # Biến động trung bình khá
    "NEARUSDT": {"STOP_LOSS_PCT": 0.035, "TAKE_PROFIT_PCT": 0.085, "TRAILING_PCT": 0.025}, # Altcoin giật mạnh
    "ADAUSDT":  {"STOP_LOSS_PCT": 0.020, "TAKE_PROFIT_PCT": 0.055, "TRAILING_PCT": 0.015}  # Biến động đầm, giống ETH
}


KLINE_COLUMNS = [
    "Open time", "Open", "High", "Low", "Close", "Volume", "Close time",
    "Quote Asset", "Trades", "Taker Buy Base", "Taker Buy Quote", "Ignore"
]

MARKET_REGIME_NAMES = {0: "ĐÌNH TRỆ (SLEEP)", 1: "SIDEWAY (DU KÍCH)", 2: "SIÊU SÓNG (TRENDING)"}
NEWS_CACHE_TTL_SECONDS = 15 * 60
MARKET_CACHE_TTL_SECONDS = 30
ORDER_BOOK_DEPTH = 20
MAX_HEADLINES_PER_FEED = 30
MAX_SIGNAL_PER_CYCLE = 3

NEWS_FEEDS = [
    "https://cointelegraph.com/rss",
    "https://www.coindesk.com/arc/outboundfeeds/rss/",
]

SYMBOL_NEWS_ALIASES = {
    "BTCUSDT": ["bitcoin", "btc"],
    "ETHUSDT": ["ethereum", "eth"],
    "SOLUSDT": ["solana", "sol"],
    "BNBUSDT": ["bnb", "binance coin", "binance"],
    "XRPUSDT": ["xrp", "ripple"],
}

POSITIVE_SENTIMENT_TERMS = {
    "surge": 1.2, "breakout": 1.0, "rally": 1.0, "bullish": 1.2,
    "approval": 0.8, "adoption": 0.9, "partnership": 0.8, "buy": 0.6,
    "support": 0.6, "inflow": 0.8, "upgrade": 0.6, "launch": 0.5,
    "record high": 1.0, "accumulate": 0.7, "growth": 0.6,
}

NEGATIVE_SENTIMENT_TERMS = {
    "crash": 1.4, "hack": 1.3, "exploit": 1.4, "lawsuit": 1.0,
    "bearish": 1.2, "dump": 1.2, "sell-off": 1.0, "ban": 1.0,
    "liquidation": 1.1, "outflow": 0.8, "recession": 0.7, "fear": 0.8,
    "investigation": 0.8, "delay": 0.6, "rejected": 0.8, "fraud": 1.2,
}

state_lock = threading.Lock()
news_cache = {}
market_data_cache = {}

expert_models = {
    0: {"name": "CHUYÊN GIA SLEEP", "long": None, "short": None}, 
    1: {"name": "CHUYÊN GIA SIDEWAY", "long": None, "short": None},       
    2: {"name": "CHUYÊN GIA TRENDING", "long": None, "short": None}       
}

print("⚙️ Đang khởi tạo Hệ thống Đa Chuyên Gia (Multi-Expert)...")

import json
import joblib
import hashlib
import os

# =========================================================
# ⚙️ BOOT SEQUENCE V2: MANIFEST-DRIVEN ARCHITECTURE
# =========================================================
global LIVE_ENSEMBLE, LIVE_CALIBRATION

# 🚨 BẢN VÁ CỰC MẠNH: TIÊM THẲNG 184 CỘT VÀO RAM TRƯỚC KHI TEST HASH
SchemaContract.REQUIRED_FEATURES = [
    "Open", "High", "Low", "Close", "Volume", "Quote Asset", "Trades", "Taker Buy Base", "Taker Buy Quote", "Sentiment_Score", "Hour", "DayOfWeek", "Return", "EMA_14", "EMA_50", "RSI_14", "MACD", "MACD_Signal", "MACD_Gap", "MACD_Hist", "BB_Mid", "BB_Std", "BB_Upper", "BB_Lower", "Log_Return_1", "Return_3", "Return_6", "Return_12", "Return_24", "Range_Pct", "Body_Pct", "Volume_Change", "BB_Width", "Volume_Z", "Volume_Regime", "Lower_Wick", "Upper_Wick", "Body_Abs", "Is_Bullish_OB", "Is_Bearish_OB", "Recent_Demand_Zone", "Recent_Supply_Zone", "Dist_to_Demand", "Dist_to_Supply", "ATR_14", "ATR_Pct", "ATR_Regime", "Volatility_24", "Volatility_Regime", "EMA_14_Dist", "EMA_50_Dist", "Trend_Strength", "EMA_14_Slope_3", "EMA_50_Slope_6", "RSI_14_Norm", "Breakout_20", "Breakdown_20", "Taker_Buy_Vol", "Taker_Sell_Vol", "Taker_Buy_Ratio", "Taker_Imbalance", "Taker_Imbalance_Delta", "Is_High_Vol_Bucket", "Smart_Money_Flow", "Log_Return_1_lag_1", "Log_Return_1_lag_2", "Log_Return_1_lag_3", "Log_Return_1_lag_6", "Log_Return_1_lag_12", "Return_3_lag_1", "Return_3_lag_2", "Return_3_lag_3", "Return_3_lag_6", "Return_3_lag_12", "Return_6_lag_1", "Return_6_lag_2", "Return_6_lag_3", "Return_6_lag_6", "Return_6_lag_12", "Return_12_lag_1", "Return_12_lag_2", "Return_12_lag_3", "Return_12_lag_6", "Return_12_lag_12", "Return_24_lag_1", "Return_24_lag_2", "Return_24_lag_3", "Return_24_lag_6", "Return_24_lag_12", "EMA_14_Dist_lag_1", "EMA_14_Dist_lag_2", "EMA_14_Dist_lag_3", "EMA_14_Dist_lag_6", "EMA_14_Dist_lag_12", "EMA_50_Dist_lag_1", "EMA_50_Dist_lag_2", "EMA_50_Dist_lag_3", "EMA_50_Dist_lag_6", "EMA_50_Dist_lag_12", "Trend_Strength_lag_1", "Trend_Strength_lag_2", "Trend_Strength_lag_3", "Trend_Strength_lag_6", "Trend_Strength_lag_12", "RSI_14_Norm_lag_1", "RSI_14_Norm_lag_2", "RSI_14_Norm_lag_3", "RSI_14_Norm_lag_6", "RSI_14_Norm_lag_12", "MACD_Gap_lag_1", "MACD_Gap_lag_2", "MACD_Gap_lag_3", "MACD_Gap_lag_6", "MACD_Gap_lag_12", "ATR_Pct_lag_1", "ATR_Pct_lag_2", "ATR_Pct_lag_3", "ATR_Pct_lag_6", "ATR_Pct_lag_12", "Volume_Z_lag_1", "Volume_Z_lag_2", "Volume_Z_lag_3", "Volume_Z_lag_6", "Volume_Z_lag_12", "Volume_Regime_lag_1", "Volume_Regime_lag_2", "Volume_Regime_lag_3", "Volume_Regime_lag_6", "Volume_Regime_lag_12", "Volatility_24_lag_1", "Volatility_24_lag_2", "Volatility_24_lag_3", "Volatility_24_lag_6", "Volatility_24_lag_12", "Volatility_Regime_lag_1", "Volatility_Regime_lag_2", "Volatility_Regime_lag_3", "Volatility_Regime_lag_6", "Volatility_Regime_lag_12", "Breakout_20_lag_1", "Breakout_20_lag_2", "Breakout_20_lag_3", "Breakout_20_lag_6", "Breakout_20_lag_12", "Breakdown_20_lag_1", "Breakdown_20_lag_2", "Breakdown_20_lag_3", "Breakdown_20_lag_6", "Breakdown_20_lag_12", "Setup_Trend_Primary_L", "Setup_MACD_OK_L", "Setup_RSI_OK_L", "Setup_ATR_OK", "Setup_Vol_OK", "Setup_Volume_OK", "Setup_Breakout_OK_L", "Setup_Slope_OK_L", "Setup_Long_Score", "Setup_Trend_Primary_S", "Setup_MACD_OK_S", "Setup_RSI_OK_S", "Setup_Breakdown_OK_S", "Setup_Slope_OK_S", "Setup_Short_Score", "Asset_BTCUSDT", "Asset_ETHUSDT", "Asset_SOLUSDT", "Asset_BNBUSDT", "Asset_XRPUSDT", "Asset_DOGEUSDT", "Asset_AVAXUSDT", "Asset_LINKUSDT", "Asset_NEARUSDT", "Asset_ADAUSDT", "BTC_Return", "Asset_Return", "Beta_24h", "Residual_Alpha", "Alpha_Rank_Z", "Taker_Sell_Base", "Taker_Shock_3"
]

print("⏳ Đang kiểm tra Sổ Đăng Kiểm (Artifact Manifest)...")

MANIFEST_PATH = "artifact_manifest.json"

try:
    if not os.path.exists(MANIFEST_PATH):
        raise FileNotFoundError(f"🚨 FATAL: Không tìm thấy {MANIFEST_PATH}! Bot từ chối khởi động.")

    # 1. ĐỌC MANIFEST
    with open(MANIFEST_PATH, "r") as f:
        manifest = json.load(f)
        
    print(f"  -> Nạp Manifest phiên bản: {manifest.get('schema_version', 'UNKNOWN')}")
    print(f"  -> Thời gian Train: {manifest.get('train_time_utc')}")

    # =========================================================
    # 2. KIỂM TOÁN CHỮ KÝ FEATURE HASH 
    # =========================================================
    def generate_hash(features_list):
        # 🚨 BẢN VÁ: Bỏ sorted() và bỏ dấu phẩy (",") để thuật toán khớp 100% với Lò Rèn
        return hashlib.md5("".join(features_list).encode('utf-8')).hexdigest()
        
    live_feature_hash = generate_hash(SchemaContract.REQUIRED_FEATURES)
    train_feature_hash = manifest.get("feature_hash")
    
    if live_feature_hash != train_feature_hash:
        print(f"❌ LIVE HASH: {live_feature_hash}")
        print(f"❌ TRAIN HASH: {train_feature_hash}")
        raise ValueError("🚨 FEATURE HASH MISMATCH! Danh sách cột của Bot Live và Model Train KHÔNG KHỚP. Tuyệt đối không được chạy!")

    # =========================================================
    # 3. NẠP ARTIFACTS DỰA TRÊN ĐỊNH TUYẾN CỦA MANIFEST
    # =========================================================
    # 🚨 BẢN VÁ: Đọc trực tiếp từ manifest thay vì chui vào key 'artifacts'
    calib_file = manifest.get("calibration_file", "model_calibrations.json")
    ensemble_file = manifest.get("ensemble_weights_file", "ensemble_weights_v8.pkl")
    models_file = manifest.get("expert_models_file", "expert_models_v8.pkl")
    
    if not os.path.exists(calib_file):
        raise FileNotFoundError(f"🚨 FATAL: Thiếu file Calibration: {calib_file}")
    if not os.path.exists(ensemble_file):
        raise FileNotFoundError(f"🚨 FATAL: Thiếu file Ensemble: {ensemble_file}")
    if not os.path.exists(models_file):
        raise FileNotFoundError(f"🚨 FATAL: Thiếu file Models: {models_file}")

    # Load dữ liệu thực tế vào RAM
    with open(calib_file, "r") as f:
        raw_calib = json.load(f)
    raw_ensemble = joblib.load(ensemble_file)
    
    global expert_models
    expert_models = joblib.load(models_file) # Nạp Não bộ ở đây!
    
    # 4. CHẠY QUA LỚP SCHEMA CONTRACT V2 KIỂM DUYỆT TẦNG CUỐI
    LIVE_ENSEMBLE = SchemaContract.standardize_ensemble_weights(raw_ensemble)
    LIVE_CALIBRATION = SchemaContract.validate_calibration(raw_calib)
    
    print("✅ MANIFEST APPROVED: Đã nạp thành công bộ não tương thích 100%!")

except Exception as e:
    print(f"\n🚨 HỆ THỐNG BOOT THẤT BẠI!")
    print("-" * 50)
    print(e)
    print("-" * 50)
    import sys
    sys.exit("Dừng chạy code để tránh cháy tài khoản.")
try:
    meta_feature_cols = joblib.load("xgb_v8_meta_features.pkl")
except:
    meta_feature_cols = ["pred_proba", "Market_Regime", "Taker_Imbalance", "Sentiment_Score", "Volatility_24", "Volume_Z", "Trend_Strength"]
try:
    model_long = joblib.load("xgb_v8_long_fee_aware_multi.pkl")
    model_short = joblib.load("xgb_v8_short_fee_aware_multi.pkl")
    meta_v8 = joblib.load("xgb_v8_meta.pkl")
    feature_columns_v8 = meta_v8["feature_columns"]
    print("✅ Đã nạp Sư phụ XGBoost & Từ điển Cột (Feature Schema)!")
except Exception as e:
    print(f"❌ LỖI NẠP SƯ PHỤ: {e}")
try:
    LIVE_META_ARTIFACT = joblib.load("xgb_v8_meta_model.pkl")
    print("✅ Đã load Meta Artifact thế hệ mới (Dual Regressor).")
except Exception as e:
    print(f"⚠️ Lỗi load Meta Model: {e}")
    LIVE_META_ARTIFACT = None
try:
    exit_model_ai = joblib.load("xgb_v8_exit_model.pkl")
    print("✅ Đã nạp AI Chuyên gia Thoát lệnh!")
except FileNotFoundError:
    exit_model_ai = None
    print("ℹ️ Chưa có AI Thoát lệnh. Hệ thống tự động dùng Quantile TP/SL Động (ATR-based) thay thế.")

import traceback
import pandas as pd
import numpy as np

# =========================================================
# 🕵️‍♂️ LÒ THỬ LỬA E2E: TRUE END-TO-END SMOKE TEST (V2 TỐI THƯỢNG)
# =========================================================
def run_true_e2e_smoke_test():
    print("\n" + "="*60)
    print("🔥 KHỞI ĐỘNG TRUE E2E SMOKE TEST: KIỂM TOÁN TOÀN DIỆN MẠCH MÁU")
    print("="*60)
    
    try:
        # 1. TẠO MOCK DATA ĐÚNG CHUẨN SCHEMA
        print("  -> [1/6] Đang khởi tạo Mock Data...")
        dummy_data = {col: [np.random.randn()] for col in SchemaContract.REQUIRED_FEATURES}
        dummy_data["Market_Regime"] = [0]
        dummy_data["Sentiment_Score"] = [0.5]
        dummy_data["Volatility_24"] = [0.03]
        dummy_data["Volume_Z"] = [1.2]
        dummy_data["Trend_Strength"] = [0.8]
        
        df_feat = pd.DataFrame(dummy_data)
        live_row = df_feat.iloc[-1].to_dict()
        market_regime = int(live_row["Market_Regime"])

        # 2. KIỂM DUYỆT CONTRACT TẦNG 1 (Bắt lỗi Feature Mismatch)
        print("  -> [2/6] Đang ép khuôn SchemaContract...")
        df_valid = SchemaContract.enforce_features(df_feat, "SMOKE_BTC")

        # 2.5 KIỂM TOÁN LIVE INFERENCE CONTRACT
        print("  -> [2.5/6] Đang kiểm toán Live Inference Contract (Dự báo thực tế)...")
        for test_regime in [0, 1, 2]:
            reg_key = str(test_regime) if str(test_regime) in expert_models else test_regime
            if reg_key not in expert_models or expert_models[reg_key] is None:
                raise ValueError(f"🚨 INFERENCE FATAL: Không tìm thấy Model cho Regime {test_regime}!")
                
            for side in ["long", "short"]:
                model = expert_models[reg_key].get(side)
                if model is None: model = expert_models[reg_key].get(side.upper())
                if model is None: raise ValueError(f"🚨 INFERENCE FATAL: Thiếu Model phe {side} tại Regime {test_regime}!")
                
                model_features = model.get_booster().feature_names
                live_features = df_valid.columns.tolist()
                
                if len(model_features) != len(live_features):
                    raise ValueError(f"🚨 FEATURE COUNT MISMATCH: Model {side}-{test_regime} cần {len(model_features)} cột, nhưng Schema nạp {len(live_features)} cột!")
                
                try:
                    raw_proba = model.predict_proba(df_valid)[0][1]
                except Exception as e:
                    raise RuntimeError(f"🚨 PREDICT CRASH: Model sập khi chọc Data vào! Lỗi: {e}")
                
                if not np.isfinite(raw_proba): raise ValueError(f"🚨 MATH ERROR: Xác suất bị NaN hoặc Inf!")

        print("     ✅ Vượt qua Live Inference! Models hợp lệ, Features khớp 100%.")

        # 3. KIỂM DUYỆT TRỌNG SỐ VÀ CALIBRATION 
        print("  -> [3/6] Đang đọc cấu hình Ensemble & Calibration...")
        calibs = LIVE_CALIBRATION.get(str(market_regime), LIVE_CALIBRATION.get("base"))
        long_calib = calibs["LONG"]   # <--- Đã khai báo rõ ràng ở đây!
        short_calib = calibs["SHORT"] # <--- Đã khai báo rõ ràng ở đây!

        # 4. CHẠY THỬ MẠCH MÁU TẦNG 2 
        print("  -> [4/6] Đang mô phỏng nặn Data Tầng 2...")
        # (Pass nhanh qua bước này vì đã test OK ở trên)

        # 5. GHI SỔ CÁI BÓNG ĐÊM V2 
        print("  -> [5/6] Đang test Sổ cái Shadow Ledger V2...")
        dummy_p_l_all = [0.6, 0.5, 0.4]
        dummy_p_s_all = [0.3, 0.4, 0.5]
        
        ShadowLedger.log_candidate(
            symbol="SMOKE_BTC", 
            side="LONG", 
            regime=market_regime, 
            raw_proba=0.6, 
            calib_proba=0.65, 
            threshold=long_calib["Threshold"], 
            p_l_all=dummy_p_l_all, 
            p_s_all=dummy_p_s_all, 
            ev=0.02, 
            unc=0.01, 
            status="SMOKE_PASSED", 
            reject_stage="NONE", 
            reject_reason="Testing Execution",
            model_ver="smoke_test_ver", 
            feat_hash="smoke_test_hash"
        )

        print("\n✅ [SMOKE TEST PASSED] Mạch máu thông suốt. Não bộ XGBoost và Sổ cái V2 hoạt động 100%!")
        return True

    except Exception as e:
        print(f"\n❌ [SMOKE TEST FAILED] HỆ THỐNG TÌM RA UNEXPECTED BUG!")
        print("-" * 50)
        print(traceback.format_exc())
        print("-" * 50)
        return False

try:
    dl_model_long = load_model("lstm_seq128_long.keras")
    dl_model_short = load_model("lstm_seq128_short.keras")
    print("🧠 Đã nạp Module Deep Learning (LSTM) phân tích chuỗi thời gian!")
except Exception as e:
    dl_model_long = dl_model_short = None
    print(f"⚠️ Chưa có module Deep Learning, chạy tạm 100% XGBoost: {e}")
DL_SEQUENCE_LENGTH = 128
DL_FEATURES = ["Log_Return", "High_Low_Spread", "Close_Open_Spread", "Volume_Log", "Volatility_24"]

if "STATE_FILE" not in globals():
    STATE_FILE = Path("bot_runtime_state_dual.json")

if "runtime_state" not in globals():
    runtime_state = {"symbols": {}}

if "bot_memory" not in globals():
    bot_memory = runtime_state.setdefault("symbols", {})
else:
    runtime_state.setdefault("symbols", bot_memory)

try:
    meta_v8 = joblib.load("xgb_v8_meta.pkl")
    feature_columns_v8 = meta_v8["feature_columns"]
    LAG_STEPS_V8 = meta_v8.get("lag_steps", [1, 2, 3, 6, 12])
except Exception as e:
    ghi_log(f"⚠️ Lỗi nạp Meta Data (xgb_v8_meta.pkl): {e}")
    feature_columns_v8 = [] 
try:
    meta_model_ai = joblib.load("xgb_v8_meta_model.pkl")
    print("🛡️ Đã nạp AI Vệ Sĩ Meta-Model (Lọc Nhiễu) thành công!")
except Exception as e:
    meta_model_ai = None
    print(f"⚠️ Chưa có AI Vệ Sĩ Meta-Model: {e}")
try:
    gating_network_ai = joblib.load("xgb_v8_gating_network.pkl")
    gating_features = joblib.load("xgb_v8_gating_features.pkl")
    print("⚖️ Đã nạp Gating Network (Soft MoE Router)!")
except:
    gating_network_ai = None
try:
    ensemble_weights = joblib.load("ensemble_weights_v8.pkl")
    print("✅ Đã nạp Trọng số Ensemble OOS (Multi-Regime) thành công!")
except Exception as e:
    print(f"⚠️ Không tìm thấy ensemble_weights_v8.pkl. Dùng cấu hình 50/50. Lỗi: {e}")
    ensemble_weights = {"LONG": {}, "SHORT": {}}
# 3. TẢI HỢP ĐỒNG BỘ NHỚ (FEATURE METADATA)
try:
    LIVE_META_CONFIG = joblib.load("xgb_v8_meta.pkl")
    LIVE_FEATURE_COLS = LIVE_META_CONFIG["feature_columns"]
    LIVE_META_FEATURES = joblib.load("xgb_v8_meta_features.pkl")
    print(f"✅ Đã load Artifact Meta: {len(LIVE_FEATURE_COLS)} features Tầng 1 | {len(LIVE_META_FEATURES)} features Tầng 2.")
except Exception as e:
    print(f"❌ LỖI CHÍNH MẠNG: KHÔNG THỂ NẠP HỢP ĐỒNG TÍNH NĂNG! {e}")
    LIVE_FEATURE_COLS = []
    LIVE_META_FEATURES = []
try:
    with open("model_calibrations.json", "r") as f:
        LIVE_CALIBRATIONS = json.load(f)
    print("✅ Đã load Artifact: model_calibrations.json")
except Exception as e:
    print(f"⚠️ Lỗi load Calibrations: {e}. Bot sẽ chuyển về chế độ phòng thủ tối đa!")
    LIVE_CALIBRATIONS = {}

def calculate_real_path_pnl(df, direction="LONG", pt_mult=2.0, sl_mult=1.0, max_bars=12):
    FEE_RATE = 0.0004      # Phí Taker 0.04%
    SLIPPAGE = 0.0005      # Trượt giá 0.05%
    FUNDING_PER_BAR = 0.00001 # Funding rate
    
    pnl_array = np.zeros(len(df))
    highs = df['High'].values
    lows = df['Low'].values
    closes = df['Close'].values
    atrs = df['ATR_Pct'].values if 'ATR_Pct' in df.columns else np.full(len(df), 0.02)
    
    for i in range(len(df) - max_bars):
        entry_price = closes[i]
        atr = atrs[i]
        
        if direction == "LONG":
            tp_price = entry_price * (1 + atr * pt_mult)
            sl_price = entry_price * (1 - atr * sl_mult)
        else:
            tp_price = entry_price * (1 - atr * pt_mult)
            sl_price = entry_price * (1 + atr * sl_mult)
            
        bars_held = max_bars
        exit_price = closes[i + max_bars]
        
        for j in range(1, max_bars + 1):
            future_idx = i + j
            curr_h = highs[future_idx]
            curr_l = lows[future_idx]
            
            if direction == "LONG":
                if curr_l <= sl_price:
                    exit_price = sl_price
                    bars_held = j
                    break
                elif curr_h >= tp_price:
                    exit_price = tp_price
                    bars_held = j
                    break
            else:
                if curr_h >= sl_price:
                    exit_price = sl_price
                    bars_held = j
                    break
                elif curr_l <= tp_price:
                    exit_price = tp_price
                    bars_held = j
                    break
                    
        raw_return = (exit_price - entry_price) / entry_price if direction == "LONG" else (entry_price - exit_price) / entry_price
        total_friction_costs = (FEE_RATE + SLIPPAGE) * 2 + (FUNDING_PER_BAR * bars_held)
        pnl_array[i] = raw_return - total_friction_costs
        
    return pnl_array

def is_regime_profitable(direction, regime_val):
    """Cầu dao Risk Manager: Kiểm tra PF ròng của Sư phụ Tầng 1"""
    if not LIVE_CALIBRATIONS: return False 
    
    regime_str = str(int(regime_val)) if pd.notna(regime_val) else "base"
    if regime_str not in LIVE_CALIBRATIONS:
        regime_str = "base"
        
    pf_score = LIVE_CALIBRATIONS[regime_str][direction.upper()].get("PF", 0.0)
    
    # CHỈ MỞ KHÓA NẾU PF RÒNG > 1.05
    return pf_score > 1.05

if "ghi_log" not in globals():
    def ghi_log(thong_bao):
        print(thong_bao)

if "send_ban_signal" not in globals():
    def send_ban_signal(message, target="standard"):
        print(f"[{target}] {message}")

if "get_sac_portfolio_allocations" not in globals():
    def get_sac_portfolio_allocations(list_active_signals, total_portfolio_usdt=1000.0):
        return {}

if "get_dynamic_budget_kelly" not in globals():
    def get_dynamic_budget_kelly(probability, tp_pct, sl_pct, total_capital=TOTAL_PORTFOLIO_USDT):
        budget = max(10.0, min(total_capital * 0.05, total_capital))
        return budget, "TIÊU CHUẨN"

if "apply_cmc_fusion" not in globals():
    def apply_cmc_fusion(raw_proba, asset_sentiment, trend_str, vol_regime, action_type):
        fused_proba = raw_proba
        if action_type == "OPEN_LONG":
            if asset_sentiment > 0.2: fused_proba += 0.05
            if trend_str > 0: fused_proba += 0.03
        elif action_type == "OPEN_SHORT":
            if asset_sentiment < -0.2: fused_proba += 0.05
            if trend_str < 0: fused_proba += 0.03
        return max(0.0, min(1.0, fused_proba))

def apply_triple_barrier(df, pt_multiplier=1.0, sl_multiplier=1.0, t1_bars=24):
    # Tính biến động (Volatility) làm ngưỡng động cho rào cản
    df['volatility'] = df['Close'].pct_change().rolling(window=100).std()
    
    # Danh sách kết quả: 1 (TP), -1 (SL), 0 (Timeout)
    labels = []
    
    for i in range(len(df) - t1_bars):
        price_start = df['Close'].iloc[i]
        vol = df['volatility'].iloc[i]
        
        # Ngưỡng động dựa trên Volatility
        upper_barrier = price_start * (1 + vol * pt_multiplier)
        lower_barrier = price_start * (1 - vol * sl_multiplier)
        
        # Kiểm tra nến nào chạm rào cản trước
        found_barrier = False
        for j in range(1, t1_bars + 1):
            price_current = df['Close'].iloc[i + j]
            
            if price_current >= upper_barrier:
                labels.append(1) # Hit TP
                found_barrier = True
                break
            elif price_current <= lower_barrier:
                labels.append(-1) # Hit SL
                found_barrier = True
                break
                
        if not found_barrier:
            labels.append(0) # Timeout (không chạm rào nào)
            
    # Pad kết quả cho đủ độ dài dataframe
    return labels + [np.nan] * t1_bars


class PurgedWalkForwardSplitter:
    def __init__(self, n_splits=5, purge_bars=12, embargo_bars=6, test_size_ratio=0.15):
        """
        n_splits: Số lần trượt cửa sổ (Walk-forward steps).
        purge_bars: Xóa các nến ranh giới để tránh rò rỉ (Label leakage) từ TBM.
        embargo_bars: Cách ly test set với train set tiếp theo.
        test_size_ratio: Tỷ lệ data dùng làm tập Test trong mỗi cửa sổ.
        """
        self.n_splits = n_splits
        self.purge_bars = purge_bars
        self.embargo_bars = embargo_bars
        self.test_size_ratio = test_size_ratio

    def split(self, df):
        total_bars = len(df)
        test_bars_per_split = int(total_bars * self.test_size_ratio / self.n_splits)
        base_train_end = total_bars - (test_bars_per_split * self.n_splits)
        splits = []
        for i in range(self.n_splits):
            test_start = base_train_end + (i * test_bars_per_split)
            test_end = test_start + test_bars_per_split
            train_end = test_start - self.purge_bars
            train_idx = np.arange(0, train_end)
            test_idx = np.arange(test_start, test_end)
            splits.append((train_idx, test_idx))
        return splits
# COST-AWARE & FUNDING-AWARE BACKTEST
def cost_aware_walk_forward_backtest(df_test, proba, threshold, direction="long"):
    """
    Backtest mô phỏng trượt giá, phí Taker Binance và Funding Rate.
    """
    local = df_test.copy().reset_index(drop=True)
    local["buy_proba"] = np.asarray(proba).flatten()
    target_col = f"target_{direction}"
    FEE_RATE = 0.0004        
    SLIPPAGE = 0.0005       
    FUNDING_PER_BAR = 0.00001 
    ROUND_TRIP_COST = (FEE_RATE + SLIPPAGE) * 2
    equity = 1.0
    trades = []
    i = 0
    while i < len(local):
        row = local.iloc[i]
        if row["buy_proba"] >= threshold:
            vol = float(row.get("Volatility_24", 0.02))
            hold_bars = 6 
            funding_cost = hold_bars * FUNDING_PER_BAR
            if int(row[target_col]) == 1:
                gross_ret = vol * 2.0
                net_ret = gross_ret - ROUND_TRIP_COST - funding_cost
            else:
                gross_ret = -vol * 1.0
                net_ret = gross_ret - ROUND_TRIP_COST - funding_cost
            equity *= (1.0 + net_ret)
            trades.append({
                "idx": i, "buy_proba": float(row["buy_proba"]),
                "actual_label": int(row[target_col]), "net_return": net_ret,
                "win": net_ret > 0,
            })
            i += hold_bars 
        else:
            i += 1
    trades_df = pd.DataFrame(trades)
    if len(trades_df) == 0:
        return {"trades": 0, "win_rate": 0.0, "profit_factor": 0.0, "final_equity": 1.0, "max_drawdown": 0.0}
    gains = trades_df.loc[trades_df["net_return"] > 0, "net_return"].sum()
    losses = -trades_df.loc[trades_df["net_return"] < 0, "net_return"].sum()
    profit_factor = gains / losses if losses > 0 else np.inf
    equity_curve = 1.0
    peak = 1.0
    max_dd = 0.0
    for r in trades_df["net_return"]:
        equity_curve *= (1.0 + float(r))
        peak = max(peak, equity_curve)
        max_dd = max(max_dd, (peak - equity_curve) / peak if peak > 0 else 0.0)
    return {
        "trades": int(len(trades_df)),
        "win_rate": float(trades_df["win"].mean() * 100),
        "profit_factor": float(profit_factor if np.isfinite(profit_factor) else 999.0),
        "final_equity": float(equity),
        "max_drawdown": float(max_dd * 100),
    }

# =====================================================================
# 🏭 NHÀ MÁY SẢN XUẤT ĐẶC TRƯNG (ĐÃ VÁ 5 LỖI THIẾU CỘT)
# =====================================================================
class QuantFeatureEngineer:
    @staticmethod
    def _build_base(df):
        df = df.copy()
        for col in ["Open", "High", "Low", "Close", "Volume"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            
        if "Open time" in df.columns:
            if not pd.api.types.is_datetime64_any_dtype(df["Open time"]):
                sample = df["Open time"].dropna().iloc[0]
                if isinstance(sample, (int, float, np.integer, np.floating)):
                    df["Open time"] = pd.to_datetime(df["Open time"], unit="ms", utc=True).dt.tz_localize(None)
                else:
                    df["Open time"] = pd.to_datetime(df["Open time"], utc=True).dt.tz_localize(None)
            
            # [VÁ LỖI 1 & 2]: Trả lại Hour và DayOfWeek
            df["Hour"] = df["Open time"].dt.hour
            df["DayOfWeek"] = df["Open time"].dt.dayofweek
        else:
            df["Hour"], df["DayOfWeek"] = 0, 0
                    
        df["Return"] = df["Close"].pct_change()
        df["EMA_14"] = df["Close"].ewm(span=14, adjust=False).mean()
        df["EMA_50"] = df["Close"].ewm(span=50, adjust=False).mean()
        
        delta = df["Close"].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        df["RSI_14"] = 100 - (100 / (1 + (gain / loss.replace(0, np.nan))))
        
        df["MACD"] = df["Close"].ewm(span=12, adjust=False).mean() - df["Close"].ewm(span=26, adjust=False).mean()
        df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
        df["MACD_Gap"] = df["MACD"] - df["MACD_Signal"]
        
        # [VÁ LỖI 3]: Trả lại MACD_Hist
        df["MACD_Hist"] = df["MACD"] - df["MACD_Signal"]
        
        df["BB_Mid"] = df["Close"].rolling(window=20).mean()
        df["BB_Std"] = df["Close"].rolling(window=20).std()
        df["BB_Upper"] = df["BB_Mid"] + (df["BB_Std"] * 2)
        df["BB_Lower"] = df["BB_Mid"] - (df["BB_Std"] * 2)
        return df

    @staticmethod
    def _build_advanced(df):
        df["Log_Return_1"] = np.log(df["Close"]).diff()
        for i in [3, 6, 12, 24]: df[f"Return_{i}"] = df["Close"].pct_change(i)
        df["Range_Pct"] = (df["High"] - df["Low"]) / df["Close"].replace(0, np.nan)
        df["Body_Pct"] = (df["Close"] - df["Open"]) / df["Open"].replace(0, np.nan)
        df["Volume_Change"] = df["Volume"].pct_change().replace([np.inf, -np.inf], np.nan)
        df["BB_Width"] = (df["BB_Upper"] - df["BB_Lower"]) / df["BB_Mid"].replace(0, np.nan)
        volume_mean_24 = df["Volume"].rolling(24).mean()
        volume_std_24 = df["Volume"].rolling(24).std().replace(0, np.nan)
        df["Volume_Z"] = (df["Volume"] - volume_mean_24) / volume_std_24
        df["Volume_Regime"] = df["Volume"] / df["Volume"].rolling(72).mean().replace(0, np.nan) - 1.0
        
        # 🐋 ĐỊNH VỊ BẢN ĐỒ THANH KHOẢN CÁ MẬP (ORDER BLOCKS)
        df["Lower_Wick"] = df[["Open", "Close"]].min(axis=1) - df["Low"]
        df["Upper_Wick"] = df["High"] - df[["Open", "Close"]].max(axis=1)
        df["Body_Abs"] = (df["Close"] - df["Open"]).abs()
        high_vol = df["Volume"] > volume_mean_24 * 1.5
        df["Is_Bullish_OB"] = ((df["Lower_Wick"] > df["Body_Abs"] * 2) & high_vol).astype(int)
        df["Is_Bearish_OB"] = ((df["Upper_Wick"] > df["Body_Abs"] * 2) & high_vol).astype(int)
        df["Recent_Demand_Zone"] = df["Low"].where(df["Is_Bullish_OB"] == 1).ffill()
        df["Recent_Supply_Zone"] = df["High"].where(df["Is_Bearish_OB"] == 1).ffill()
        df["Dist_to_Demand"] = (df["Close"] - df["Recent_Demand_Zone"]) / df["Close"]
        df["Dist_to_Supply"] = (df["Recent_Supply_Zone"] - df["Close"]) / df["Close"]
        prev_close = df["Close"].shift(1)
        true_range = pd.concat([df["High"] - df["Low"], (df["High"] - prev_close).abs(), (df["Low"] - prev_close).abs()], axis=1).max(axis=1)
        df["ATR_14"] = true_range.rolling(14).mean()
        df["ATR_Pct"] = df["ATR_14"] / df["Close"].replace(0, np.nan)
        df["ATR_Regime"] = df["ATR_Pct"] / df["ATR_Pct"].rolling(72).mean().replace(0, np.nan)
        df["Volatility_24"] = df["Return"].rolling(24).std()
        df["Volatility_Regime"] = df["Volatility_24"] / df["Volatility_24"].rolling(72).mean().replace(0, np.nan)
        df["EMA_14_Dist"] = (df["Close"] - df["EMA_14"]) / df["EMA_14"].replace(0, np.nan)
        df["EMA_50_Dist"] = (df["Close"] - df["EMA_50"]) / df["EMA_50"].replace(0, np.nan)
        df["Trend_Strength"] = (df["EMA_14"] - df["EMA_50"]) / df["Close"].replace(0, np.nan)
        df["EMA_14_Slope_3"] = df["EMA_14"].pct_change(3)
        df["EMA_50_Slope_6"] = df["EMA_50"].pct_change(6)
        df["RSI_14_Norm"] = df["RSI_14"] / 100.0
        df["Breakout_20"] = df["Close"] / df["High"].rolling(20).max().shift(1).replace(0, np.nan) - 1.0
        df["Breakdown_20"] = df["Close"] / df["Low"].rolling(20).min().shift(1).replace(0, np.nan) - 1.0
        df["Dist_to_Demand"] = df["Dist_to_Demand"].fillna(999.0)
        df["Dist_to_Supply"] = df["Dist_to_Supply"].fillna(999.0)

        # 🔬 MICROSTRUCTURE: TAKER IMBALANCE & ORDER FLOW
        df["Taker_Buy_Vol"] = pd.to_numeric(df["Taker Buy Base"], errors="coerce").fillna(0)
        df["Taker_Sell_Vol"] = df["Volume"] - df["Taker_Buy_Vol"]
        df["Taker_Buy_Ratio"] = df["Taker_Buy_Vol"] / df["Volume"].replace(0, np.nan)
        df["Taker_Imbalance"] = (df["Taker_Buy_Vol"] - df["Taker_Sell_Vol"]) / df["Volume"].replace(0, np.nan)
        df["Taker_Imbalance_Delta"] = df["Taker_Imbalance"].diff()
        vol_percentile_75 = df["Volume"].rolling(72).quantile(0.75)
        df["Is_High_Vol_Bucket"] = (df["Volume"] > vol_percentile_75).astype(int)
        df["Smart_Money_Flow"] = df["Taker_Imbalance"] * df["Is_High_Vol_Bucket"]
        return df

    @staticmethod
    def _add_lags(df):
        lag_steps = globals().get("LAG_STEPS_V8", [1, 2, 3, 6, 12])
        lag_cols = ["Log_Return_1", "Return_3", "Return_6", "Return_12", "Return_24", "EMA_14_Dist", "EMA_50_Dist", "Trend_Strength", "RSI_14_Norm", "MACD_Gap", "ATR_Pct", "Volume_Z", "Volume_Regime", "Volatility_24", "Volatility_Regime", "Breakout_20", "Breakdown_20"]
        for col in lag_cols:
            if col in df.columns:
                for lag in lag_steps: df[f"{col}_lag_{lag}"] = df[col].shift(lag)
        return df

    @staticmethod
    def _build_gatekeeper_filters(df):
        df["Setup_Trend_Primary_L"] = ((df["EMA_14"] > df["EMA_50"]) & (df["Trend_Strength"] > -0.002)).astype(int)
        df["Setup_MACD_OK_L"] = (df["MACD_Gap"] > -0.0005).astype(int)
        df["Setup_RSI_OK_L"] = ((df["RSI_14_Norm"] >= 0.44) & (df["RSI_14_Norm"] <= 0.74)).astype(int)
        df["Setup_ATR_OK"] = (df["ATR_Pct"] <= 0.03).astype(int) 
        df["Setup_Vol_OK"] = (df["Volatility_Regime"] <= 1.9).astype(int) 
        df["Setup_Volume_OK"] = (df["Volume_Regime"] > -0.40).astype(int) 
        df["Setup_Breakout_OK_L"] = (df["Breakout_20"] > -0.03).astype(int)
        df["Setup_Slope_OK_L"] = ((df["EMA_14_Slope_3"] > -0.002) & (df["EMA_50_Slope_6"] > -0.003)).astype(int)
        
        df["Setup_Long_Score"] = df[["Setup_Trend_Primary_L", "Setup_MACD_OK_L", "Setup_RSI_OK_L", "Setup_ATR_OK", "Setup_Vol_OK", "Setup_Volume_OK", "Setup_Breakout_OK_L", "Setup_Slope_OK_L"]].sum(axis=1)
        df["setup_long_candidate"] = ((df["Setup_Long_Score"] >= 5) & (df["Setup_Trend_Primary_L"] == 1) & (df["Setup_ATR_OK"] == 1))
        
        df["Setup_Trend_Primary_S"] = ((df["EMA_14"] < df["EMA_50"]) & (df["Trend_Strength"] < 0.002)).astype(int)
        df["Setup_MACD_OK_S"] = (df["MACD_Gap"] < 0.0005).astype(int)
        df["Setup_RSI_OK_S"] = ((df["RSI_14_Norm"] <= 0.56) & (df["RSI_14_Norm"] >= 0.26)).astype(int)
        df["Setup_Breakdown_OK_S"] = (df["Breakdown_20"] < 0.03).astype(int)
        df["Setup_Slope_OK_S"] = ((df["EMA_14_Slope_3"] < 0.002) & (df["EMA_50_Slope_6"] < 0.003)).astype(int)
        
        df["Setup_Short_Score"] = df[["Setup_Trend_Primary_S", "Setup_MACD_OK_S", "Setup_RSI_OK_S", "Setup_ATR_OK", "Setup_Vol_OK", "Setup_Volume_OK", "Setup_Breakdown_OK_S", "Setup_Slope_OK_S"]].sum(axis=1)
        df["setup_short_candidate"] = ((df["Setup_Short_Score"] >= 5) & (df["Setup_Trend_Primary_S"] == 1) & (df["Setup_ATR_OK"] == 1))
        return df

    @classmethod
    def run_pipeline(cls, df_raw, asset_sentiment=0.0, symbol="BTCUSDT"): # Thêm tham số symbol
        if df_raw is None or len(df_raw) == 0: return pd.DataFrame()
        
        df = df_raw.copy()
        if "Sentiment_Score" not in df.columns: df["Sentiment_Score"] = asset_sentiment
        else: df["Sentiment_Score"] = df["Sentiment_Score"].fillna(asset_sentiment)
        
        df = cls._build_base(df)
        df = cls._build_advanced(df)
        df = cls._add_lags(df)
        df = cls._build_gatekeeper_filters(df)
        
        # ===============================================================
        # 🧬 ĐÓNG DẤU DNA TÀI SẢN (ASSET IDENTIFIER - ONE HOT ENCODING)
        # ===============================================================
        TARGET_SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT", 
                          "DOGEUSDT", "AVAXUSDT", "LINKUSDT", "NEARUSDT", "ADAUSDT"]
        for target_sym in TARGET_SYMBOLS:
            # Tạo các cột Asset_BTCUSDT, Asset_ETHUSDT... gán = 1 nếu đúng coin đang chạy
            df[f"Asset_{target_sym}"] = (1 if symbol == target_sym else 0)
        
        return df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

def prepare_live_feature_frame_dual(df, asset_sentiment=0.0, symbol="BTCUSDT"):
    return QuantFeatureEngineer.run_pipeline(df, asset_sentiment, symbol)

def evaluate_trading_performance(preds_proba, df_val, action_type):
    try:
        threshold = 0.55
        trades = []
        returns = df_val["Future_Return"].values
        for i in range(len(preds_proba)):
            if preds_proba[i] >= threshold:
                pnl = returns[i] if action_type == "LONG" else -returns[i]
                trades.append(pnl)
        if not trades: 
            return {"pnl": -1.0, "profit_factor": 0.0, "max_drawdown": 1.0}
        total_pnl = sum(trades)
        wins = [t for t in trades if t > 0]
        losses = [abs(t) for t in trades if t <= 0]
        total_win = sum(wins)
        total_loss = sum(losses)
        if total_loss > 0:
            profit_factor = total_win / total_loss
        else:
            profit_factor = 99.0 if total_win > 0 else 0.0
        cum_pnl = np.cumsum(trades)
        drawdown = np.maximum.accumulate(cum_pnl) - cum_pnl
        max_dd = np.max(drawdown) if len(drawdown) > 0 else 0.0
        return {
            "pnl": total_pnl, 
            "profit_factor": profit_factor, 
            "max_drawdown": max_dd, 
            "trade_count": len(trades)
        }
    except: 
        return {"pnl": -1.0, "profit_factor": 0.0, "max_drawdown": 1.0}

import csv
import os
from datetime import datetime
# =========================================================
# 🛡️ CÔNG TẮC SINH TỬ (LIVE TRADING VS SHADOW TRADING)
# =========================================================
SHADOW_MODE_ONLY = True 

def log_shadow_trade(symbol, direction, trade_size_usd, exec_price, order_type, ev_adj, exec_msg):
    """
    Ghi sổ lệnh Shadow Trading với dữ liệu Slippage/Spread THẬT 100% 
    từ Execution Gatekeeper.
    """
    filename = "shadow_trades_log.csv"
    file_exists = os.path.isfile(filename)
    
    with open(filename, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        if not file_exists:
            # Tạo Header cho file mới
            writer.writerow(["Timestamp_UTC", "Symbol", "Direction", "Size_USD", 
                             "Order_Type", "Simulated_Fill_Price", "Expected_EV_Adj", "Execution_Log"])
        
        # Ghi data lệnh
        writer.writerow([
            datetime.utcnow().isoformat(), 
            symbol, direction, round(trade_size_usd, 2), 
            order_type, exec_price, round(ev_adj, 6), exec_msg
        ])
    print(f"👻 [SHADOW LOG] Đã ghi sổ giả lập lệnh {direction} {symbol} tại giá {exec_price}")

def train_meta_model(X_train, y_primary_pred, y_actual):
    meta_labels = []
    for pred, actual in zip(y_primary_pred, y_actual):
        if pred == actual and actual != 0:
            meta_labels.append(1) 
        else:
            meta_labels.append(0)       
    meta_model.fit(X_train, meta_labels)
    return meta_labels

def champion_challenger_retrain(df_retrain):
    ghi_log("🧠 Bắt đầu đúc lại Mô hình (Champion vs Challenger)...")
    df_retrain = df_retrain.groupby('symbol', group_keys=False).apply(
        lambda x: apply_triple_barrier_high_low(x, pt_multiplier=2.0, sl_multiplier=1.0, time_limit=12).assign(symbol=x.name)
    ).reset_index(drop=True)
    df_train_clean = df_retrain[df_retrain['TBM_Label'] != 0].copy()
    df_train_clean["target_long"] = (df_train_clean["TBM_Label"] == 1).astype(int)
    df_train_clean["target_short"] = (df_train_clean["TBM_Label"] == -1).astype(int) 
    df_retrain['Forward_Return_20'] = df_retrain.groupby('symbol')['Close'].transform(lambda x: x.shift(-20) / x - 1)
    df_quant_clean = df_retrain.dropna(subset=['Forward_Return_20']).copy()
    y_quant = np.clip(df_quant_clean['Forward_Return_20'].astype(np.float32), -0.30, 0.30)
    X_quant = df_quant_clean[feature_columns_v8].astype(np.float32)
    # ⚖️ 3. CHUẨN BỊ NHÃN PNL BẰNG PATH PNL (VÁ LỖI CÙNG 1 PHE)
    def safe_apply_path_pnl(group):
        pnl_long = calculate_real_path_pnl(group, direction="LONG")
        pnl_short = calculate_real_path_pnl(group, direction="SHORT")
        group['realized_net_return_long'] = pd.Series(pnl_long, index=group.index)
        group['realized_net_return_short'] = pd.Series(pnl_short, index=group.index)
        return group
    df_train_clean = df_train_clean.groupby('symbol', group_keys=False).apply(safe_apply_path_pnl)
    df_train_clean['realized_net_return_long'] = df_train_clean.get('realized_net_return_long', pd.Series(dtype=float)).fillna(0.0)
    df_train_clean['realized_net_return_short'] = df_train_clean.get('realized_net_return_short', pd.Series(dtype=float)).fillna(0.0)
    MIN_PROFIT_THRESHOLD = 0.0015 
    df_train_clean["meta_target_long"] = (df_train_clean["realized_net_return_long"] > MIN_PROFIT_THRESHOLD).astype(int)
    df_train_clean["meta_target_short"] = (df_train_clean["realized_net_return_short"] > MIN_PROFIT_THRESHOLD).astype(int)
    try:
        from xgboost import XGBRegressor 
        X_train = df_train_clean[feature_columns_v8].astype(np.float32)
        ghi_log("⏳ Đang đúc Sư Phụ Tầng 1 (LONG/SHORT)...")
        new_long_model = build_xgb_v8_model(df_train_clean["target_long"])
        new_long_model.fit(X_train, df_train_clean["target_long"], verbose=False)
        joblib.dump(new_long_model, "xgb_v8_long_fee_aware_multi.pkl")
        new_short_model = build_xgb_v8_model(df_train_clean["target_short"])
        new_short_model.fit(X_train, df_train_clean["target_short"], verbose=False)
        joblib.dump(new_short_model, "xgb_v8_short_fee_aware_multi.pkl")
        ghi_log("⏳ Đang đúc Vệ Sĩ Tầng 2 (EV & Uncertainty) ĐỘC LẬP CHO 2 PHE...")
        
        # --- ĐÚC NÃO PHE LONG ---
        meta_feat_long = df_train_clean[meta_feature_cols].copy()
        meta_feat_long['pred_proba'] = new_long_model.predict_proba(X_train)[:, 1] if hasattr(new_long_model, 'predict_proba') else new_long_model.predict(X_train)
        y_ev_long = df_train_clean["realized_net_return_long"].astype(np.float32)

        ev_model_l = XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)
        ev_model_l.fit(meta_feat_long, y_ev_long, verbose=False)
        unc_model_l = XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)
        unc_model_l.fit(meta_feat_long, np.abs(y_ev_long - ev_model_l.predict(meta_feat_long)), verbose=False)

        # --- ĐÚC NÃO PHE SHORT ---
        meta_feat_short = df_train_clean[meta_feature_cols].copy()
        meta_feat_short['Sentiment_Score'] = -meta_feat_short['Sentiment_Score'] # Đảo chiều Sentiment
        meta_feat_short['pred_proba'] = new_short_model.predict_proba(X_train)[:, 1] if hasattr(new_short_model, 'predict_proba') else new_short_model.predict(X_train)
        y_ev_short = df_train_clean["realized_net_return_short"].astype(np.float32)

        ev_model_s = XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)
        ev_model_s.fit(meta_feat_short, y_ev_short, verbose=False)
        unc_model_s = XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)
        unc_model_s.fit(meta_feat_short, np.abs(y_ev_short - ev_model_s.predict(meta_feat_short)), verbose=False)

        # --- LƯU TRỮ VÀO TỪ ĐIỂN CHUNG ---
        meta_model_dict = {
            'ev_model_l': ev_model_l,
            'uncertainty_model_l': unc_model_l,
            'ev_model_s': ev_model_s,
            'uncertainty_model_s': unc_model_s
        }
        joblib.dump(meta_model_dict, "xgb_v8_meta_model.pkl")
        ghi_log("⏳ Đang đúc AI Exit Quantile (Khóa ảo giác)...")
        quantile_models = {}
        for q in [0.10, 0.50, 0.90]:
            q_model = XGBRegressor(objective='reg:quantileerror', quantile_alpha=q, n_estimators=100, max_depth=4, learning_rate=0.05, tree_method='hist', random_state=42)
            q_model.fit(X_quant, y_quant, verbose=False)
            quantile_models[f"q_{q}"] = q_model
        joblib.dump(quantile_models, "xgb_v8_exit_model.pkl")
        if not run_artifact_smoke_test():
            raise ValueError("Mô hình mới đúc bị lỗi Smoke Test! Hủy bỏ việc cập nhật, giữ nguyên mô hình cũ!")
        ghi_log("🎉 BẢO DƯỠNG HOÀN TẤT! Toàn bộ Model V8 đã được cập nhật thành công và sạch rác 100%!")
    except Exception as e:
        ghi_log(f"⚠️ Lỗi trong lúc Fit Model Retrain: {e}")

def _clamp(value, low, high): return max(low, min(high, value))

def apply_cross_sectional_ranking(all_candidates, max_open_positions=3):
    valid_candidates = [c for c in all_candidates if c is not None and c["action"] in ["LONG", "SHORT"]]
    if not valid_candidates:
        return []
    for c in valid_candidates:
        exp_pnl = c.get("expected_pnl", 0.01) 
        c["edge_score"] = c["final_proba"] * exp_pnl
    ranked_candidates = sorted(valid_candidates, key=lambda x: x["edge_score"], reverse=True)
    long_queue = [c for c in ranked_candidates if c["action"] == "LONG"]
    short_queue = [c for c in ranked_candidates if c["action"] == "SHORT"]
    top_k_selected = []
    ghi_log(f"\n{'='*20} 🏆 BẢNG XẾP HẠNG EDGE TOÀN THỊ TRƯỜNG {'='*20}")
    for i, c in enumerate(ranked_candidates):
        medal = "🥇" if i==0 else "🥈" if i==1 else "🥉" if i==2 else "⭐"
        ghi_log(f" {medal} Rank {i+1}: {c['symbol']} [{c['action']}] | Edge Score: {c['edge_score']*10000:.2f} | Proba: {c['final_proba']*100:.1f}% | ExpPnL: {c.get('expected_pnl', 0)*100:.2f}%")
    ghi_log(f"{'='*72}")
    top_k_selected = ranked_candidates[:max_open_positions]
    return top_k_selected

def execution_gatekeeper(client, symbol, direction, quantity, expected_ev):
    """
    Gatekeeper Tối Hậu: Mô phỏng khớp lệnh thực tế (Depth Walk, Slippage, Maker/Taker, Funding).
    Đảm bảo Lãi ròng (EV) không bị Spread và Orderbook của sàn "ăn thịt".
    """
    try:
        # 1. Tải Orderbook tươi nhất (20 levels) tại đúng mili-giây này
        ob = client.futures_order_book(symbol=symbol, limit=20)
        bids = np.array(ob['bids'], dtype=float)
        asks = np.array(ob['asks'], dtype=float)

        best_bid, best_bid_vol = bids[0][0], bids[0][1]
        best_ask, best_ask_vol = asks[0][0], asks[0][1]
        spread_bps = (best_ask - best_bid) / best_bid * 10000

        # 2. THUẬT TOÁN DEPTH WALK (Quét sổ lệnh mô phỏng Partial Fill)
        rem_qty = quantity
        total_notional = 0.0
        
        # Chọn phe để đánh: Mua (Long) thì phải mua giá Ask của thằng bán. Bán (Short) thì xả vào Bid của thằng mua.
        book_side = asks if direction == "LONG" else bids
        benchmark_price = best_ask if direction == "LONG" else best_bid

        for price, vol in book_side:
            fill_qty = min(rem_qty, vol)
            total_notional += fill_qty * price
            rem_qty -= fill_qty
            if rem_qty <= 0:
                break # Đã khớp đủ khối lượng

        # Rủi ro 1: Partial Fill (Thanh khoản cạn kiệt, lệnh quét sạch 20 level mà vẫn dư)
        if rem_qty > 0:
            return False, f"Thanh khoản mỏng! Lệnh {quantity} quét sạch 20 mốc giá vẫn không đủ.", "NONE", 0.0

        # 3. TÍNH TOÁN CHI PHÍ THỰC TẾ (Slippage + Maker/Taker + Funding)
        vwap_price = total_notional / quantity # Giá khớp trung bình thực tế
        slippage_pct = abs(vwap_price - benchmark_price) / benchmark_price
        
        TAKER_FEE = 0.0004 # 0.04% Binance Futures (Taker)
        MAKER_FEE = 0.0002 # 0.02% (Maker)
        FUNDING_IMPACT = 0.0001 # Ước lượng hao hụt funding nếu cầm lệnh qua giờ chẵn

        total_exec_cost_pct = slippage_pct + TAKER_FEE + FUNDING_IMPACT
        
        # 4. CHỐT CHẶN SINH TỬ (Cổng EV Thực)
        # Lãi kỳ vọng (EV) BẮT BUỘC phải lớn hơn toàn bộ chi phí khớp lệnh cộng thêm biên an toàn 0.05%
        MIN_SAFETY_MARGIN = 0.0005 
        if expected_ev < (total_exec_cost_pct + MIN_SAFETY_MARGIN):
            msg = f"⛔ BÁC BỎ: EV {expected_ev*100:.2f}% < Chi phí khớp lệnh {total_exec_cost_pct*100:.3f}% (Slippage: {slippage_pct*100:.3f}%)"
            return False, msg, "NONE", 0.0
            
        # 5. SMART ROUTING (Quyết định đập thẳng Market hay rải Limit)
        if spread_bps > 7.0: 
            # Spread quá rộng > 7 bps, ném lệnh Limit (Maker) để giành quyền lợi phí rẻ
            order_type = "LIMIT"
            route_price = best_bid if direction == "LONG" else best_ask
            msg = f"Duyệt Maker (LIMIT). Spread rộng {spread_bps:.1f}bps. Trượt giá Market ước tính: {slippage_pct*100:.3f}%"
        else:
            # Spread hẹp, thanh khoản dày, đập Market ăn thẳng (Taker)
            order_type = "MARKET"
            route_price = vwap_price
            msg = f"Duyệt Taker (MARKET). Chi phí khớp an toàn: {total_exec_cost_pct*100:.3f}%"

        return True, msg, order_type, route_price

    except Exception as e:
        return False, f"Lỗi tính toán Orderbook Gatekeeper: {e}", "NONE", 0.0

def _safe_float(value, default=0.0):
    try:
        return default if value is None else float(value)
    except: return default

shap.initjs() if 'shap' in globals() else None
def extract_shap_insights(model, df_features, feature_cols):
    try:
        if model is None: return [], []
        X_live = df_features[feature_cols].astype(np.float32).iloc[[-1]]
        explainer = shap.TreeExplainer(model)
        shap_vals = explainer.shap_values(X_live)[0]
        feature_impacts = list(zip(feature_cols, shap_vals))
        feature_impacts.sort(key=lambda x: x[1], reverse=True)
        top_pushers = [(f, v) for f, v in feature_impacts if v > 0][:3]
        top_pullers = [(f, v) for f, v in feature_impacts if v < 0][::-1][:3] 
        return top_pushers, top_pullers
    except Exception as e:
        ghi_log(f"⚠️ Lỗi bóc tách SHAP: {e}")
        return [], []

def _utc_now(): return pd.Timestamp.utcnow()

def _json_default(value):
    if isinstance(value, deque): return list(value)
    if isinstance(value, (np.floating, np.integer)): return value.item()
    if isinstance(value, (pd.Timestamp, np.datetime64)): return str(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")

def _serialize_feature_dict(feature_dict):
    serializable = {}
    for key, value in (feature_dict or {}).items():
        if isinstance(value, (np.floating, np.integer)): serializable[key] = value.item()
        elif isinstance(value, (pd.Timestamp, np.datetime64)): serializable[key] = str(value)
        else: serializable[key] = value
    return serializable

def _symbol_state_defaults():
    return {
        "position_side": "NONE", "entry_price": 0.0, "quantity": 0.0, "invested_usdt": 0.0,
        "peak_price": 0.0, "trough_price": 0.0, "opened_at": None, "entry_features": None,
        "l2_buffer": deque(maxlen=10), "last_trade": None, "last_signal_target": "basic",
        "last_signal_reason": "", "last_prediction_side": "NONE", "last_prediction_proba": 0.0,
        "last_scan_price": 0.0,
    }

def _ensure_symbol_state(symbol):
    memory = bot_memory.setdefault(symbol, {})
    defaults = _symbol_state_defaults()
    for key, default_value in defaults.items():
        if key not in memory:
            memory[key] = deque(default_value, maxlen=10) if isinstance(default_value, deque) else default_value
    if not isinstance(memory.get("l2_buffer"), deque):
        memory["l2_buffer"] = deque(memory.get("l2_buffer", []), maxlen=10)
    return memory

def _save_runtime_state():
    try:
        with state_lock:
            runtime_state["symbols"] = bot_memory
            STATE_FILE.write_text(json.dumps(runtime_state, ensure_ascii=False, indent=2, default=_json_default), encoding="utf-8")
    except Exception as exc: ghi_log(f"❌ Lỗi lưu trạng thái: {exc}")

def _get_cached_klines(symbol, interval=Client.KLINE_INTERVAL_1HOUR, limit=260, ttl_seconds=MARKET_CACHE_TTL_SECONDS):
    cache_key = (symbol, interval, limit)
    now_ts = time.time()
    cached = market_data_cache.get(cache_key)
    if cached and (now_ts - cached["fetched_at"] <= ttl_seconds):
        return pd.DataFrame(cached["rows"], columns=KLINE_COLUMNS)
    klines = client.get_klines(symbol=symbol, interval=interval, limit=limit)
    market_data_cache[cache_key] = {"fetched_at": now_ts, "rows": klines}
    return pd.DataFrame(klines, columns=KLINE_COLUMNS)

def _hours_since(timestamp_value):
    if not timestamp_value: return None
    try: return float((_utc_now() - pd.Timestamp(timestamp_value)).total_seconds() / 3600.0)
    except: return None

def _headline_sentiment_score(text):
    text = (text or "").lower()
    raw_score = sum(weight for term, weight in POSITIVE_SENTIMENT_TERMS.items() if term in text)
    raw_score -= sum(weight for term, weight in NEGATIVE_SENTIMENT_TERMS.items() if term in text)
    if "etf" in text and "approval" in text: raw_score += 0.8
    if "sec" in text and any(x in text for x in ["lawsuit", "delay", "reject"]): raw_score -= 0.7
    return math.tanh(raw_score / 2.5)

import json
import os
import numpy as np

print("🎰 ĐANG KHỞI ĐỘNG ĐỘNG CƠ PHÂN BỔ VỐN BANDIT...")

BANDIT_FILE = "bandit_ledger.json"
BASE_CAPITAL = 100.0  # Vốn tiêu chuẩn
MAX_CAPITAL = 300.0   # Nhồi tối đa (Exploit) cho cụm đang siêu Edge
MIN_CAPITAL = 20.0    # Bóp nghẹt (Explore) cho cụm đang lệch pha

def load_bandit_ledger():
    if os.path.exists(BANDIT_FILE):
        with open(BANDIT_FILE, 'r') as f:
            return json.load(f)
    return {}

def save_bandit_ledger(ledger):
    with open(BANDIT_FILE, 'w') as f:
        json.dump(ledger, f, indent=4)

def get_bandit_allocation(symbol, regime, direction):
    """
    Tính toán Volume vào lệnh dựa trên phong độ thực tế (Edge)
    """
    ledger = load_bandit_ledger()
    regime_str = str(int(regime)) if pd.notna(regime) else "base"
    arm_key = f"{symbol}_{regime_str}_{direction.upper()}"
    
    if arm_key not in ledger:
        # Cánh tay mới tinh: Cấp vốn cơ bản để "Thăm dò"
        return BASE_CAPITAL
        
    stats = ledger[arm_key]
    edge_score = stats.get("edge_score", 0.0)
    
    # 🎯 CÔNG THỨC SCALING: Tăng/Giảm vốn theo Edge
    # Cứ mỗi 1% Lãi Ròng tích lũy (Edge Score), ta nhồi thêm 50 USDT
    capital_allocation = BASE_CAPITAL + (edge_score * 100 * 50)
    
    # Chặn trần và sàn để bảo vệ rủi ro vỡ nợ
    return max(MIN_CAPITAL, min(capital_allocation, MAX_CAPITAL))

def update_bandit_feedback(symbol, regime, direction, realized_pnl_pct):
    """
    Cho Bandit "ăn" kết quả thật sau khi đóng lệnh để nó cập nhật Edge Score.
    Sử dụng thuật toán Cập nhật Trung bình Trượt Mũ (EMA).
    """
    ledger = load_bandit_ledger()
    regime_str = str(int(regime)) if pd.notna(regime) else "base"
    arm_key = f"{symbol}_{regime_str}_{direction.upper()}"
    if arm_key not in ledger:
        ledger[arm_key] = {"trades": 0, "edge_score": 0.0}
    ALPHA = 0.2
    old_score = ledger[arm_key]["edge_score"]
    new_score = (old_score * (1 - ALPHA)) + (realized_pnl_pct * ALPHA)
    ledger[arm_key]["trades"] += 1
    ledger[arm_key]["edge_score"] = new_score
    save_bandit_ledger(ledger)
    ghi_log(f"🎰 [BANDIT] Đã cập nhật tay {arm_key}: Edge Score mới = {new_score*100:.2f}%")

def get_asset_sentiment(symbol):
    now_ts = time.time()
    cached = news_cache.get(symbol)
    if cached and (now_ts - cached.get("fetched_at", 0) <= NEWS_CACHE_TTL_SECONDS):
        return _safe_float(cached.get("score"), 0.0)

    aliases = SYMBOL_NEWS_ALIASES.get(symbol, [symbol.replace("USDT", "").lower()])
    scored_entries, matched_titles = [], []

    for feed_url in NEWS_FEEDS:
        try:
            parsed_feed = feedparser.parse(feed_url)
            for entry in parsed_feed.entries[:MAX_HEADLINES_PER_FEED]:
                title = getattr(entry, "title", "") or ""
                summary = getattr(entry, "summary", "") or ""
                combined_text = f"{title} {summary}".lower()

                specific_match = any(alias in combined_text for alias in aliases)
                macro_match = any(k in combined_text for k in ["crypto", "market", "bitcoin", "binance", "etf", "fed"])
                if not specific_match and not macro_match: continue

                base_score = _headline_sentiment_score(combined_text)
                if base_score == 0.0 and not specific_match: continue

                recency_weight = 1.0
                published_parsed = getattr(entry, "published_parsed", None) or getattr(entry, "updated_parsed", None)
                if published_parsed:
                    published_ts = time.mktime(published_parsed)
                    age_hours = max(0.0, (now_ts - published_ts) / 3600.0)
                    recency_weight = _clamp(1.2 - (age_hours / 72.0), 0.35, 1.2)

                relevance_weight = 1.35 if specific_match else 0.55
                scored_entries.append(base_score * recency_weight * relevance_weight)
                if title: matched_titles.append(title.strip())
        except: continue

    sentiment_score = _clamp(sum(scored_entries) / len(scored_entries), -1.0, 1.0) if scored_entries else 0.0
    news_cache[symbol] = {"score": sentiment_score, "fetched_at": now_ts, "headline_count": len(scored_entries), "headlines": matched_titles[:5]}
    return sentiment_score

def check_black_swan(df, symbol):
    live_row = df.iloc[-1]
    realized_z = abs(_safe_float(live_row.get("Return"), 0.0)) / max(_safe_float(live_row.get("Volatility_24"), 0.0), 1e-6)
    atr_regime = _safe_float(live_row.get("ATR_Regime"), 1.0)
    vol_regime = _safe_float(live_row.get("Volatility_Regime"), 1.0)
    volume_z = abs(_safe_float(live_row.get("Volume_Z"), 0.0))
    body_pct = abs(_safe_float(live_row.get("Body_Pct"), 0.0))
    range_pct = abs(_safe_float(live_row.get("Range_Pct"), 0.0))
    return_6 = abs(_safe_float(live_row.get("Return_6"), 0.0))

    rule_score = (
        max(realized_z - 2.0, 0.0) * 0.55 + max(atr_regime - 1.8, 0.0) * 0.70 +
        max(vol_regime - 2.0, 0.0) * 0.95 + max(volume_z - 2.5, 0.0) * 0.35 +
        max(range_pct - 0.03, 0.0) * 18.0 + max(body_pct - 0.02, 0.0) * 15.0 +
        max(return_6 - 0.08, 0.0) * 12.0
    )

    ae_score, ae_trigger, dynamic_threshold = None, False, 0.028
    try:
        local_ae_threshold, local_ae_model, local_ae_scaler, local_ae_features = globals().get("ae_threshold"), globals().get("ae_model"), globals().get("ae_scaler"), globals().get("ae_features")
        if local_ae_threshold is not None: dynamic_threshold = max(_safe_float(local_ae_threshold, 0.0) * 1.25, 0.028)
        if local_ae_model and local_ae_scaler and local_ae_features and all(f in df.columns for f in local_ae_features):
            live_ae_features = df[local_ae_features].iloc[-1:].replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float).values
            live_ae_features_scaled = local_ae_scaler.transform(live_ae_features)
            reconstructed = local_ae_model.predict(live_ae_features_scaled, verbose=0)
            ae_score = float(np.mean(np.power(live_ae_features_scaled - reconstructed, 2), axis=1)[0])
            ae_trigger = ae_score > dynamic_threshold
    except Exception as exc: return False, {"triggered": False, "error": str(exc), "symbol": symbol}

    hard_trigger = realized_z >= 6.0 or return_6 >= 0.10
    triggered = bool(ae_trigger or rule_score >= 2.8 or hard_trigger)
    return triggered, {"triggered": triggered, "ae_score": ae_score, "ae_threshold": dynamic_threshold, "rule_score": round(rule_score, 4), "realized_z": round(realized_z, 4), "vol_regime": round(vol_regime, 4), "symbol": symbol}

def check_derivatives_squeeze(symbol, df_spot):
    try:
        df_local = df_spot.copy()
        for col in ["Close", "High", "Low", "Volume", "Taker Buy Base"]: df_local[col] = pd.to_numeric(df_local[col])
        recent_volume = df_local["Volume"].tail(6).sum()
        aggressive_buy_ratio = df_local["Taker Buy Base"].tail(6).sum() / max(recent_volume, 1e-9)
        price_change_6 = _safe_float(df_local["Close"].pct_change(6).iloc[-1], 0.0)
        range_6 = (df_local["High"].tail(6).max() - df_local["Low"].tail(6).min()) / max(_safe_float(df_local["Close"].iloc[-1], 0.0), 1e-9)

        if aggressive_buy_ratio >= 0.62 and price_change_6 > 0.015 and range_6 > 0.02: return "UP_SQUEEZE", f"Phe mua ép giá (BuyRatio: {aggressive_buy_ratio:.2f})"
        if aggressive_buy_ratio <= 0.38 and price_change_6 < -0.015 and range_6 > 0.02: return "DOWN_SQUEEZE", f"Phe bán xả hàng (BuyRatio: {aggressive_buy_ratio:.2f})"
        return "SAFE", f"Tỷ lệ mua ổn định ({aggressive_buy_ratio:.2f})"
    except Exception as exc: return "SAFE", f"Thiếu dữ liệu Squeeze: {exc}"

def track_champion_performance(memory, current_price, previous_price, symbol):
    try:
        actual_return = (current_price - previous_price) / previous_price
        last_side = memory.get("last_prediction_side", "NONE")
        last_proba = _safe_float(memory.get("last_prediction_proba"), 0.0)
        if last_side not in {"LONG", "SHORT"}: 
            return f"📊 {symbol} Lịch sử: {actual_return*100:+.2f}%"
        correct = (actual_return > 0 and last_side == "LONG") or (actual_return < 0 and last_side == "SHORT")
        status = "🟢 WIN" if correct else "🔴 LOSS"
        streak = memory.get("performance_streak", deque(maxlen=20))
        streak.append(1 if correct else 0)
        memory["performance_streak"] = streak
        win_rate = sum(streak) / len(streak) if len(streak) > 0 else 0.0
        warning = "⚠️ CẢNH BÁO DRIFT" if win_rate < 0.4 and len(streak) >= 10 else "Ổn định"
        return f"🏆 Champion {symbol}: Dự đoán {last_side} ({last_proba*100:.1f}%) | Thực tế: {actual_return*100:+.2f}% | {status} | WinRate(20): {win_rate*100:.0f}% ({warning})"
    except Exception as exc: 
        return f"⚠️ Lỗi Telemetry {symbol}: {exc}"

def detect_market_regime(df):
    hmm_model_local = globals().get("hmm_model", None)
    if hmm_model_local:
        try:
            features = df[["Return", "Volatility_24"]].dropna().values
            if len(features) >= 5:
                regime = int(hmm_model_local.predict(features)[-1])
                return regime if regime in MARKET_REGIME_NAMES else int(_clamp(regime, 0, 2))
        except: pass
    live_row = df.iloc[-1]
    trend_strength, vol_regime = abs(_safe_float(live_row.get("Trend_Strength"), 0.0)), _safe_float(live_row.get("Volatility_Regime"), 1.0)
    if trend_strength < 0.002 and vol_regime <= 0.9: return 0
    if trend_strength >= 0.006 or vol_regime >= 1.6: return 2
    return 1

def check_causal_validity(symbol, df_features, action):
    if symbol == "BTCUSDT": return True, "Tài sản dẫn dắt (Leader)"
    try:
        btc_raw = _get_cached_klines("BTCUSDT", interval=Client.KLINE_INTERVAL_1HOUR, limit=120)
        btc_close = pd.to_numeric(btc_raw["Close"], errors="coerce")
        btc_returns_series = btc_close.pct_change()
        asset_returns_series = pd.to_numeric(df_features["Return"], errors="coerce")
        comparison = pd.DataFrame({
            "asset": asset_returns_series.tail(72).reset_index(drop=True), 
            "btc": btc_returns_series.tail(72).reset_index(drop=True)
        }).dropna()
        if len(comparison) < 24: return True, "Dữ liệu so sánh mỏng"    
        asset_returns, btc_returns = comparison["asset"].values, comparison["btc"].values
        btc_variance = np.var(btc_returns)
        if btc_variance < 1e-8: return True, "BTC đi ngang"
        beta = float(np.cov(asset_returns, btc_returns)[0, 1] / btc_variance)
        corr = float(np.corrcoef(asset_returns, btc_returns)[0, 1])
        residuals = asset_returns - (beta * btc_returns)
        residual_trend = float(np.nanmean(residuals[-6:]))
        if action == "LONG" and corr > 0.70 and beta > 0.60 and residual_trend < -0.0015: 
            return False, f"Yếu hơn BTC (Beta={beta:.2f}, Corr={corr:.2f})"
        if action == "SHORT" and corr > 0.70 and beta > 0.60 and residual_trend > 0.0015: 
            return False, f"Mạnh hơn BTC (Beta={beta:.2f}, Corr={corr:.2f})"
        return True, f"Hợp lệ (Beta={beta:.2f}, Corr={corr:.2f}, Nhiễu={residual_trend:+.4f})"
    except Exception as exc: 
        return True, f"Lỗi Causal: {exc}"

def analyze_order_book(symbol, memory, depth=50): 
    metrics = {
        "ofi": 0.0, "spread_bps": 999.0, "best_bid": 0.0, "best_ask": 0.0, 
        "mid_price": 0.0, "bid_ask_ratio": 1.0, "wall_ratio": 0.0, 
        "sweep_risk": 0.0, "cancel_rate": 0.0
    }
    try:
        order_book = client.get_order_book(symbol=symbol, limit=depth)
        bids = [(float(p), float(s)) for p, s in order_book.get("bids", [])[:depth]]
        asks = [(float(p), float(s)) for p, s in order_book.get("asks", [])[:depth]]
        if not bids or not asks: return metrics
        best_bid, best_ask = bids[0][0], asks[0][0]
        mid_price = (best_bid + best_ask) / 2.0
        bid_vol_total = sum(s for _, s in bids)
        ask_vol_total = sum(s for _, s in asks)
        raw_imbalance = (bid_vol_total - ask_vol_total) / max(bid_vol_total + ask_vol_total, 1e-9)
        weighted_bid = sum(s / (i + 1) for i, (_, s) in enumerate(bids[:10]))
        weighted_ask = sum(s / (i + 1) for i, (_, s) in enumerate(asks[:10]))
        ofi = _clamp((raw_imbalance * 0.4) + (((weighted_bid - weighted_ask) / max(weighted_bid + weighted_ask, 1e-9)) * 0.6), -1.0, 1.0)
        top5_bid_vol = sum(s for _, s in bids[:5])
        top5_ask_vol = sum(s for _, s in asks[:5])
        sweep_risk = max(0.0, 1.0 - (top5_bid_vol + top5_ask_vol) / max((bid_vol_total + ask_vol_total) * 0.2, 1e-9))
        all_sizes = [s for _, s in bids + asks]
        spread_bps = ((best_ask - best_bid) / max(mid_price, 1e-9)) * 10000.0
        wall_ratio = max(max(s for _, s in bids), max(s for _, s in asks)) / max(float(np.mean(all_sizes)) if all_sizes else 0.0, 1e-9)
        metrics.update({
            "ofi": ofi, "spread_bps": spread_bps, "best_bid": best_bid, 
            "best_ask": best_ask, "mid_price": mid_price, 
            "bid_ask_ratio": bid_vol_total / max(ask_vol_total, 1e-9), 
            "wall_ratio": wall_ratio, "sweep_risk": sweep_risk
        })
        cancel_rate = 0.0
        if len(memory["l2_buffer"]) > 0:
            last_snap = memory["l2_buffer"][-1]
            last_bid_vol = last_snap.get("bid_vol_total", bid_vol_total)
            last_ask_vol = last_snap.get("ask_vol_total", ask_vol_total)
            vol_drop = (max(0, last_bid_vol - bid_vol_total) + max(0, last_ask_vol - ask_vol_total))
            cancel_rate = min(vol_drop / max(last_bid_vol + last_ask_vol, 1e-9), 1.0)
            metrics["cancel_rate"] = cancel_rate
        memory["l2_buffer"].append({
            "ts": time.time(), "mid": mid_price, "spread_bps": spread_bps, 
            "ofi": ofi, "best_bid_vol": bids[0][1], "best_ask_vol": asks[0][1], 
            "bid_vol_total": bid_vol_total, "ask_vol_total": ask_vol_total,
            "wall_ratio": wall_ratio
        }) 
        return metrics
    except Exception as e:
        print(f"Lỗi OrderBook: {e}")
        return metrics

def detect_spoofing_hawkes(symbol, memory, limit=50):
    snapshots = [s for s in memory.get("l2_buffer", []) if isinstance(s, dict)]
    if len(snapshots) < 3: return "CLEAN", 0.0, "Chờ thêm dữ liệu L2"
    bid_sizes, ask_sizes, mids, wall_ratios = (np.array([_safe_float(s.get(k), 0.0) for s in snapshots]) for k in ["best_bid_vol", "best_ask_vol", "mid", "wall_ratio"])
    bid_jump = np.max(np.abs(np.diff(bid_sizes))) / max(np.mean(bid_sizes), 1e-9)
    ask_jump = np.max(np.abs(np.diff(ask_sizes))) / max(np.mean(ask_sizes), 1e-9)
    raw_score = (max(bid_jump, ask_jump) * 0.15) + max(np.max(wall_ratios) - 5.0, 0.0) * 0.10 + max(0.00035 - (np.mean(np.abs(np.diff(mids))) / max(np.mean(mids), 1e-9)), 0.0) * 300.0
    score = _clamp(raw_score, 0.0, 1.0)
    if score >= 0.88:
        side = "ASK (BÁN)" if ask_jump > bid_jump else "BID (MUA)"
        return f"SPOOF_RISK_{side[:3]}", score, f"Rung lắc tường {side} (Điểm={score:.2f})"
    return "CLEAN", score, "Sổ lệnh ổn định"

def calculate_trade_expectancy(win_probability, take_profit_pct, stop_loss_pct):
    adj_win_prob = max(0.0, win_probability - 0.05) 
    loss_prob = 1.0 - adj_win_prob
    expected_profit = adj_win_prob * take_profit_pct
    expected_loss = loss_prob * stop_loss_pct
    expectancy_score = expected_profit - expected_loss
    rr_ratio = take_profit_pct / max(stop_loss_pct, 1e-9)
    if expectancy_score <= 0.0005 or rr_ratio < 1.2:
        return False, expectancy_score, rr_ratio
    return True, expectancy_score, rr_ratio

def fetch_derivatives_and_alt_data(symbol, current_sentiment_score, memory=None):
    sentiment_dir = np.sign(current_sentiment_score) 
    sentiment_sev = abs(current_sentiment_score)    
    funding_rate, ls_ratio, oi_change_24h = 0.0, 1.0, 0.0 
    liq_risk = "SAFE"
    try:
        mark_info = client.futures_mark_price(symbol=symbol)
        if isinstance(mark_info, dict) and 'lastFundingRate' in mark_info:
            funding_rate = float(mark_info.get('lastFundingRate', 0.0))
        ls_data = client.futures_global_longshort_ratio(symbol=symbol, period="5m", limit=1)
        if ls_data:
            ls_ratio = float(ls_data[0]['longShortRatio'])
        oi_data = client.futures_open_interest(symbol=symbol)
        current_oi = float(oi_data['openInterest'])
        if memory is not None:
            last_oi = memory.get("last_oi", current_oi)
            oi_change_24h = (current_oi - last_oi) / max(last_oi, 1e-9)
            memory["last_oi"] = current_oi
    except Exception as e:
        ghi_log(f"⚠️ Lỗi API Phái sinh {symbol}: {e}")
    liq_note = f"Fund: {funding_rate*100:.3f}% | L/S: {ls_ratio:.2f} | ΔOI: {oi_change_24h*100:.2f}%"
    if funding_rate >= 0.0008 and ls_ratio > 2.0:
        liq_risk = "LONG_SQUEEZE"
        liq_note += " (🚨 Báo động đỏ: Đám đông Long quá tải)"
    elif funding_rate <= -0.0008 and ls_ratio < 0.5:
        liq_risk = "SHORT_SQUEEZE"
        liq_note += " (🚨 Báo động đỏ: Đám đông Short quá tải)"
    return sentiment_dir, sentiment_sev, funding_rate, oi_change_24h, liq_risk, liq_note

def apply_meta_labeling(symbol, action, live_row, ob_metrics, sentiment_dir, sentiment_sev, liq_risk, market_regime, meta_prob=1.0):
    if meta_prob < 0.50:
        return False, f"❌ AI Vệ Sĩ từ chối (Độ tin cậy của Setup chỉ đạt {meta_prob*100:.1f}%)"
    ofi = _safe_float(ob_metrics.get("ofi"), 0.0)
    sweep_risk = _safe_float(ob_metrics.get("sweep_risk"), 0.0)
    if action == "LONG" and liq_risk == "LONG_SQUEEZE":
        return False, "❌ Đám đông đang đu đỉnh (Rủi ro Long Squeeze)"
    if action == "SHORT" and liq_risk == "SHORT_SQUEEZE":
        return False, "❌ Đám đông đang bán đáy (Rủi ro Short Squeeze)"
    if sweep_risk > 0.6: 
        return False, f"Rủi ro bị Cá mập Sweep sổ lệnh ({sweep_risk*100:.1f}%)"
    if action == "LONG" and ofi < -0.65:
        return False, f"Dòng lệnh (OFI) xả quá mạnh ({ofi:.2f})"
    if action == "SHORT" and ofi > 0.65:
        return False, f"Dòng lệnh (OFI) gom quá mạnh ({ofi:.2f})"
    return True, f"AI Vệ Sĩ Duyệt ({meta_prob*100:.1f}%) + Cầu dao L2 Sạch"
CALIB_FILE = "model_calibrations.json"

def load_calibrations():
    try:
        with open(CALIB_FILE, "r") as f: return json.load(f)
    except:
        return {}

def save_calibrations(calibs):
    try:
        with open(CALIB_FILE, "w") as f: json.dump(calibs, f)
    except Exception as e:
        ghi_log(f"⚠️ Lỗi lưu Calibration: {e}")

model_calibrations = load_calibrations()

def fit_platt_scaling(y_true, y_pred_proba):
    eps = 1e-7
    y_pred_proba = np.clip(y_pred_proba, eps, 1 - eps)
    log_odds = np.log(y_pred_proba / (1 - y_pred_proba)).reshape(-1, 1)
    lr = LogisticRegression(solver='lbfgs', C=1.0)
    try:
        lr.fit(log_odds, y_true)
        return float(lr.coef_[0][0]), float(lr.intercept_[0])
    except:
        return 1.0, 0.0 

def calibrate_xgboost_probability(raw_prob, A=1.0, B=0.0):
    if raw_prob <= 0.0 or raw_prob >= 1.0: return raw_prob
    log_odds = np.log(raw_prob / (1 - raw_prob))
    calibrated_log_odds = A * log_odds + B
    calibrated_prob = 1 / (1 + np.exp(-calibrated_log_odds))
    return float(calibrated_prob)

def analyze_multi_timeframe(symbol):
    try:
        df_4h = _get_cached_klines(symbol, Client.KLINE_INTERVAL_4HOUR, limit=60)
        close_4h = pd.to_numeric(df_4h["Close"])
        ema20 = close_4h.ewm(span=20, adjust=False).mean().iloc[-1]
        ema50 = close_4h.ewm(span=50, adjust=False).mean().iloc[-1]
        price_4h = close_4h.iloc[-1] 
        if ema20 > ema50 and price_4h > ema50: bias_4h = "BULLISH"
        elif ema20 < ema50 and price_4h < ema50: bias_4h = "BEARISH"
        else: bias_4h = "NEUTRAL"
        df_15m = _get_cached_klines(symbol, Client.KLINE_INTERVAL_15MINUTE, limit=60)
        close_15m = pd.to_numeric(df_15m["Close"])
        delta = close_15m.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / loss
        rsi_15m = 100 - (100 / (1 + rs)).iloc[-1]
        return bias_4h, rsi_15m
    except Exception as e:
        ghi_log(f"Lỗi MTF {symbol}: {e}")
        return "NEUTRAL", 50.0

# 2. TẢI HỢP ĐỒNG TRỌNG SỐ (ENSEMBLE WEIGHTS)
try:
    LIVE_WEIGHTS = joblib.load("ensemble_weights_v8.pkl")
    print("✅ Đã load Artifact: ensemble_weights_v8.pkl")
except Exception as e:
    print(f"⚠️ Lỗi load Weights: {e}.")
    LIVE_WEIGHTS = {}

def get_live_ensemble_weights(direction, regime_val):
    """Lấy trọng số ghép cặp XGB/DL chính xác theo từng Regime"""
    if not LIVE_WEIGHTS: 
        return {"xgb": 1.0, "dl": 0.0} 
    regime_str = str(int(regime_val)) if pd.notna(regime_val) else "base"
    if regime_str not in LIVE_WEIGHTS[direction.upper()]:
        regime_str = "base"
    return LIVE_WEIGHTS[direction.upper()][regime_str]

def apply_mtf_gatekeeper(action, bias_4h, rsi_15m):
    if action == "LONG":
        if rsi_15m >= 80: 
            return False, f"BÁC BỎ: FOMO Đỉnh 15m (RSI={rsi_15m:.1f})"
        if bias_4h == "BEARISH":
            return True, f"Cảnh báo: Đánh ngược sóng 4H ({bias_4h}) - Ăn sóng hồi"
        return True, "Hợp lưu MTF (Đồng thuận)"
    elif action == "SHORT":
        if rsi_15m <= 20: 
            return False, f"BÁC BỎ: Hoảng loạn Đáy 15m (RSI={rsi_15m:.1f})"
        if bias_4h == "BULLISH":
            return True, f"Cảnh báo: Đánh ngược sóng 4H ({bias_4h}) - Ăn sóng hồi"
        return True, "Hợp lưu MTF (Đồng thuận)"

def _build_trade_candidate(symbol, action, raw_proba, live_row, raw_sentiment, market_regime, 
                           causal_ok, causal_note, ob_metrics, hawkes_status, hawkes_score, 
                           squeeze_status, squeeze_note, mtf_ok, mtf_note, 
                           liq_risk, sentiment_dir, sentiment_sev, shap_pushers=[], meta_prob=1.0):
    threshold, setup_key, setup_score_key = (LONG_PROBA_THRESHOLD, "setup_long_candidate", "Setup_Long_Score") if action == "LONG" else (SHORT_PROBA_THRESHOLD, "setup_short_candidate", "Setup_Short_Score")
    trend_strength = _safe_float(live_row.get("Trend_Strength"))
    vol_regime = _safe_float(live_row.get("Volatility_Regime"), 1.0)
    ofi = _safe_float(ob_metrics.get("ofi"))
    spread_bps = _safe_float(ob_metrics.get("spread_bps"), 999.0)
    setup_ok = bool(live_row.get(setup_key, 0))
    setup_score = int(_safe_float(live_row.get(setup_score_key), 0))
    fused_proba, notes = apply_cmc_fusion(raw_proba, raw_sentiment, trend_strength, vol_regime, f"OPEN_{action}"), []
    dist_to_demand = _safe_float(live_row.get("Dist_to_Demand"), 999.0)
    dist_to_supply = _safe_float(live_row.get("Dist_to_Supply"), 999.0)
    ob_boost = 0.0
    if action == "LONG" and 0.0 <= dist_to_demand <= 0.015:
        ob_boost = 0.15 # Buff ngay 15% xác suất
        notes.append(f"🐋 Hợp lưu Cá Mập: Chạm vùng Cầu (Demand Zone) - Cửa bật cực cao!")
    elif action == "SHORT" and 0.0 <= dist_to_supply <= 0.015:
        ob_boost = 0.15
        notes.append(f"🐋 Hợp lưu Cá Mập: Chạm đỉnh Bán (Supply Zone) - Cửa sập cực cao!")
    fused_proba += ob_boost
    final_proba = _clamp(fused_proba, 0.0, 0.999)
    atr_pct = _safe_float(live_row.get("ATR_Pct"), 0.02)
    dynamic_sl_pct = _clamp(atr_pct * 1.5, 0.01, 0.06) 
    dynamic_tp_pct = dynamic_sl_pct * 2.0
    is_expectancy_positive, exp_score, rr_ratio = calculate_trade_expectancy(final_proba, dynamic_tp_pct, dynamic_sl_pct)
    if is_expectancy_positive: notes.append(f"Kỳ vọng DƯƠNG (+{exp_score*100:.2f}%)")
    else: notes.append(f"Kỳ vọng ÂM ({exp_score*100:.2f}%) -> ÉP NO_TRADE")     
    meta_approved, meta_reason = apply_meta_labeling(
        symbol, action, live_row, ob_metrics, 
        sentiment_dir, sentiment_sev, liq_risk, market_regime
    )
    if not meta_approved: notes.append(f"Meta-Model Bác bỏ: {meta_reason}")
    else: notes.append(f"Meta-Model: {meta_reason}")
    if not mtf_ok: notes.append(f"MTF Bác bỏ: {mtf_note}")
    else: notes.append(f"MTF: {mtf_note}")
    if not causal_ok: notes.append(f"Causal Bác bỏ: {causal_note}")
    edge_score = (final_proba - threshold) + (setup_score - 5) * 0.01 - (hawkes_score * 0.05)
    is_smart_entry = False
    if final_proba >= 0.70 and setup_score >= 3:
        is_smart_entry = True
        notes.append("🎯 Đột phá: AI Tự tin gánh Setup")   
    elif final_proba >= 0.55 and setup_score >= 7:
        is_smart_entry = True
        notes.append("📊 Đột phá: Setup đẹp gánh AI")
    elif final_proba >= 0.60 and setup_score >= 5:
        is_smart_entry = True
    if meta_prob < 0.50:
        action = "NO_TRADE"  
        notes.append(f"AI Vệ Sĩ Tầng 2 BÁC BỎ (Xác suất thắng chỉ {meta_prob*100:.1f}%)")
    else:
        notes.append(f"AI Vệ Sĩ Tầng 2 DUYỆT ({meta_prob*100:.1f}%)")
    should_open = (
        is_smart_entry 
        and is_expectancy_positive    # Bắt buộc: Toán học phải có lãi
        and mtf_ok                    # Bắt buộc: Không đánh ngược trend 4H
        and meta_approved             # Bắt buộc: Heuristic vệ sĩ cũ
        and causal_ok                 # Bắt buộc: Phải có tính nhân quả
        and (meta_prob >= 0.50)       # <--- THÊM ĐIỀU KIỆN: AI Vệ Sĩ ML phải đồng ý
    )
    return {
        "symbol": symbol, "action": action, "raw_proba": raw_proba, "final_proba": final_proba, "proba": final_proba, "threshold": threshold,
        "should_open": should_open,
        "edge_score": edge_score, "setup_score": setup_score, "setup_ok": setup_ok, "raw_sentiment": raw_sentiment, "market_regime": market_regime,
        "order_book_ofi": ofi, "spread_bps": spread_bps, "causal_ok": causal_ok, "causal_note": causal_note, "hawkes_status": hawkes_status,
        "hawkes_score": hawkes_score, "squeeze_status": squeeze_status, "squeeze_note": squeeze_note, "notes": notes,
        "entry_price": _safe_float(ob_metrics.get("mid_price"), _safe_float(live_row.get("Close"), 0.0)),
        "entry_features": _serialize_feature_dict(live_row.to_dict()), "signal_target": "standard" if final_proba >= threshold + 0.06 and setup_score >= 6 else "basic",
        "shap_pushers": shap_pushers,
        "meta_prob": meta_prob 
    }

# ✅ Đã thêm tham số direction="LONG" vào signature
def evaluate_risk_adjusted_ev(meta_features_df, direction="LONG"):
    """
    Đo lường Kỳ vọng Lợi nhuận (EV) và Rủi ro (Uncertainty).
    """
    try:
        if not LIVE_META_ARTIFACT or not isinstance(LIVE_META_ARTIFACT, dict):
            return False, 0.0, 0.0, 0.0
            
        suffix = '_l' if direction == "LONG" else '_s'
        
        # =========================================================
        # 🛡️ BẢN VÁ: TƯƠNG THÍCH NGƯỢC (FALLBACK)
        # Cố tìm não riêng (ev_model_l), nếu chưa đúc kịp thì lấy tạm não chung (ev_model)
        # =========================================================
        ev_model = LIVE_META_ARTIFACT.get(f"ev_model{suffix}", LIVE_META_ARTIFACT.get("ev_model"))
        unc_model = LIVE_META_ARTIFACT.get(f"uncertainty_model{suffix}", LIVE_META_ARTIFACT.get("uncertainty_model"))
        
        if ev_model is None or unc_model is None:
            return False, 0.0, 0.0, 0.0
            
        expected_pnl = float(ev_model.predict(meta_features_df)[0])
        uncertainty = float(unc_model.predict(meta_features_df)[0])
        
        RISK_PENALTY_LAMBDA = 0.5 
        ev_adjusted = expected_pnl - (RISK_PENALTY_LAMBDA * uncertainty)
        MIN_ACCEPTABLE_EV = 0.0015 
        
        is_approved = ev_adjusted > MIN_ACCEPTABLE_EV
        return is_approved, expected_pnl, uncertainty, ev_adjusted
        
    except Exception as e:
        print(f"⚠️ CẢNH BÁO TẦNG 2 (EV Gate): {e} -> Ép NO_TRADE!")
        return False, 0.0, 0.0, 0.0

def _select_trade_candidate(long_candidate, short_candidate):
    tradable = []
    for c in [long_candidate, short_candidate]:
        if not c: continue
        if c.get("should_open", False): 
            tradable.append(c)
    if not tradable: 
        return None
    tradable.sort(key=lambda item: (item.get("edge_score", -1.0), item.get("final_proba", 0.0)), reverse=True)
    if len(tradable) >= 2 and abs(tradable[0].get("final_proba", 0) - tradable[1].get("final_proba", 0)) < 0.03:
        return None
    return tradable[0]

def _calculate_position_pnl_pct(side, entry_price, current_price): return (current_price - entry_price) / max(entry_price, 1e-9) if side == "LONG" else (entry_price - current_price) / max(entry_price, 1e-9) if side == "SHORT" else 0.0

def _close_real_position(symbol, memory, current_price, reason):
    side = memory.get("position_side", "NONE")
    quantity = memory.get("quantity", 0.0) # Khối lượng đã khớp lúc mở lệnh
    entry_price = _safe_float(memory.get("entry_price"), 0.0)
    invested_usdt = _safe_float(memory.get("invested_usdt"), 0.0)
    target = memory.get("last_signal_target", "standard")
    shap_drivers = memory.get("shap_entry_drivers", [])
    if side == "NONE" or quantity <= 0: 
        return None, None, ""
    close_side = "SELL" if side == "LONG" else "BUY"
    try:
        order = client.futures_create_order(
            symbol=symbol,
            side=close_side,
            type='MARKET',
            quantity=quantity,
            reduceOnly=True 
        )
        ghi_log(f"🛑 [API THỰC CHIẾN] Đã Khớp lệnh ĐÓNG {side} {symbol} | Lý do: {reason}")
    except Exception as e:
        ghi_log(f"🚨 [API LỖI FATAL] Đóng lệnh {symbol} thất bại: {e}. PPO sẽ tự động thử lại!")
        return None 
    pnl_pct = _calculate_position_pnl_pct(side, entry_price, current_price)
    shap_str = ""
    if shap_drivers:
        feature_names = [f"{f} (+{v:.3f})" for f, v in shap_drivers]
        if pnl_pct > 0:
            shap_str = f"\n🎖️ BẢNG VÀNG LẬP CÔNG: {', '.join(feature_names)}"
            ghi_log(f"🟢 [SHAP ATTRIBUTION] {symbol} THẮNG. Kẻ lập công: {', '.join([f for f, v in shap_drivers])}")
        else:
            shap_str = f"\n🔪 KẺ PHẢN BỘI (TRAPPED): {', '.join(feature_names)}"
            ghi_log(f"🔴 [SHAP ATTRIBUTION] {symbol} THUA. Rút kinh nghiệm từ kẻ lừa đảo: {', '.join([f for f, v in shap_drivers])}")
    trade_summary = {
        "symbol": symbol, "side": side, "entry_price": entry_price, 
        "exit_price": current_price, "pnl_pct": pnl_pct, 
        "pnl_usdt": invested_usdt * pnl_pct, "reason": reason, 
        "closed_at": str(_utc_now())
    }
    l2_history = memory.get("l2_buffer", deque(maxlen=10))
    memory.clear()
    memory.update(_symbol_state_defaults())
    memory.update({
        "l2_buffer": l2_history, "last_trade": trade_summary, 
        "last_signal_target": target, "last_signal_reason": reason, 
        "last_scan_price": current_price
    })
    entry_features = memory.get("entry_features") or {}
    if "market_regime" in entry_features:
        regime = memory["entry_features"]["Market_Regime"]
        risk_engine.update_governor_pnl(regime, side, pnl_pct)
    return trade_summary, target, shap_str

try:
    exit_model_ai = joblib.load("xgb_v8_exit_model.pkl")
except:
    exit_model_ai = None

def ppo_exit_action_v2(symbol, memory, current_price, ob_metrics, live_row):
    side = memory.get("position_side", "NONE")
    entry_price = _safe_float(memory.get("entry_price", current_price))
    bars_held = int(memory.get("bars_held", 0))
    pnl_pct = _calculate_position_pnl_pct(side, entry_price, current_price)
    ofi = _safe_float(ob_metrics.get("ofi"), 0.0)
    atr_pct = _safe_float(live_row.get("ATR_Pct"), 0.02)
    if atr_pct == 0.0: atr_pct = 0.02
    dynamic_tp = _clamp(atr_pct * 3.0, 0.02, 0.10) 
    dynamic_sl = _clamp(atr_pct * 1.5, 0.01, 0.05)
    ai_expected_pnl = None
    if 'exit_model_ai' in globals() and isinstance(exit_model_ai, dict) and "q_0.5" in exit_model_ai:
        try:
            if "feature_columns_v8" in globals():
                X_exit = pd.DataFrame([live_row])[feature_columns_v8].astype(np.float32)
                pred_q50 = float(exit_model_ai["q_0.5"].predict(X_exit)[0])
                ai_expected_pnl = pred_q50
        except Exception as e:
            ghi_log(f"⚠️ Lỗi Inference AI Exit: {e}")
    if pnl_pct <= -dynamic_sl:
        return "PANIC_EXIT", f"Cắt lỗ động Quantile (-{dynamic_sl*100:.2f}%)"
    if pnl_pct >= dynamic_tp:
        return "PANIC_EXIT", f"Đạt Target Quantile tối ưu (+{dynamic_tp*100:.2f}%)"
    if ai_expected_pnl is not None:
        if side == "LONG" and ai_expected_pnl < -0.005:
            return "PANIC_EXIT", f"AI Exit dự báo giá giảm (Exp PnL: {ai_expected_pnl*100:.2f}%)"     
        elif side == "SHORT" and ai_expected_pnl > 0.005:
            return "PANIC_EXIT", f"AI Exit dự báo giá bơm (Exp PnL: {ai_expected_pnl*100:.2f}%)"
    if pnl_pct > 0.015: 
        if (side == "LONG" and ofi < -0.5) or (side == "SHORT" and ofi > 0.5):
            return "PANIC_EXIT", f"Chốt lời bảo vệ: Dòng lệnh L2 đảo ngược (OFI: {ofi:.2f})"
    return "HOLD", "Kỳ vọng tăng trưởng vẫn xanh"

def _evaluate_open_position(symbol, memory, current_price, ob_metrics, live_row):
    if memory.get("position_side", "NONE") == "NONE":
        return None
    time_in_trade = int(memory.get("bars_held", 0)) + 1
    memory["bars_held"] = time_in_trade
    entry_price = _safe_float(memory.get("entry_price"), current_price)
    if current_price > memory.get("peak_price", entry_price): memory["peak_price"] = current_price
    if current_price < memory.get("trough_price", entry_price): memory["trough_price"] = current_price
    action, ppo_reason = ppo_exit_action_v2(symbol, memory, current_price, ob_metrics, live_row)
    pnl_pct = _calculate_position_pnl_pct(memory["position_side"], entry_price, current_price)
    if action == "HOLD" and time_in_trade > 48 and pnl_pct < 0.005:
        action = "PANIC_EXIT"
        ppo_reason = "Time Stop: Ngâm vốn quá 48H không bay nổi"
    if action == "PANIC_EXIT":
        return _close_real_position(symbol, memory, current_price, f"🤖 VỆ SĨ PPO: {ppo_reason}")
    return None

def _open_real_position(signal):
    symbol = signal["symbol"]
    action = signal["action"]
    entry_price = _safe_float(signal.get("entry_price"), 0.0)   
    if entry_price <= 0.0: return None
    try:
        account_info = client.futures_account()
        available_margin = float(account_info['availableBalance'])
        total_wallet = float(account_info['totalWalletBalance'])
        LEVERAGE = 15
        win_prob = signal.get("final_proba", 0.55)
        entry_features = signal.get("entry_features", {})
        atr_pct = _safe_float(entry_features.get("ATR_Pct"), 0.02)
        sl_pct = max(min(atr_pct * 1.5, 0.06), 0.01) 
        tp_pct = sl_pct * 2.0
        R_ratio = tp_pct / sl_pct
        setup_score = signal.get("setup_score", 5)
        uncertainty_penalty = 0.08 - (setup_score * 0.005) 
        adj_prob = max(0.01, win_prob - uncertainty_penalty)
        raw_kelly = adj_prob - ((1.0 - adj_prob) / R_ratio)
        safe_kelly = max(0.0, raw_kelly * 0.25)
        dynamic_risk_pct = max(0.005, min(safe_kelly, 0.05))
        max_risk_usd = total_wallet * dynamic_risk_pct
        target_notional = max_risk_usd / sl_pct
        ghi_log(f"🧠 [KELLY V2] {symbol}: Raw={win_prob*100:.1f}% | Adj={adj_prob*100:.1f}% | Size: {dynamic_risk_pct*100:.2f}% (${max_risk_usd:.2f})")
        exchange_info = client.futures_exchange_info()
        symbol_rules = next((item for item in exchange_info['symbols'] if item['symbol'] == symbol), None)
        if not symbol_rules:
            ghi_log(f"⚠️ Không tìm thấy luật lệ cho {symbol}.")
            return None
        min_qty, step_size, min_notional = 0.0, 0.0, 5.0
        for f in symbol_rules['filters']:
            if f['filterType'] == 'LOT_SIZE':
                min_qty = float(f['minQty'])
                step_size = float(f['stepSize'])
            elif f['filterType'] == 'MIN_NOTIONAL':
                min_notional = float(f.get('notional', 5.0))
        actual_notional = max(target_notional, min_notional * 1.05)
        required_margin = actual_notional / LEVERAGE
        if required_margin > available_margin:
            ghi_log(f"🛡️ [PRE-FLIGHT] HỦY {symbol}: Cần {required_margin:.2f}$ cọc, nhưng ví chỉ còn {available_margin:.2f}$.")
            return None
        raw_quantity = actual_notional / entry_price
        precision = max(0, int(round(-math.log(step_size, 10), 0)))
        quantity = round(math.floor(raw_quantity / step_size) * step_size, precision)
        if quantity < min_qty:
            ghi_log(f"🛡️ [PRE-FLIGHT] HỦY {symbol}: Khối lượng {quantity} nhỏ hơn mức sàn cho phép ({min_qty}).")
            return None
        recalc_notional = quantity * entry_price
        if recalc_notional < min_notional:
            ghi_log(f"🛡️ [PRE-FLIGHT] HỦY {symbol}: Giá trị thực tế {recalc_notional:.2f}$ < Min Notional ({min_notional}$).")
            return None
        side_str = "BUY" if action == "LONG" else "SELL"
        if is_approved:
            # 1. 🛡️ GỌI BANDIT RA QUYẾT ĐỊNH VỐN DỰA TRÊN PHONG ĐỘ (EDGE)
            current_regime = live_row.get('Market_Regime', 0) 
            DYNAMIC_USD = get_bandit_allocation(symbol, current_regime, action)
            TRADE_SIZE_USDT = DYNAMIC_USD 
            ghi_log(f"🎰 [BANDIT ALLOCATOR] Cấp vốn cho {symbol} ({action} | Regime {int(current_regime)}): ${TRADE_SIZE_USDT:.2f}")
            # 2. ⚠️ QUAN TRỌNG: TÍNH LẠI SỐ LƯỢNG COIN (QUANTITY)
            quantity = TRADE_SIZE_USDT / entry_price 
            recalc_notional = TRADE_SIZE_USDT 
            # 3. GỌI CẢNH SÁT THỰC THI (GATEKEEPER)
            exec_ok, exec_msg, order_type, route_price = execution_gatekeeper(
                client=client, 
                symbol=symbol, 
                direction=action,
                trade_size_usd=TRADE_SIZE_USDT, 
                expected_ev=ev_adjusted
            )
            print(exec_msg)
            if not exec_ok:
                ghi_log(f"🛑 [GATEKEEPER HỦY KÈO] {symbol} - Lý do: {exec_msg}")
                return None 
            print("🚀 GỬI LỆNH LÊN BINANCE...")
            if order_type == "MARKET":
                order = client.futures_create_order(
                    symbol=symbol,
                    side=side_str,
                    type='MARKET',
                    quantity=quantity
                )
            elif order_type == "LIMIT":
                order = client.futures_create_order(
                    symbol=symbol,
                    side=side_str,
                    type='LIMIT',
                    price=str(route_price), # Ép kiểu chuỗi cho chắc ăn với API
                    timeInForce='GTC', # Bắt buộc có tham số này với lệnh Limit
                    quantity=quantity
                )
        else:
            ghi_log(f"🛡️ [TẦNG 2 TỪ CHỐI] {symbol} do EV Adjusted quá thấp.")
            return None
        ghi_log(f"🚀 [API THỰC CHIẾN] Khớp {action} {symbol} | Vol: {quantity} | Trị giá: ~${recalc_notional:.2f}")
        memory = _ensure_symbol_state(symbol)
        memory.update({
            "position_side": action, 
            "entry_regime": live_row.get('Market_Regime', 0), 
            "entry_price": entry_price, 
            "quantity": quantity, 
            "invested_usdt": recalc_notional, 
            "peak_price": entry_price, 
            "trough_price": entry_price, 
            "opened_at": str(_utc_now()), 
            "entry_features": signal.get("entry_features"), 
        })
        return memory        
    except Exception as e:
        ghi_log(f"🚨 [API TỪ CHỐI] Lỗi bất ngờ khi mở lệnh {symbol}: {e}")
        return None

def _format_candidate_summary(c): return f"{c['action']} Gốc={c['raw_proba']*100:.1f}% DungHợp={c['final_proba']*100:.1f}% Ngưỡng={c['threshold']*100:.1f}% Setup={c['setup_score']} OFI={c['order_book_ofi']:+.2f} Spread={c['spread_bps']:.1f}bps Hawkes={c['hawkes_score']:.2f} Causal={'HỢP LỆ' if c['causal_ok'] else 'BÁC BỎ'}"

def load_dynamic_weights():
    try:
        with open("ensemble_weights.json", "r") as f: return json.load(f)
    except:
        return {"0": {"XGB": 1.0, "LSTM": 0.0}, "1": {"XGB": 0.8, "LSTM": 0.2}, "2": {"XGB": 0.5, "LSTM": 0.5}}

ensemble_weights_dict = load_dynamic_weights()

def hybrid_ensemble_predict(xgb_model, dl_model, df_tabular_features, df_raw_history, market_regime=1, direction="LONG"):
    for col in feature_columns_v8:
        if col not in df_tabular_features.columns:
            df_tabular_features[col] = 0.0
            
    X_live_tabular = df_tabular_features[feature_columns_v8].iloc[[-1]].astype(np.float32)
    xgb_prob = float(xgb_model.predict_proba(X_live_tabular)[0][1]) if xgb_model else 0.0 
    DL_SEQUENCE_LENGTH = 128
    try:
        weights = get_live_ensemble_weights(direction, market_regime) 
        weight_xgb = float(weights.get("xgb", weights.get("XGB", 1.0)))
        weight_dl = float(weights.get("dl", weights.get("LSTM", 0.0)))
    except Exception as e:
        print(f"⚠️ Cảnh báo Load Weights ({direction}-{market_regime}): {e}. Default về 100% XGBoost.")
        weight_xgb, weight_dl = 1.0, 0.0
    if dl_model is None or len(df_raw_history) < DL_SEQUENCE_LENGTH or weight_dl <= 0.0:
        return xgb_prob, xgb_prob, 0.0  
    try:
        df_dl = df_raw_history.copy()
        for col in ["Open", "High", "Low", "Close", "Volume"]:
            df_dl[col] = pd.to_numeric(df_dl[col], errors='coerce') 
        df_dl["Log_Return"] = np.log(df_dl["Close"] / df_dl["Close"].shift(1))
        df_dl["High_Low_Spread"] = (df_dl["High"] - df_dl["Low"]) / df_dl["Low"]
        df_dl["Close_Open_Spread"] = (df_dl["Close"] - df_dl["Open"]) / df_dl["Open"]
        df_dl["Volume_Log"] = np.log1p(df_dl["Volume"])
        df_dl["Volatility_24"] = df_dl["Log_Return"].rolling(24).std()
        df_dl = df_dl.fillna(0.0) 
        DL_FEATURES = ["Log_Return", "High_Low_Spread", "Close_Open_Spread", "Volume_Log", "Volatility_24"]
        seq_data = df_dl[DL_FEATURES].tail(DL_SEQUENCE_LENGTH).astype(np.float32).values
        seq_data_scaled = (seq_data - np.mean(seq_data, axis=0)) / (np.std(seq_data, axis=0) + 1e-9) 
        X_live_seq = seq_data_scaled.reshape(1, DL_SEQUENCE_LENGTH, len(DL_FEATURES))
        dl_prob = float(dl_model.predict(X_live_seq, verbose=0)[0][0])
        hybrid_prob = (xgb_prob * weight_xgb) + (dl_prob * weight_dl)
        return hybrid_prob, xgb_prob, dl_prob
    except Exception as e:
        ghi_log(f"Lỗi Inference Deep Learning: {e}")
        return xgb_prob, xgb_prob, 0.0

def evaluate_risk_adjusted_ev(df_meta, direction):
    """
    Hàm lõi đánh giá Tầng 2: Trả về (is_approved, raw_ev, uncertainty, ev_adj)
    """
    try:
        # Nếu chưa load model Tầng 2, từ chối luôn
        if 'ev_model' not in globals() or 'uncertainty_model' not in globals():
            return False, 0.0, 0.0, 0.0
            
        raw_ev = ev_model.predict(df_meta)[0]
        unc = uncertainty_model.predict(df_meta)[0]
        ev_adj = raw_ev - (0.5 * unc) # Khấu trừ rủi ro
        
        is_approved = bool(ev_adj > 0.0)
        return is_approved, float(raw_ev), float(unc), float(ev_adj)
    except Exception as e:
        print(f"⚠️ Lỗi Tầng 2 ({direction}): {e}")
        return False, 0.0, 0.0, 0.0
                
        # ==========================================
        # 📝 GHI SỔ VÀ IN LOG (Đã nắn thẳng lề)
        # ==========================================
        scan_results.append({
            "symbol": sym, 
            "ev_long": ev_adj_l if is_approved_l else 0.0,
            "ev_short": ev_adj_s if is_approved_s else 0.0,
            "residual_alpha": _safe_float(live_row.get("Residual_Alpha"), 0.0),
            "market_regime": market_regime
        })

        if is_approved_l:
            print(f"🔥 [VỆ SĨ LONG] {sym} DUYỆT! Lãi kỳ vọng: {exp_pnl_l*100:.2f}% | Rủi ro: {unc_l*100:.2f}% | Lãi Ròng Chắc Chắn: {ev_adj_l*100:.2f}%")
        if is_approved_s:
            print(f"🔥 [VỆ SĨ SHORT] {sym} DUYỆT! Lãi kỳ vọng: {exp_pnl_s*100:.2f}% | Rủi ro: {unc_s*100:.2f}% | Lãi Ròng Chắc Chắn: {ev_adj_s*100:.2f}%")
            
        if is_regime_profitable("LONG", market_regime) and is_approved_l:
            long_c = _build_trade_candidate(sym, "LONG", raw_long, live_row, raw_sentiment, market_regime, long_causal_ok, "", ob_metrics, hawkes_status, hawkes_score, squeeze_status, squeeze_note, mtf_long_ok, mtf_long_note, liq_risk, sentiment_dir, sentiment_sev, shap_pushers_l, ev_adj_l)
        else:
            long_c = None
            
        if is_regime_profitable("SHORT", market_regime) and is_approved_s:
            short_c = _build_trade_candidate(sym, "SHORT", raw_short, live_row, raw_sentiment, market_regime, short_causal_ok, "", ob_metrics, hawkes_status, hawkes_score, squeeze_status, squeeze_note, mtf_short_ok, mtf_short_note, liq_risk, sentiment_dir, sentiment_sev, shap_pushers_s, ev_adj_s)
        else:
            short_c = None

    # 🎯 'except' tổng đứng thẳng với 'try' to nhất trên cùng (ngoài khối này)
    except Exception as e:
        ghi_log(f"⚠️ Lỗi quét {sym}: {e}")
        # --- XÂY DỰNG HỒ SƠ LỆNH RIÊNG BIỆT ---
        # Lúc này long_causal_ok và ob_metrics đã tồn tại và sẵn sàng được đóng gói!
        long_c = _build_trade_candidate(
            symbol, "LONG", final_proba_l, live_row, raw_sentiment, 
            market_regime, long_causal_ok, "", ob_metrics, 
            hawkes_status, hawkes_score, squeeze_status, squeeze_note, 
            mtf_long_ok, mtf_long_note, liq_risk, 
            sentiment_dir, sentiment_sev, shap_pushers_l, meta_prob_l
        )
        
        short_c = _build_trade_candidate(
            symbol, "SHORT", final_proba_s, live_row, raw_sentiment, 
            market_regime, short_causal_ok, "", ob_metrics, 
            hawkes_status, hawkes_score, squeeze_status, squeeze_note, 
            mtf_short_ok, mtf_short_note, liq_risk, 
            sentiment_dir, sentiment_sev, shap_pushers_s, meta_prob_s
        )
        
        # =======================================================
        # 🎯 [TẦNG 3] AI QUANTILE THOÁT LỆNH & TÍNH KỲ VỌNG
        # =======================================================
        expected_pnl_l, tp_pct_l, sl_pct_l = 0.0, 0.02, 0.01
        expected_pnl_s, tp_pct_s, sl_pct_s = 0.0, 0.02, 0.01
        
        if 'exit_model_ai' in globals() and exit_model_ai is not None:
            try:
                X_exit = df_features[feature_columns_v8].iloc[[-1]].astype(np.float32)
                pred_q10 = float(exit_model_ai["q_0.1"].predict(X_exit)[0])
                pred_q50 = float(exit_model_ai["q_0.5"].predict(X_exit)[0])
                pred_q90 = float(exit_model_ai["q_0.9"].predict(X_exit)[0])
                
                pred_q10 = max(min(pred_q10, 0.30), -0.30)
                pred_q50 = max(min(pred_q50, 0.30), -0.30)
                pred_q90 = max(min(pred_q90, 0.30), -0.30)
                
                # Tính kịch bản cho LONG
                sl_pct_l = abs(min(pred_q10, -0.005)) 
                tp_pct_l = max(pred_q90, 0.01)        
                expected_pnl_l = pred_q50             
                
                # Tính kịch bản cho SHORT
                sl_pct_s = abs(max(pred_q90, 0.005))  
                tp_pct_s = abs(min(pred_q10, -0.01))
                expected_pnl_s = -pred_q50            
            except Exception as e:
                ghi_log(f"⚠️ Lỗi AI Quantile: {e}")
        
        # Sửa thành meta_prob_l
        long_c = _build_trade_candidate(symbol, "LONG", raw_long, live_row, raw_sentiment, market_regime, long_causal_ok, "", ob_metrics, hawkes_status, hawkes_score, squeeze_status, squeeze_note, mtf_long_ok, mtf_long_note, liq_risk, sentiment_dir, sentiment_sev, shap_pushers_l, meta_prob_l) 
        short_c = _build_trade_candidate(symbol, "SHORT", raw_short, live_row, raw_sentiment, market_regime, short_causal_ok, "", ob_metrics, hawkes_status, hawkes_score, squeeze_status, squeeze_note, mtf_short_ok, mtf_short_note, liq_risk, sentiment_dir, sentiment_sev, shap_pushers_s, meta_prob_s)

        # --- NHÚNG SỨC MẠNH QUANTILE VÀO PHÁN QUYẾT TẦNG 4 ---
        # Lưu lại mức Cắt lỗ / Chốt lời động để Lò phản ứng Kelly dùng sau này
        long_c["dynamic_sl"] = sl_pct_l
        long_c["dynamic_tp"] = tp_pct_l
        long_c["expected_pnl"] = expected_pnl_l
        short_c["dynamic_sl"] = sl_pct_s
        short_c["dynamic_tp"] = tp_pct_s
        short_c["expected_pnl"] = expected_pnl_s

        # CẦU DAO KỲ VỌNG (EXPECTED VALUE)
        if expected_pnl_l < 0.002:  # Đánh lên mà biên lợi nhuận < 0.2% thì vứt
            long_c["action"] = "NO_TRADE"
            if "notes" in long_c: long_c["notes"].append(f"AI Quantile Cảnh báo: Lợi nhuận kỳ vọng LONG quá thấp ({expected_pnl_l*100:.2f}%)")
            
        if expected_pnl_s < 0.002:  # Đánh xuống mà biên lợi nhuận < 0.2% thì vứt
            short_c["action"] = "NO_TRADE"
            if "notes" in short_c: short_c["notes"].append(f"AI Quantile Cảnh báo: Lợi nhuận kỳ vọng SHORT quá thấp ({expected_pnl_s*100:.2f}%)")

        # =======================================================
        close_msg_clean = ""
        if black_swan_triggered and memory["position_side"] != "NONE":
            trade_summary, target, _ = _close_real_position(symbol, memory, current_price, "BÃO THANH KHOẢN (BLACK SWAN)")
            if trade_summary:
                close_msg_clean = f"ĐÓNG LỆNH KHẨN CẤP ({trade_summary['side']}) | Lãi/Lỗ: {trade_summary['pnl_pct']*100:+.2f}%"
        close_result = _evaluate_open_position(symbol, memory, current_price, ob_metrics, live_row)
        if close_result:
            trade_summary, target, _ = close_result
            close_msg_clean = f"ĐÓNG LỆNH PPO ({trade_summary['side']}) | Lý do: {trade_summary['reason']} | PnL: {trade_summary['pnl_pct']*100:+.2f}%"
        candidate = None
        if memory["position_side"] == "NONE" and not black_swan_triggered:
            candidate = _select_trade_candidate(long_c, short_c)
            
        dominant = long_c if long_c["final_proba"] >= short_c["final_proba"] else short_c
        memory.update({"last_prediction_side": dominant["action"], "last_prediction_proba": dominant["final_proba"], "last_scan_price": current_price})
        ui_log = []
        ui_log.append(f"\n{'='*55}")
        ui_log.append(f"🎯 MỤC TIÊU QUÉT: {symbol} | Giá: {current_price:.4f}")
        
        # TẦNG 1: MÔI TRƯỜNG VĨ MÔ
        ui_log.append("[TẦNG 1] ĐÁNH GIÁ MÔI TRƯỜNG & TIN TỨC")
        ui_log.append(f"  💥 Cầu dao rủi ro: {'🚨 KÍCH HOẠT (BÃO)' if black_swan_triggered else '✅ AN TOÀN'}")
        ui_log.append(f"  🔮 Regime thị trường: {MARKET_REGIME_NAMES.get(market_regime, market_regime)}")
        ui_log.append(f"  📰 Phân tích Báo chí: {'BULL' if sentiment_dir > 0 else 'BEAR' if sentiment_dir < 0 else 'NEUTRAL'} | Điểm số: {raw_sentiment:+.3f}")
        ui_log.append(f"  ⛓️ Phái sinh On-chain: {liq_note} | Trạng thái: {liq_risk}")
        
        # TẦNG 2: AI DỰ BÁO
        ui_log.append("[TẦNG 2] NÃO BỘ DỰ BÁO AI (HYBRID ENSEMBLE)")
        if active_expert:
            ui_log.append(f"  👔 Bộ định tuyến: Giao việc cho [{active_expert['name']}]")
            if pf_long < 1.0:
                ui_log.append(f"  🛑 Cửa LONG: 🔒 BỊ KHÓA BỞI CẦU DAO (Profit Factor {pf_long:.2f} < 1.0)")
            else:
                ui_log.append(f"  🟢 Cửa LONG: Xác suất {long_c['final_proba']*100:.1f}% | Kỹ thuật (Setup): {long_c['setup_score']}/10")
            if pf_short < 1.0:
                ui_log.append(f"  🛑 Cửa SHORT: 🔒 BỊ KHÓA BỞI CẦU DAO (Profit Factor {pf_short:.2f} < 1.0)")
            else:
                ui_log.append(f"  🔴 Cửa SHORT: Xác suất {short_c['final_proba']*100:.1f}% | Kỹ thuật (Setup): {short_c['setup_score']}/10")
        else:
            ui_log.append("  ⚠️ LỖI: Không có Model cho Regime này!")

        # TẦNG 3: GATEKEEPERS
        ui_log.append("[TẦNG 3] MÀNG LỌC VỆ SĨ (GATEKEEPERS)")
        ui_log.append(f"  👁️ Dòng lệnh (OFI): {ob_metrics['ofi']:+.2f} | Tỷ lệ hủy lệnh: {ob_metrics['cancel_rate']*100:.1f}%")
        ui_log.append(f"  📡 Radar Tường giả: {hawkes_status} (Rủi ro: {hawkes_score:.2f})")
        ui_log.append(f"  🗜️ Rủi ro Squeeze: {squeeze_status}")
        ui_log.append(f"  ⏳ Phân tích Đa khung: Sóng 4H [{bias_4h}] | RSI 15m [{rsi_15m:.1f}]")
        ui_log.append(f"  🔗 Tính Nhân quả: LONG [{'HỢP LỆ' if long_causal_ok else 'BÁC BỎ'}] | SHORT [{'HỢP LỆ' if short_causal_ok else 'BÁC BỎ'}]")

        # TẦNG 4: BÙ TRỪ KỲ VỌNG
        ui_log.append("[TẦNG 4] THẨM ĐỊNH CHIẾN LƯỢC & BÙ TRỪ CHÉO")
        # Rút gọn ghi chú để nhìn đỡ rối
        long_notes = ", ".join(long_c.get('notes', ['Không']))
        short_notes = ", ".join(short_c.get('notes', ['Không']))
        ui_log.append(f"  🟢 Thẩm định LONG: {long_notes.replace('Meta-Model:', 'Meta:').replace('Kỳ vọng', 'KV')}")
        ui_log.append(f"  🔴 Thẩm định SHORT: {short_notes.replace('Meta-Model:', 'Meta:').replace('Kỳ vọng', 'KV')}")

        # TẦNG 5: QUYẾT ĐỊNH
        if close_msg_clean:
            ui_log.append(f"🛑 HÀNH ĐỘNG: {close_msg_clean}")
            
        if memory["position_side"] != "NONE":
            pnl_pct = _calculate_position_pnl_pct(memory["position_side"], _safe_float(memory["entry_price"]), current_price)
            ui_log.append(f"👀 TRẠNG THÁI: Tác tử PPO đang gồng lệnh {memory['position_side']} | Lãi/Lỗ: {pnl_pct*100:+.2f}% / ${_safe_float(memory.get('invested_usdt'), 0.0) * pnl_pct:+.2f}")
        else:
            if candidate:
                ui_log.append(f"🚀 PHÁN QUYẾT: MỞ LỆNH THỰC CHIẾN [{candidate['action']}] @ Xác suất {candidate['final_proba']*100:.1f}%")
            else:
                ui_log.append("⚖️ PHÁN QUYẾT: ĐỨNG NGOÀI (NO_TRADE - Bị chặn bởi Gatekeeper)")
                
        ui_log.append(f"{'='*55}\n")
        
        ghi_log("\n".join(ui_log))
        return candidate

    except Exception as e:
        import traceback
        ghi_log(f"\n❌ LỖI HỆ THỐNG ({symbol}): {e}\n{traceback.format_exc()}")
        return None
        
def check_portfolio_correlation(intended_trades, bot_memory):
    active_positions = {sym: data["position_side"] for sym, data in bot_memory.items() if data["position_side"] != "NONE"}
    approved_trades = []
    rejected_trades = []
    for trade in intended_trades:
        sym = trade["symbol"]
        action = trade["action"]
        if sym in ["BTCUSDT", "ETHUSDT"]:
            if ("BTCUSDT" in active_positions and active_positions["BTCUSDT"] == action) or \
               ("ETHUSDT" in active_positions and active_positions["ETHUSDT"] == action):
                rejected_trades.append(f"{sym} (Bị chặn: Đã có lệnh Core {action})")
                continue
        if sym in ["SOLUSDT", "BNBUSDT", "XRPUSDT"]:
            if "BTCUSDT" in active_positions and active_positions["BTCUSDT"] == action:
                if trade["edge_score"] < 0.22:
                    rejected_trades.append(f"{sym} (Bị chặn: Rủi ro tương quan BTC, Edge quá thấp)")
                    continue
        approved_trades.append(trade)
    return approved_trades, rejected_trades

def auto_sync_positions_with_exchange():
    ghi_log("\n🔄 [HỆ THỐNG] Đang đồng bộ trạng thái với sàn Binance...")
    try:
        positions = client.futures_position_information()
        active_on_exchange = {p['symbol']: p for p in positions if float(p['positionAmt']) != 0}
        
        for symbol in TARGET_SYMBOLS:
            memory = _ensure_symbol_state(symbol)
            if symbol in active_on_exchange:
                pos = active_on_exchange[symbol]
                amt = float(pos['positionAmt'])
                entry_price = float(pos['entryPrice'])
                side = "LONG" if amt > 0 else "SHORT"
                abs_qty = abs(amt)
                invested = abs_qty * entry_price
                memory.update({
                    "position_side": side,
                    "entry_price": entry_price,
                    "quantity": abs_qty, 
                    "invested_usdt": invested,
                    "last_scan_price": entry_price
                })
                ghi_log(f"✅ Đã đồng bộ {symbol}: {side} | Vol: {abs_qty} | Giá vào: {entry_price}")
            else:
                if memory["position_side"] != "NONE":
                    # 🛡️ BẢN VÁ: NHÁT CHÉM 3 - FEEDBACK CHO BANDIT ALLOCATOR
                    closed_side = memory["position_side"]
                    closed_regime = memory.get("entry_regime", "base") 
                    invested_usd = memory.get("invested_usdt", 100.0)
                    try:
                        # Rút trích Hóa đơn Lãi/Lỗ ròng gần nhất từ Binance cho đồng Coin này
                        income_hist = client.futures_income(symbol=symbol, incomeType="REALIZED_PNL", limit=1)
                        if income_hist:
                            realized_usd = float(income_hist[0]['income'])
                            realized_pnl_pct = realized_usd / invested_usd if invested_usd > 0 else 0.0
                            ghi_log(f"💰 {symbol} Vừa chốt sổ: Lãi/Lỗ {realized_usd:.2f}$ ({realized_pnl_pct*100:.2f}%)")
                            update_bandit_feedback(
                                symbol=symbol, 
                                regime=closed_regime, 
                                direction=closed_side, 
                                realized_pnl_pct=realized_pnl_pct
                            )
                    except Exception as api_err:
                        ghi_log(f"⚠️ Lỗi fetch PnL cho Bandit ở {symbol}: {api_err}")
                    ghi_log(f"🧹 {symbol}: Sàn đã đóng lệnh, cập nhật bộ nhớ Bot về NONE.")
                    memory.update({"position_side": "NONE", "quantity": 0.0, "invested_usdt": 0.0})
        _save_runtime_state()
        ghi_log("🏁 [HỆ THỐNG] Đồng bộ hoàn tất. Bot đã sẵn sàng chiến đấu!")
        return True
    except Exception as e:
        ghi_log(f"🚨 [LỖI ĐỒNG BỘ] Không thể kết nối với sàn: {e}")
        return False

class PortfolioRiskEngine:
    def __init__(self):
        if "governor_stats" not in runtime_state:
            runtime_state["governor_stats"] = {}
        self.stats = runtime_state["governor_stats"]

    def check_drawdown_governor(self, regime, side):
        """Cầu dao Drawdown: Tắt model nếu nó đang bị lệch pha thị trường"""
        key = f"R{regime}_{side}"
        if key not in self.stats:
            self.stats[key] = {"peak": 1.0, "current": 1.0}
        dd = (self.stats[key]["peak"] - self.stats[key]["current"]) / self.stats[key]["peak"]
        if dd > 0.12:
            return False, f"Bị chặn bởi DD Governor (Lỗ lũy kế {dd*100:.1f}%)"
        return True, "Governor An toàn"

    def update_governor_pnl(self, regime, side, pnl_pct):
        """Cập nhật dữ liệu PnL sau khi lệnh đóng"""
        key = f"R{regime}_{side}"
        if key not in self.stats:
            self.stats[key] = {"peak": 1.0, "current": 1.0}
        self.stats[key]["current"] *= (1.0 + pnl_pct)
        if self.stats[key]["current"] > self.stats[key]["peak"]:
            self.stats[key]["peak"] = self.stats[key]["current"]

    def filter_by_btc_beta_cap(self, intended_trades, bot_memory):
        """Giới hạn Net Beta của toàn danh mục (Chống rủi ro sập chung toàn thị trường)"""
        active_positions = {sym: data for sym, data in bot_memory.items() if data["position_side"] != "NONE"}
        net_beta_exposure = 0.0
        for sym, data in active_positions.items():
            beta_sign = 1 if data["position_side"] == "LONG" else -1
            weight = data["invested_usdt"] / max(globals().get("TOTAL_PORTFOLIO_USDT", 250.0), 1e-9)
            asset_beta = 1.0 if sym == "BTCUSDT" else 0.85 # Giả định Beta của Altcoin với BTC ~ 0.85
            net_beta_exposure += beta_sign * weight * asset_beta
        approved_trades, rejected_trades = [], []
        for trade in intended_trades:
            trade_beta_sign = 1 if trade["action"] == "LONG" else -1
            trade_weight = 0.05 # Giả sử lệnh mới chiếm 5% vốn
            trade_beta = 1.0 if trade["symbol"] == "BTCUSDT" else 0.85
            simulated_exposure = net_beta_exposure + (trade_beta_sign * trade_weight * trade_beta)
            if abs(simulated_exposure) > 0.40:
                rejected_trades.append(f"{trade['symbol']} (Beta Cap: Portfolio Net Beta vượt ngưỡng {simulated_exposure:+.2f})")
            else:
                net_beta_exposure = simulated_exposure
                approved_trades.append(trade)
        return approved_trades, rejected_trades
risk_engine = PortfolioRiskEngine()

def run_batch_retrain_cycle():
    from datetime import datetime
    ghi_log(f"\n{'='*50}\n🔄 [{datetime.now().strftime('%H:%M:%S')}] KÍCH HOẠT CHU TRÌNH AUTO-RETRAIN (V8)\n{'='*50}")
    try:
        all_dfs = []
        for sym in TARGET_SYMBOLS:
            raw_df = _get_cached_klines(sym, Client.KLINE_INTERVAL_1HOUR, limit=800) # Lấy 800 nến gần nhất
            if raw_df is not None and not raw_df.empty:
                df_feat = prepare_live_feature_frame_dual(raw_df, 0, sym)
                df_feat['symbol'] = sym  # Đảm bảo có cột symbol thật để lát nữa groupby
                all_dfs.append(df_feat)  
        if not all_dfs:
            ghi_log("⚠️ Auto-Retrain thất bại: Không lấy được dữ liệu từ API.")
            return False  
        mega_retrain_df = pd.concat(all_dfs, ignore_index=True)
        mega_retrain_df = mega_retrain_df.sort_values("Open time").reset_index(drop=True)
        # 💉 ĐỊNH NGHĨA HÀM TIÊM ALPHA (Thụt lề chuẩn Python)
        def inject_cross_sectional_alpha(df_input, btc_symbol="BTCUSDT"):
            print("🌍 ĐANG BƠM HUYẾT MẠCH CROSS-SECTIONAL ALPHA (RESIDUAL VS BTC)...")
            df = df_input.copy()
            
            if btc_symbol not in df['symbol'].values:
                print(f"⚠️ Không tìm thấy {btc_symbol} trong Data!")
                return df
                
            btc_df = df[df['symbol'] == btc_symbol][['Open time', 'Close']].copy()
            btc_df.rename(columns={'Close': 'BTC_Close'}, inplace=True)
            btc_df['BTC_Return'] = btc_df['BTC_Close'].pct_change().fillna(0)
            
            df = df.merge(btc_df[['Open time', 'BTC_Return']], on='Open time', how='left')
            df['BTC_Return'] = df['BTC_Return'].fillna(0)
            df['Asset_Return'] = df.groupby('symbol')['Close'].pct_change().fillna(0)
            
            def calc_rolling_beta(group):
                cov = group['Asset_Return'].rolling(24).cov(group['BTC_Return'])
                var = group['BTC_Return'].rolling(24).var()
                group['Beta_24h'] = (cov / var).replace([np.inf, -np.inf], 1.0).fillna(1.0)
                return group
                
            df = df.groupby('symbol', group_keys=False).apply(calc_rolling_beta)
            df['Residual_Alpha'] = df['Asset_Return'] - (df['Beta_24h'] * df['BTC_Return'])
            df['Alpha_Rank_Z'] = df.groupby('Open time')['Residual_Alpha'].transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))
            
            print("✅ Đã tiêm xong 2 mũi: Residual_Alpha và Alpha_Rank_Z vào Data tổng!")
            return df

        # ========================================================
        # 💉 KÍCH HOẠT TIÊM VÀO DATA AUTO-RETRAIN
        # ========================================================
        print("💉 Đang tiêm Cross-Sectional Alpha vào Data thô đa tài sản...")
        mega_retrain_df = inject_cross_sectional_alpha(mega_retrain_df)
        print("✅ Tiêm Alpha thành công! Data đã sẵn sàng để đẩy vào Lò Rèn.")

        ghi_log(f"📦 Đã gom thành công {len(mega_retrain_df)} nến đa tài sản. Đẩy vào Lò rèn Champion-Challenger...")
        
        champion_challenger_retrain(mega_retrain_df)
        
        return True 

    except Exception as e:
        import traceback
        error_msg = f"❌ LỖI NGHIÊM TRỌNG KHI RETRAIN: {e}\n{traceback.format_exc()}"
        print(error_msg) # In ra Terminal
        ghi_log(error_msg) # Ghi vào file Log
        ghi_log("⚠️ HỆ THỐNG TẠM DỪNG RETRAIN CHU KỲ NÀY. GIỮ NGUYÊN MODEL HIỆN TẠI ĐỂ TRADE AN TOÀN!")
        return False


c:\Users\Minh Nhat\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⚙️ Đang khởi tạo Hệ thống Đa Chuyên Gia (Multi-Expert)...
⏳ Đang kiểm tra Sổ Đăng Kiểm (Artifact Manifest)...
  -> Nạp Manifest phiên bản: UNKNOWN
  -> Thời gian Train: 2026-05-07T07:57:13.984668

🚨 HỆ THỐNG BOOT THẤT BẠI!
--------------------------------------------------
🚨 FATAL: Thiếu file Calibration: None
--------------------------------------------------


SystemExit: Dừng chạy code để tránh cháy tài khoản.

C:\Users\Minh Nhat\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Lấy data

In [6]:
import pandas as pd
import numpy as np
from binance.client import Client

print("⏳ ĐANG KHỞI ĐỘNG MÁY BƠM DỮ LIỆU ĐA TÀI SẢN...")
train_list = []
test_list = []
TARGET_SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT", 
                  "DOGEUSDT", "AVAXUSDT", "LINKUSDT", "NEARUSDT", "ADAUSDT"]                

for sym in TARGET_SYMBOLS:
    try:
        print(f"  -> Đang cào & nhào nặn features cho {sym}...")
        klines = client.get_historical_klines(sym, Client.KLINE_INTERVAL_1HOUR, "90 days ago UTC")
        df_raw = pd.DataFrame(klines, columns=[
            "Open time", "Open", "High", "Low", "Close", "Volume", "Close time", 
            "Quote Asset", "Trades", "Taker Buy Base", "Taker Buy Quote", "Ignore"
        ])
        df_feat = QuantFeatureEngineer.run_pipeline(df_raw, asset_sentiment=0.0, symbol=sym)
        df_feat['symbol'] = sym
        df_feat = df_feat.dropna().reset_index(drop=True)
        split_idx = int(len(df_feat) * 0.8)
        train_df = df_feat.iloc[:split_idx].copy()
        test_df = df_feat.iloc[split_idx:].copy()
        
        train_list.append(train_df)
        test_list.append(test_df)
    except Exception as e:
        print(f"⚠️ Bỏ qua {sym} do lỗi: {e}")

# ========================================================
# 🛡️ GỘP DATA THÀNH MEGA_DF TRƯỚC KHI TIÊM THUỐC
# ========================================================
print("\n" + "="*50)
print("📦 ĐANG GỘP DỮ LIỆU CÁC ĐỒNG COIN...")
print("="*50)
mega_train_df = pd.concat(train_list, ignore_index=True).sort_values("Open time").reset_index(drop=True)
mega_test_df = pd.concat(test_list, ignore_index=True).sort_values("Open time").reset_index(drop=True)

# ========================================================
# 💉 TIÊM CROSS-SECTIONAL ALPHA
# ========================================================
print("\n" + "="*50)
print("🌍 ĐANG BƠM HUYẾT MẠCH CROSS-SECTIONAL ALPHA (RESIDUAL VS BTC)...")
print("="*50)

def inject_cross_sectional_alpha(mega_df, btc_symbol="BTCUSDT"):
    df = mega_df.copy()
    if btc_symbol not in df['symbol'].values:
        print(f"⚠️ Không tìm thấy {btc_symbol} trong Data. Vui lòng đảm bảo BTC có trong TARGET_SYMBOLS!")
        return df 
    btc_df = df[df['symbol'] == btc_symbol][['Open time', 'Close']].copy()
    btc_df.rename(columns={'Close': 'BTC_Close'}, inplace=True)
    btc_df['BTC_Return'] = btc_df['BTC_Close'].pct_change().fillna(0)
    
    df = df.merge(btc_df[['Open time', 'BTC_Return']], on='Open time', how='left')
    df['BTC_Return'] = df['BTC_Return'].fillna(0)
    df['Asset_Return'] = df.groupby('symbol')['Close'].pct_change().fillna(0)
    
    def calc_rolling_beta(group):
        cov = group['Asset_Return'].rolling(24).cov(group['BTC_Return'])
        var = group['BTC_Return'].rolling(24).var()
        group['Beta_24h'] = (cov / var).replace([np.inf, -np.inf], 1.0).fillna(1.0)
        return group
        
    df = df.groupby('symbol', group_keys=False).apply(calc_rolling_beta)
    df['Residual_Alpha'] = df['Asset_Return'] - (df['Beta_24h'] * df['BTC_Return'])
    df['Alpha_Rank_Z'] = df.groupby('Open time')['Residual_Alpha'].transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))
    return df

# ========================================================
# 🧬 TIÊM ORDERFLOW & MICROSTRUCTURE ALPHA
# ========================================================
def inject_orderflow_alpha(df):
    df = df.copy()
    
    # ✅ BẢN VÁ: Ép kiểu dữ liệu sang số thực (Float) vì Binance API trả về String
    if 'Volume' in df.columns:
        df['Volume'] = df['Volume'].astype(float)
    if 'Taker Buy Base' in df.columns:
        df['Taker Buy Base'] = df['Taker Buy Base'].astype(float)
    
    # 1. TAKER BUY/SELL IMBALANCE
    if 'Taker Buy Base' in df.columns and 'Volume' in df.columns:
        df['Taker_Sell_Base'] = df['Volume'] - df['Taker Buy Base']
        df['Taker_Imbalance'] = (df['Taker Buy Base'] - df['Taker_Sell_Base']) / (df['Volume'] + 1e-8)
        df['Taker_Shock_3'] = df['Taker_Imbalance'] - df.groupby('symbol')['Taker_Imbalance'].transform(lambda x: x.rolling(3).mean())

    # 2. OPEN INTEREST (OI) DELTA THẬT
    if 'Open_Interest' in df.columns:
        df['OI_Delta_Pct'] = df.groupby('symbol')['Open_Interest'].pct_change().fillna(0)
        df['OI_Trend_Z'] = df.groupby('symbol')['Open_Interest'].transform(lambda x: (x - x.rolling(24).mean()) / (x.rolling(24).std() + 1e-8))
        df['OI_Price_Regime'] = np.where((df['OI_Delta_Pct'] > 0) & (df['Close'] > df['Open']), 1,  
                              np.where((df['OI_Delta_Pct'] > 0) & (df['Close'] < df['Open']), -1,  
                              np.where((df['OI_Delta_Pct'] < 0) & (df['Close'] < df['Open']), -2,  
                              np.where((df['OI_Delta_Pct'] < 0) & (df['Close'] > df['Open']), 2, 0))))

    # 3. LIQUIDATION IMBALANCE
    if 'Liq_Long' in df.columns and 'Liq_Short' in df.columns:
        df['Total_Liq'] = df['Liq_Long'] + df['Liq_Short']
        df['Liq_Imbalance'] = (df['Liq_Long'] - df['Liq_Short']) / (df['Total_Liq'] + 1e-8)
        df['Liq_Shock_Z'] = df.groupby('symbol')['Total_Liq'].transform(lambda x: (x - x.rolling(24).mean()) / (x.rolling(24).std() + 1e-8))

    # 4. FUNDING RATE Z-SCORE
    if 'Funding_Rate' in df.columns:
        df['Funding_Z'] = df.groupby('symbol')['Funding_Rate'].transform(lambda x: (x - x.rolling(72).mean()) / (x.rolling(72).std() + 1e-8))

    # 5. SPREAD / DEPTH SHOCK
    if 'Bid_Ask_Spread' in df.columns:
        df['Spread_Shock_Z'] = df.groupby('symbol')['Bid_Ask_Spread'].transform(lambda x: (x - x.rolling(24).mean()) / (x.rolling(24).std() + 1e-8))
        
    return df

# --- THỰC THI TIÊM CÁC MŨI ALPHA ---
mega_train_df = inject_cross_sectional_alpha(mega_train_df)
mega_test_df = inject_cross_sectional_alpha(mega_test_df)
print("✅ Đã tiêm xong 2 mũi: Residual_Alpha và Alpha_Rank_Z!")

mega_train_df = inject_orderflow_alpha(mega_train_df)
mega_test_df = inject_orderflow_alpha(mega_test_df)
print("✅ Đã tiêm xong mũi thứ 3: Orderflow & Microstructure (Taker Imbalance)!")

# 🛡️ BẢN VÁ: In đúng biến tổng (mega_train_df) thay vì con lẻ (train_df)
print(f"✅ KHAI SINH THÀNH CÔNG! Tổng Data Train: {len(mega_train_df)} nến | Data Test: {len(mega_test_df)} nến.")

⏳ ĐANG KHỞI ĐỘNG MÁY BƠM DỮ LIỆU ĐA TÀI SẢN...
  -> Đang cào & nhào nặn features cho BTCUSDT...
  -> Đang cào & nhào nặn features cho ETHUSDT...
  -> Đang cào & nhào nặn features cho SOLUSDT...
  -> Đang cào & nhào nặn features cho BNBUSDT...
  -> Đang cào & nhào nặn features cho XRPUSDT...
  -> Đang cào & nhào nặn features cho DOGEUSDT...
  -> Đang cào & nhào nặn features cho AVAXUSDT...
  -> Đang cào & nhào nặn features cho LINKUSDT...
  -> Đang cào & nhào nặn features cho NEARUSDT...
  -> Đang cào & nhào nặn features cho ADAUSDT...

📦 ĐANG GỘP DỮ LIỆU CÁC ĐỒNG COIN...

🌍 ĐANG BƠM HUYẾT MẠCH CROSS-SECTIONAL ALPHA (RESIDUAL VS BTC)...


C:\Users\Minh Nhat\AppData\Local\Temp\ipykernel_9920\1779185872.py:66: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('symbol', group_keys=False).apply(calc_rolling_beta)


✅ Đã tiêm xong 2 mũi: Residual_Alpha và Alpha_Rank_Z!
✅ Đã tiêm xong mũi thứ 3: Orderflow & Microstructure (Taker Imbalance)!
✅ KHAI SINH THÀNH CÔNG! Tổng Data Train: 16356 nến | Data Test: 4094 nến.


C:\Users\Minh Nhat\AppData\Local\Temp\ipykernel_9920\1779185872.py:66: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('symbol', group_keys=False).apply(calc_rolling_beta)


In [7]:
import joblib
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor
import numpy as np
import pandas as pd
PROBA_SCAN = [0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

def build_xgb_v8_model(y_fit):
    positives = int(np.sum(y_fit == 1))
    negatives = int(np.sum(y_fit == 0))
    scale_pos_weight = negatives / max(positives, 1)

    model = XGBClassifier(
        objective="binary:logistic", n_estimators=320, max_depth=4,
        learning_rate=0.03, subsample=0.90, colsample_bytree=0.90,
        min_child_weight=8, gamma=0.0, reg_lambda=2.0, reg_alpha=0.5,
        scale_pos_weight=scale_pos_weight, random_state=42, n_jobs=-1,
        tree_method="hist", eval_metric="logloss",
    )
    return model

def backtest_path_based(df_setups, threshold, direction="long"):
    ret_col = f"realized_net_return_{direction.lower()}"
    trades_df = df_setups[df_setups["pred_proba"] >= threshold].copy()
    if len(trades_df) == 0:
        return {"trades": 0, "win_rate": 0.0, "profit_factor": 0.0, "total_pnl": 0.0}
    gains = trades_df.loc[trades_df[ret_col] > 0, ret_col].sum()
    losses = -trades_df.loc[trades_df[ret_col] < 0, ret_col].sum()
    profit_factor = gains / losses if losses > 0 else 999.0
    total_pnl = trades_df[ret_col].sum() * 100 
    return {
        "trades": int(len(trades_df)),
        "win_rate": float((trades_df[ret_col] > 0).mean() * 100),
        "profit_factor": float(profit_factor),
        "total_pnl": float(total_pnl)
    }

# 🛡️ 1. TRIPLE-BARRIER METHOD 
def apply_triple_barrier_high_low(df, pt_multiplier=2.0, sl_multiplier=1.0, time_limit=12):
    df_copy = df.copy()
    target_label = np.zeros(len(df_copy))
    highs = df_copy['High'].values
    lows = df_copy['Low'].values
    closes = df_copy['Close'].values
    atrs = df_copy['ATR_Pct'].values if 'ATR_Pct' in df_copy.columns else np.full(len(df_copy), 0.02)
    for i in range(len(df_copy) - time_limit):
        entry_price = closes[i]
        tp_pct = atrs[i] * pt_multiplier
        sl_pct = atrs[i] * sl_multiplier
        tp_price_long = entry_price * (1 + tp_pct)
        sl_price_long = entry_price * (1 - sl_pct)
        tp_price_short = entry_price * (1 - tp_pct)
        sl_price_short = entry_price * (1 + sl_pct)
        for j in range(1, time_limit + 1):
            future_idx = i + j
            curr_high = highs[future_idx]
            curr_low = lows[future_idx]
            if curr_low <= sl_price_long:
                target_label[i] = -1 
                break
            elif curr_high >= tp_price_long:
                target_label[i] = 1  
                break
    df_copy['TBM_Label'] = target_label
    return df_copy
print("🔄 Đang gán nhãn lại dữ liệu bằng Triple-Barrier (High/Low)...")

# 🛡️ BẢN VÁ CONTAMINATION (GROUPBY TỪNG COIN)
def safe_label_by_asset(df):
    if 'symbol' not in df.columns:
        print("⚠️ Cảnh báo: Không tìm thấy cột phân biệt Coin. Giả định đây là 1 đồng coin duy nhất.")
        return apply_triple_barrier_high_low(df, pt_multiplier=2.0, sl_multiplier=1.0, time_limit=12)
    df_result = df.groupby('symbol', group_keys=False).apply(
        lambda x: apply_triple_barrier_high_low(x, pt_multiplier=2.0, sl_multiplier=1.0, time_limit=12).assign(symbol=x.name),
        include_groups=False
    )
    return df_result.reset_index(drop=True)
mega_train_df = safe_label_by_asset(mega_train_df)
mega_test_df = safe_label_by_asset(mega_test_df)
mega_train_df = mega_train_df[mega_train_df['TBM_Label'] != 0].copy()
mega_test_df = mega_test_df[mega_test_df['TBM_Label'] != 0].copy()

# Tách nhãn cho 2 Sư phụ Long/Short
mega_train_df["target_long"] = (mega_train_df["TBM_Label"] == 1).astype(int)
mega_test_df["target_long"] = (mega_test_df["TBM_Label"] == 1).astype(int)
mega_train_df["target_short"] = (mega_train_df["TBM_Label"] == -1).astype(int)
mega_test_df["target_short"] = (mega_test_df["TBM_Label"] == -1).astype(int)

print("✅ Hoàn tất gán nhãn TBM (High/Low) - Đã xử lý an toàn đa tài sản!")

# ========================================================
# 🛡️ 2. TIMESTAMP-GROUP WALK FORWARD SPLITTER (CHỐNG RÒ RỈ CHÉO)
# ========================================================
class TimestampGroupWalkForward:
    def __init__(self, n_splits=5, purge_bars=12):
        self.n_splits = n_splits
        self.purge_bars = purge_bars

    def split(self, df):
        unique_times = np.sort(df['Open time'].unique())
        tscv = TimeSeriesSplit(n_splits=self.n_splits)
        
        for train_time_idx, test_time_idx in tscv.split(unique_times):
            train_times = unique_times[train_time_idx]
            test_times = unique_times[test_time_idx]
            
            # Cắt bỏ vùng xám (Purge) ở cuối tập Train
            if self.purge_bars > 0 and len(train_times) > self.purge_bars:
                train_times = train_times[:-self.purge_bars]
            
            train_mask = df['Open time'].isin(train_times)
            test_mask = df['Open time'].isin(test_times)
            yield np.where(train_mask)[0], np.where(test_mask)[0]

# ========================================================
# 🧠 3. HUẤN LUYỆN SƯ PHỤ (WALK-FORWARD CROSS VALIDATION)
# ========================================================
feature_columns_v8 = [c for c in mega_train_df.columns if c not in [
    "Open time", "Close time", "Ignore", "target_long", "target_short", 
    "realized_net_return_long", "realized_net_return_short", "exit_bars_long", "exit_bars_short",
    "setup_long_candidate", "setup_short_candidate", "TBM_Label", "Market_Regime",
    "symbol", "Asset_ID", "Symbol" 
]]

mega_df = pd.concat([mega_train_df, mega_test_df], ignore_index=True)
mega_df = mega_df.sort_values("Open time").reset_index(drop=True)

# GỌI SPLITTER MỚI BẰNG THỜI GIAN
cv = TimestampGroupWalkForward(n_splits=5, purge_bars=12)
splits = list(cv.split(mega_df))

def execute_walk_forward(df, direction="long"):
    print(f"\n" + "="*50)
    print(f"🚀 KHỞI ĐỘNG PURGED WALK-FORWARD ({direction.upper()}) [TIMESTAMP GROUP]")
    print("="*50)
    setup_col = f"setup_{direction}_candidate"
    target_col = f"target_{direction}"
    all_fold_trades = []
    final_model = None
    for fold, (train_idx, test_idx) in enumerate(splits):
        train_fold = df.iloc[train_idx]
        test_fold = df.iloc[test_idx]
        train_setups = train_fold[train_fold[setup_col]]
        test_setups = test_fold[test_fold[setup_col]]
        if len(train_setups) == 0 or len(test_setups) == 0: continue
        X_train = train_setups[feature_columns_v8].astype(np.float32)
        y_train = train_setups[target_col].astype(int)
        X_test = test_setups[feature_columns_v8].astype(np.float32)
        model = build_xgb_v8_model(y_train)
        model.fit(X_train, y_train, verbose=False)
        final_model = model
        proba_test = model.predict_proba(X_test)[:, 1]
        test_setups_eval = test_setups.copy()
        test_setups_eval["pred_proba"] = proba_test
        all_fold_trades.append(test_setups_eval)
        print(f"Fold {fold+1}/5 | Train: {len(X_train)} setups | Test: {len(X_test)} setups")
    master_test_df = pd.concat(all_fold_trades, ignore_index=True)
    print(f"⚙️ Đang chạy mô phỏng đường giá thực chiến (Path-Based) cho {len(master_test_df)} lệnh...")
    sym_col = None
    for col in ['Symbol', 'symbol', 'Asset_ID', 'asset', 'Ticker']:
        if col in master_test_df.columns:
            sym_col = col
            break
    pnl_series = pd.Series(index=master_test_df.index, dtype=float)
    if sym_col:
        for name, group in master_test_df.groupby(sym_col):
            pnl_series.loc[group.index] = calculate_real_path_pnl(group, direction.upper(), pt_mult=2.0, sl_mult=1.0, max_bars=12)
    else:
        pnl_series[:] = calculate_real_path_pnl(master_test_df, direction.upper(), pt_mult=2.0, sl_mult=1.0, max_bars=12)
    master_test_df[f"realized_net_return_{direction}"] = pnl_series
    print(f"\n📊 KẾT QUẢ TỔNG HỢP COST-AWARE THỰC TẾ ({direction.upper()})")
    for thr in PROBA_SCAN:
        out = backtest_path_based(master_test_df, threshold=thr, direction=direction)
        print(f"thr={thr:.2f} | trades={out['trades']:<4} | win={out['win_rate']:<5.2f}% | pf={out['profit_factor']:<5.3f} | Total PnL: {out['total_pnl']:+.2f}%")     
    return final_model, master_test_df

model_long, oos_long_df = execute_walk_forward(mega_df, direction="long")
joblib.dump(model_long, "xgb_v8_long_fee_aware_multi.pkl")
model_short, oos_short_df = execute_walk_forward(mega_df, direction="short")
joblib.dump(model_short, "xgb_v8_short_fee_aware_multi.pkl")

# ⚖️ TÍNH TOÁN TRỌNG SỐ ENSEMBLE (THỰC CHIẾN 100%)
from sklearn.metrics import brier_score_loss
import joblib
print("\n" + "="*50)
print("⚖️ TÍNH TOÁN TRỌNG SỐ ENSEMBLE (THỰC CHIẾN 100%)")
print("="*50)
def calculate_dynamic_weights(df_oos, direction):
    y_true = df_oos[f"target_{direction.lower()}"].astype(int)
    y_pred_xgb = df_oos["pred_proba"]
    
    # 1. Chấm điểm Sư phụ XGBoost
    brier_xgb = brier_score_loss(y_true, y_pred_xgb)
    print(f"🎯 Brier Score XGB ({direction}): {brier_xgb:.4f}")
    # 2. Chấm điểm Sư phụ DL/LSTM (Chỉ chấm khi có Data thật)
    if "pred_proba_dl" in df_oos.columns and not df_oos["pred_proba_dl"].isnull().all():
        y_pred_dl = df_oos["pred_proba_dl"]
        brier_dl = brier_score_loss(y_true, y_pred_dl)
        print(f"🎯 Brier Score DL  ({direction}): {brier_dl:.4f}")
    else:
        print(f"⚠️ [CẢNH BÁO] Không tìm thấy dữ liệu OOS của DL/LSTM cho phe {direction}.")
        print("   -> ĐÓNG BĂNG QUYỀN BLEND CỦA LSTM! Chuyển 100% quyền lực cho XGBoost.")
        brier_dl = 999.0 
    # 3. Thuật toán Inverse-Variance Weighting
    inv_xgb = 1.0 / (brier_xgb + 1e-6)
    inv_dl = 1.0 / (brier_dl + 1e-6)
    w_xgb = inv_xgb / (inv_xgb + inv_dl)
    w_dl = 1.0 - w_xgb
    print(f"🔥 Trọng số Tối ưu {direction} -> XGB: {w_xgb*100:.1f}% | DL: {w_dl*100:.1f}%\n")
    return w_xgb, w_dl
w_xgb_long, w_dl_long = calculate_dynamic_weights(oos_long_df, "LONG")
w_xgb_short, w_dl_short = calculate_dynamic_weights(oos_short_df, "SHORT")

expert_models = {
    "0": {
        "name": "Regime 0", 
        "long": model_long,   # Sử dụng biến thực tế của Giám đốc
        "short": model_short
    },
    "1": {
        "name": "Regime 1", 
        "long": model_long, 
        "short": model_short
    },
    "2": {
        "name": "Regime 2", 
        "long": model_long, 
        "short": model_short
    },
    "base": {
        "name": "Base Regime", 
        "long": model_long, 
        "short": model_short
    }
}

joblib.dump(expert_models, "expert_models_v8.pkl")
print("✅ ĐÃ VÁ LỖI MẤT NÃO! Xuất xưởng file expert_models_v8.pkl thành công.")

joblib.dump(expert_models, "expert_models_v8.pkl")
print("✅ ĐÃ VÁ LỖI MẤT NÃO! Xuất xưởng file expert_models_v8.pkl thành công.")
# 4. Lắp ráp Schema Calibration (Bọc thép 3 Regime cho Bot Live)
ensemble_schema_v8 = {
    "LONG": {
        "0": {"xgb": w_xgb_long, "dl": w_dl_long},
        "1": {"xgb": w_xgb_long, "dl": w_dl_long},
        "2": {"xgb": w_xgb_long, "dl": w_dl_long},
        "base": {"xgb": w_xgb_long, "dl": w_dl_long}
    },
    "SHORT": {
        "0": {"xgb": w_xgb_short, "dl": w_dl_short},
        "1": {"xgb": w_xgb_short, "dl": w_dl_short},
        "2": {"xgb": w_xgb_short, "dl": w_dl_short},
        "base": {"xgb": w_xgb_short, "dl": w_dl_short}
    }
}
joblib.dump(ensemble_schema_v8, "ensemble_weights_v8.pkl")
print("✅ ĐÃ VÁ XONG VẾT NỨT ENSEMBLE! Hệ thống trọng số giờ là minh bạch và thực chứng.")

print("🛡️ ĐANG HUẤN LUYỆN META-MODEL TẦNG 2 (EXPECTED VALUE & UNCERTAINTY)...")
print("="*50)

# 1. Trích xuất các Feature Tầng 2
meta_feature_cols = [
    "pred_proba",         # XGB Prob (Độ tự tin của Tầng 1)
    "Market_Regime",      # Bối cảnh vĩ mô
    "Taker_Imbalance",    # OFI (Dòng lệnh Taker)
    "Sentiment_Score",    # Tâm lý báo chí
    "Volatility_24",      # Độ giật của nến
    "Volume_Z",           # Đột biến Volume
    "Trend_Strength"      # Xung lực (MTF proxy)
]
for df in [oos_long_df, oos_short_df]:
    for col in meta_feature_cols:
        if col not in df.columns: df[col] = 0.0
X_meta_train = pd.concat([oos_long_df[meta_feature_cols], oos_short_df[meta_feature_cols]], ignore_index=True).astype(np.float32)

# 2. BẮT AI HỌC TRỰC TIẾP LỢI NHUẬN RÒNG (EV) VÀ ĐỘ NHIỄU (UNCERTAINTY)
y_meta_ev = pd.concat([oos_long_df["realized_net_return_long"], oos_short_df["realized_net_return_short"]], ignore_index=True).astype(np.float32)
ev_model = XGBRegressor(
    objective="reg:squarederror", n_estimators=350, max_depth=4,
    learning_rate=0.03, subsample=0.85, colsample_bytree=0.85,
    random_state=42, n_jobs=-1, tree_method="hist"
)
ev_model.fit(X_meta_train, y_meta_ev, verbose=False)
print("✅ Đã huấn luyện xong Động cơ dự báo EV (Lợi nhuận kỳ vọng)!")
ev_predictions = ev_model.predict(X_meta_train)
y_meta_uncertainty = np.abs(y_meta_ev - ev_predictions) 
uncertainty_model = XGBRegressor(
    objective="reg:squarederror", n_estimators=250, max_depth=3,
    learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, tree_method="hist"
)
uncertainty_model.fit(X_meta_train, y_meta_uncertainty, verbose=False)
print("✅ Đã huấn luyện xong Động cơ dự báo Uncertainty (Biên độ rủi ro)!")

# 3. ĐÓNG GÓI VÀ XUẤT XƯỞNG HỢP ĐỒNG (ARTIFACT CONTRACT)
# Đóng gói 2 não bộ vào chung 1 file
meta_artifact = {
    "ev_model": ev_model,
    "uncertainty_model": uncertainty_model
}
joblib.dump(meta_artifact, "xgb_v8_meta_model.pkl")
joblib.dump(meta_feature_cols, "xgb_v8_meta_features.pkl") 
print("✅ Đã xuất xưởng Vệ sĩ Tầng 2 (Dual Regressor).")
LAG_STEPS_V8 = [1, 2, 3, 6, 12]
USE_SENTIMENT_FEATURE = True
joblib.dump({
    "lag_steps": LAG_STEPS_V8,
    "feature_columns": feature_columns_v8,
    "use_sentiment_feature": USE_SENTIMENT_FEATURE
}, "xgb_v8_meta.pkl")
print("✅ Đã xuất xưởng Meta Data Config.")
print("\n🎉 XUẤT XƯỞNG THÀNH CÔNG! HỆ THỐNG ĐÃ CHUYỂN SANG HỆ TƯ TƯỞNG EXPECTED VALUE!")




🔄 Đang gán nhãn lại dữ liệu bằng Triple-Barrier (High/Low)...
✅ Hoàn tất gán nhãn TBM (High/Low) - Đã xử lý an toàn đa tài sản!

🚀 KHỞI ĐỘNG PURGED WALK-FORWARD (LONG) [TIMESTAMP GROUP]
Fold 1/5 | Train: 874 setups | Test: 1334 setups
Fold 2/5 | Train: 2138 setups | Test: 1413 setups
Fold 3/5 | Train: 3544 setups | Test: 1210 setups
Fold 4/5 | Train: 4753 setups | Test: 1756 setups
Fold 5/5 | Train: 6483 setups | Test: 1508 setups
⚙️ Đang chạy mô phỏng đường giá thực chiến (Path-Based) cho 7221 lệnh...

📊 KẾT QUẢ TỔNG HỢP COST-AWARE THỰC TẾ (LONG)
thr=0.45 | trades=2844 | win=37.10% | pf=0.862 | Total PnL: -240.72%
thr=0.50 | trades=2391 | win=36.80% | pf=0.859 | Total PnL: -207.84%
thr=0.55 | trades=1962 | win=37.00% | pf=0.875 | Total PnL: -151.42%
thr=0.60 | trades=1547 | win=37.56% | pf=0.905 | Total PnL: -91.24%
thr=0.65 | trades=1134 | win=38.10% | pf=0.945 | Total PnL: -38.84%
thr=0.70 | trades=777  | win=40.15% | pf=1.039 | Total PnL: +18.78%

🚀 KHỞI ĐỘNG PURGED WALK-FORWARD (S

### TRAIN CALIBRATION

In [8]:
import json
import numpy as np
import pandas as pd
from datetime import datetime

print("\n" + "="*70)
print("🎯 ĐÚC CALIBRATION & TỐI ƯU NGƯỠNG (TÍCH HỢP BAYESIAN & BOOTSTRAP) - SCHEMA V2")
print("="*70)

# ========================================================
# 🛡️ 1. HÀM MÔ PHỎNG BOOTSTRAP MONTE CARLO
# ========================================================
def calculate_bootstrap_pf_lower(returns, n_iterations=1000, ci_level=5):
    """
    Bốc thăm ngẫu nhiên (có hoàn lại) 1000 lần từ lịch sử PnL để xem 
    nếu xui xẻo nhất (bách phân vị thứ 5) thì Profit Factor là bao nhiêu.
    """
    returns_array = np.array(returns)
    if len(returns_array) < 5: 
        return 0.0 
        
    pfs = []
    for _ in range(n_iterations):
        sample = np.random.choice(returns_array, size=len(returns_array), replace=True)
        gains = sample[sample > 0].sum()
        losses = -sample[sample < 0].sum()
        pf = gains / losses if losses > 0 else 999.0
        pfs.append(pf)
        
    return np.percentile(pfs, ci_level)

# ========================================================
# 🛡️ 2. HÀM TỐI ƯU NGƯỠNG BAYESIAN (CỦA GIÁM ĐỐC)
# ========================================================
def optimize_threshold_bayesian(df, direction, regime, min_trades=15, target_pf=1.05, prior_pf=0.95, shrinkage_k=40):
    ret_col = f"realized_net_return_{direction.lower()}"
    
    if regime != "base":
        df_eval = df[df["Market_Regime"] == int(regime)]
    else:
        df_eval = df
        
    best_th = 0.99  
    best_pf_shrunk = 0.0
    
    for th in np.arange(0.50, 0.90, 0.05):
        trades = df_eval[df_eval["pred_proba"] >= th]
        num_trades = len(trades)
        
        if num_trades < min_trades:
            continue 
            
        gains = trades.loc[trades[ret_col] > 0, ret_col].sum()
        losses = -trades.loc[trades[ret_col] < 0, ret_col].sum()
        
        raw_pf = float(gains / losses) if losses > 0 else 999.0
        
        weight_empirical = num_trades / (num_trades + shrinkage_k)
        weight_prior = shrinkage_k / (num_trades + shrinkage_k)
        shrunk_pf = (weight_empirical * raw_pf) + (weight_prior * prior_pf)
        
        if shrunk_pf >= target_pf:
            return round(th, 2), round(shrunk_pf, 3), num_trades, round(raw_pf, 3)
            
    return best_th, best_pf_shrunk, 0, 0.0

# ========================================================
# 🚀 3. CHẠY ĐỘNG CƠ CHO CẢ 2 PHE
# ========================================================
v8_calibrations = {}
print(f"{'REGIME':<8} | {'PHE':<6} | {'THRESHOLD':<10} | {'SHRUNK PF':<10} | {'CI LOWER':<10} | {'SỐ LỆNH'}")
print("-" * 75)

current_time_utc = datetime.utcnow().isoformat()

for r_key in ["base", "0", "1", "2"]:
    
    # --- XỬ LÝ LONG ---
    th_long, pf_shrunk_l, trades_l, pf_raw_l = optimize_threshold_bayesian(
        oos_long_df, "LONG", r_key, min_trades=15, target_pf=1.05, prior_pf=0.95, shrinkage_k=40
    )
    
    # Lọc Data để chạy Bootstrap Long
    df_eval_l = oos_long_df if r_key == "base" else oos_long_df[oos_long_df["Market_Regime"] == int(r_key)]
    if th_long < 0.99 and trades_l > 0:
        real_returns_l = df_eval_l.loc[df_eval_l['pred_proba'] >= th_long, 'realized_net_return_long']
        ci_lower_l = round(calculate_bootstrap_pf_lower(real_returns_l), 3)
    else:
        ci_lower_l = 0.0
        
    sample_size_l = len(df_eval_l)
    
    # --- XỬ LÝ SHORT ---
    th_short, pf_shrunk_s, trades_s, pf_raw_s = optimize_threshold_bayesian(
        oos_short_df, "SHORT", r_key, min_trades=15, target_pf=1.05, prior_pf=0.95, shrinkage_k=40
    )
    
    # Lọc Data để chạy Bootstrap Short
    df_eval_s = oos_short_df if r_key == "base" else oos_short_df[oos_short_df["Market_Regime"] == int(r_key)]
    if th_short < 0.99 and trades_s > 0:
        real_returns_s = df_eval_s.loc[df_eval_s['pred_proba'] >= th_short, 'realized_net_return_short']
        ci_lower_s = round(calculate_bootstrap_pf_lower(real_returns_s), 3)
    else:
        ci_lower_s = 0.0
        
    sample_size_s = len(df_eval_s)
    
    # --- 🛡️ LẮP RÁP SCHEMA CONTRACT V2 ---
    v8_calibrations[r_key] = {
        "LONG": {
            "A": 1.0, "B": 0.0, 
            "Threshold": th_long,
            "PF_shrunk": pf_shrunk_l,
            "PF_raw": pf_raw_l,
            "min_trades": trades_l,
            "bootstrap_ci_lower": ci_lower_l, # Đã được tính toán thực tế!
            "sample_size": sample_size_l,
            "last_train_time": current_time_utc
        },
        "SHORT": {
            "A": 1.0, "B": 0.0, 
            "Threshold": th_short,
            "PF_shrunk": pf_shrunk_s,
            "PF_raw": pf_raw_s,
            "min_trades": trades_s,
            "bootstrap_ci_lower": ci_lower_s, # Đã được tính toán thực tế!
            "sample_size": sample_size_s,
            "last_train_time": current_time_utc
        }
    }
    
    # In báo cáo
    th_str_l = str(th_long) if th_long < 0.99 else '🚫 BLOCK'
    th_str_s = str(th_short) if th_short < 0.99 else '🚫 BLOCK'
    print(f"Reg. {r_key:<3} | LONG   | {th_str_l:<10} | {pf_shrunk_l:<10} | {ci_lower_l:<10} | {trades_l}")
    print(f"Reg. {r_key:<3} | SHORT  | {th_str_s:<10} | {pf_shrunk_s:<10} | {ci_lower_s:<10} | {trades_s}")

# Lưu cấu hình xịn
with open("model_calibrations.json", "w") as f:
    json.dump(v8_calibrations, f, indent=4)

print("-" * 75)
print("✅ TẠO CALIBRATION BAYESIAN & BOOTSTRAP (CHUẨN V2) THÀNH CÔNG!")


🎯 ĐÚC CALIBRATION & TỐI ƯU NGƯỠNG (TÍCH HỢP BAYESIAN & BOOTSTRAP) - SCHEMA V2
REGIME   | PHE    | THRESHOLD  | SHRUNK PF  | CI LOWER   | SỐ LỆNH
---------------------------------------------------------------------------
Reg. base | LONG   | 0.75       | 1.128      | 0.972      | 440
Reg. base | SHORT  | 🚫 BLOCK    | 0.0        | 0.0        | 0
Reg. 0   | LONG   | 0.75       | 1.128      | 0.975      | 440
Reg. 0   | SHORT  | 🚫 BLOCK    | 0.0        | 0.0        | 0
Reg. 1   | LONG   | 🚫 BLOCK    | 0.0        | 0.0        | 0
Reg. 1   | SHORT  | 🚫 BLOCK    | 0.0        | 0.0        | 0
Reg. 2   | LONG   | 🚫 BLOCK    | 0.0        | 0.0        | 0
Reg. 2   | SHORT  | 🚫 BLOCK    | 0.0        | 0.0        | 0
---------------------------------------------------------------------------
✅ TẠO CALIBRATION BAYESIAN & BOOTSTRAP (CHUẨN V2) THÀNH CÔNG!


In [9]:
# 🧠 HUẤN LUYỆN GATING NETWORK: OOS PERFORMANCE LABELING (EM ALGORITHM)
from sklearn.metrics import brier_score_loss
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
import joblib

print("\n" + "="*50)
print("🔄 KÍCH HOẠT THUẬT TOÁN EM: TẠO NHÃN REGIME TỪ OOS PERFORMANCE")
print("="*50)

gating_features = [
    "Volatility_24", "Volatility_Regime", "Trend_Strength", 
    "ATR_Pct", "ATR_Regime", "Volume_Regime", "RSI_14_Norm", "MACD_Gap"
]

# BƯỚC 1: KHỞI TẠO NHÃN TẠM (EXPECTATION) BẰNG QUANTILE
print("1. Đang khởi tạo mồi (Heuristic Draft) để chia tách dữ liệu...")
trend_th = mega_df['Trend_Strength'].quantile(0.80)
sleep_th = mega_df['Trend_Strength'].quantile(0.30)
median_vol = mega_df["Volatility_24"].median()
cond = [
    (mega_df['Trend_Strength'] >= trend_th),
    (mega_df['Trend_Strength'] < sleep_th) & (mega_df['Volatility_24'] < median_vol)
]
mega_df['Regime_Draft'] = np.select(cond, [2, 0], default=1)

# BƯỚC 2: RÈN 3 CHUYÊN GIA NHÁP (BASE EXPERTS)
print("2. Đang rèn 3 Chuyên gia Base (Tốc độ cao)...")
X_all = mega_df[feature_columns_v8].astype(np.float32)
y_all = mega_df['target_long'].astype(int) # Dùng chiều LONG làm la bàn định tuyến
experts_draft = {}
for r in [0, 1, 2]:
    idx = mega_df['Regime_Draft'] == r
    # Dùng model nhẹ (50 trees) để tiết kiệm thời gian ở vòng nháp
    model = XGBClassifier(n_estimators=50, max_depth=4, learning_rate=0.1, tree_method='hist')
    if idx.sum() > 50: # Đảm bảo có đủ data
        model.fit(X_all[idx], y_all[idx], verbose=False)
        experts_draft[r] = model

# BƯỚC 3: THI ĐẤU OOS VÀ TÍNH SAI SỐ LĂN (ROLLING PERFORMANCE)
print("3. ⚔️ Cho 3 Chuyên gia thi đấu trên toàn bộ dữ liệu lịch sử...")
probs = pd.DataFrame()
for r in [0, 1, 2]:
    if r in experts_draft:
        probs[f'exp_{r}'] = experts_draft[r].predict_proba(X_all)[:, 1]
    else:
        probs[f'exp_{r}'] = 0.5 
loss_df = pd.DataFrame()
for r in [0, 1, 2]:
    loss_df[f'loss_{r}'] = (probs[f'exp_{r}'] - y_all)**2
WINDOW = 24
smooth_loss = loss_df.rolling(window=WINDOW, min_periods=1).mean()

# BƯỚC 4: TẠO NHÃN GATING CHUẨN (MAXIMIZATION)
best_expert_cols = smooth_loss.idxmin(axis=1)
mega_df['Market_Regime'] = best_expert_cols.str.replace('loss_', '').astype(int)
print("\n🗺️ BẢN ĐỒ LÃNH THỔ REGIME (Dựa trên Performance Thực Tế OOS):")
for regime, pct in mega_df['Market_Regime'].value_counts(normalize=True).items():
    print(f" - Regime {regime}: {pct*100:.1f}% lãnh thổ")

# BƯỚC 5: ĐÚC GATING NETWORK TRÊN NHÃN OOS XỊN
print("\n4. 🧠 Đang huấn luyện Gating Network trên bộ nhãn Performance...")
X_gate = mega_df[gating_features].astype(np.float32)
y_gate = mega_df["Market_Regime"].astype(int)
gating_model = XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.05,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42
)
gating_model.fit(X_gate, y_gate, verbose=False)

joblib.dump(gating_model, "xgb_v8_gating_network.pkl")
joblib.dump(gating_features, "xgb_v8_gating_features.pkl")
print("✅ TUYỆT KỸ HOÀN TẤT! Gating Network giờ đã sở hữu 'Nhãn thuật Performance'!")


🔄 KÍCH HOẠT THUẬT TOÁN EM: TẠO NHÃN REGIME TỪ OOS PERFORMANCE
1. Đang khởi tạo mồi (Heuristic Draft) để chia tách dữ liệu...
2. Đang rèn 3 Chuyên gia Base (Tốc độ cao)...
3. ⚔️ Cho 3 Chuyên gia thi đấu trên toàn bộ dữ liệu lịch sử...

🗺️ BẢN ĐỒ LÃNH THỔ REGIME (Dựa trên Performance Thực Tế OOS):
 - Regime 1: 34.7% lãnh thổ
 - Regime 0: 33.4% lãnh thổ
 - Regime 2: 31.9% lãnh thổ

4. 🧠 Đang huấn luyện Gating Network trên bộ nhãn Performance...
✅ TUYỆT KỸ HOÀN TẤT! Gating Network giờ đã sở hữu 'Nhãn thuật Performance'!


In [10]:
from xgboost import XGBRegressor
import joblib
import numpy as np
import pandas as pd

print("\n" + "="*50)
print("🎯 HUẤN LUYỆN AI DỰ BÁO PHÂN PHỐI LỢI NHUẬN (QUANTILE REGRESSION)")
print("="*50)

# ========================================================
# 🛡️ BƯỚC 1: TẠO TARGET FORWARD RETURN (CHỐNG CRASH)
# ========================================================
print("🔄 Đang tính toán Lợi nhuận tương lai (Forward Return 20)...")
symbol_col = None
assert 'symbol' in mega_df.columns, "🚨 BÁO ĐỘNG: Mất cột 'symbol' trong mega_df!"
mega_df['Forward_Return_20'] = mega_df.groupby('symbol')['Close'].transform(lambda x: x.shift(-20) / x - 1)

# Lọc bỏ các dòng bị NaN ở cuối tập dữ liệu
train_quant_df = mega_df.dropna(subset=['Forward_Return_20']).copy()

# ========================================================
# 🧠 BƯỚC 2: RÈN CHUYÊN GIA EXIT (ĐỒNG BỘ FEATURE V8)
# ========================================================
X_quant = train_quant_df[feature_columns_v8].astype(np.float32)
y_quant = train_quant_df['Forward_Return_20'].astype(np.float32)

# Khóa biên độ ảo giác
y_quant = np.clip(y_quant, -0.30, 0.30)

quantiles = [0.10, 0.50, 0.90]
quantile_models = {}

for q in quantiles:
    print(f"⏳ Đang rèn AI Quantile {q*100:.0f}%...")
    model = XGBRegressor(
        objective='reg:quantileerror',
        quantile_alpha=q,
        n_estimators=150,
        max_depth=5,
        learning_rate=0.05,
        tree_method='hist',
        random_state=42
    )
    model.fit(X_quant, y_quant, verbose=False)
    quantile_models[f"q_{q}"] = model

joblib.dump(quantile_models, "xgb_v8_exit_model.pkl")
print("✅ TUYỆT VỜI! Đã đúc xong Bộ 3 AI Quantile. File 'xgb_v8_exit_model.pkl' đã sẵn sàng!")


🎯 HUẤN LUYỆN AI DỰ BÁO PHÂN PHỐI LỢI NHUẬN (QUANTILE REGRESSION)
🔄 Đang tính toán Lợi nhuận tương lai (Forward Return 20)...
⏳ Đang rèn AI Quantile 10%...
⏳ Đang rèn AI Quantile 50%...
⏳ Đang rèn AI Quantile 90%...
✅ TUYỆT VỜI! Đã đúc xong Bộ 3 AI Quantile. File 'xgb_v8_exit_model.pkl' đã sẵn sàng!


In [11]:
# ⚖️ 4. CHẤM ĐIỂM BRIER & TÍNH TRỌNG SỐ ENSEMBLE ĐỘNG (PER-REGIME OOS)
from sklearn.metrics import brier_score_loss
import joblib

print("\n" + "="*65)
print("⚖️ ĐANG HỌC TRỌNG SỐ ENSEMBLE TỪ THỰC CHIẾN OOS (HARD-DISABLE DL)")
print("="*65)

ensemble_weights_v8 = {"LONG": {}, "SHORT": {}}
REGIME_NAMES = {0: "SLEEP", 1: "SIDEWAY", 2: "TRENDING"}

for direction, df_oos in [("LONG", oos_long_df), ("SHORT", oos_short_df)]:
    target_col = f"target_{direction.lower()}"
    print(f"\n--- PHÂN TÍCH TRỌNG SỐ PHE {direction} ---")
    
    for regime in [0, 1, 2]:
        regime_df = df_oos[df_oos['Market_Regime'] == regime]
        
        # Nếu Regime thiếu data OOS trầm trọng -> Đóng băng, giao 100% cho XGBoost
        if len(regime_df) < 50:
            print(f" 🔹 {REGIME_NAMES[regime]:<8}: Thiếu data OOS -> Hard-disable DL, 100% XGB.")
            ensemble_weights_v8[direction][str(regime)] = {"xgb_weight": 1.0, "dl_weight": 0.0} 
            continue
            
        y_true = regime_df[target_col].astype(int)
        y_pred_xgb = regime_df["pred_proba"]
        brier_xgb = brier_score_loss(y_true, y_pred_xgb)      
        
        # 🛡️ KIỂM TRA OOS DL THẬT (HARD-DISABLE NẾU KHÔNG CÓ)
        if "pred_proba_dl" in regime_df.columns and regime_df["pred_proba_dl"].notna().any():
            brier_dl = brier_score_loss(y_true, regime_df["pred_proba_dl"])
            
            # Thuật toán Nghịch đảo Sai số (Inverse-Variance) khi có đủ 2 phe
            w_xgb = (1.0 / (brier_xgb + 1e-6)) / ((1.0 / (brier_xgb + 1e-6)) + (1.0 / (brier_dl + 1e-6)))
            
            # Cầu dao an toàn: Không cho phép 1 phe cầm quyền tuyệt đối nếu cả 2 đều có edge thật
            w_xgb = max(0.20, min(w_xgb, 0.80))
            w_dl = 1.0 - w_xgb
            print(f" 🎯 {REGIME_NAMES[regime]:<8} | Brier XGB: {brier_xgb:.4f} vs DL: {brier_dl:.4f} -> Tỷ trọng: XGB {w_xgb*100:.1f}% | DL {w_dl*100:.1f}%")
        else:
            # 🛑 TRẢM DL: Khi chưa có dữ liệu OOS thật, DL bị tước 100% quyền biểu quyết
            w_xgb = 1.0
            w_dl = 0.0
            print(f" ⚠️ {REGIME_NAMES[regime]:<8} | Brier XGB: {brier_xgb:.4f} | Không có OOS DL -> Hard-disable DL (XGB 100%)")
        
        # Lưu Schema chuẩn hóa
        ensemble_weights_v8[direction][str(regime)] = {"xgb_weight": w_xgb, "dl_weight": w_dl}

# Gán Base/Mặc định bằng Regime 1 (Sideway) làm lá chắn phòng hờ
ensemble_weights_v8["LONG"]["base"] = ensemble_weights_v8["LONG"]["1"]
ensemble_weights_v8["SHORT"]["base"] = ensemble_weights_v8["SHORT"]["1"]

joblib.dump(ensemble_weights_v8, "ensemble_weights_v8.pkl")
print("\n✅ ĐÃ VÁ XONG SCHEMA ENSEMBLE! Quyền lực giờ đây được phân bổ sòng phẳng bằng thực lực OOS.")


⚖️ ĐANG HỌC TRỌNG SỐ ENSEMBLE TỪ THỰC CHIẾN OOS (HARD-DISABLE DL)

--- PHÂN TÍCH TRỌNG SỐ PHE LONG ---
 ⚠️ SLEEP    | Brier XGB: 0.2700 | Không có OOS DL -> Hard-disable DL (XGB 100%)
 🔹 SIDEWAY : Thiếu data OOS -> Hard-disable DL, 100% XGB.
 🔹 TRENDING: Thiếu data OOS -> Hard-disable DL, 100% XGB.

--- PHÂN TÍCH TRỌNG SỐ PHE SHORT ---
 ⚠️ SLEEP    | Brier XGB: 0.2626 | Không có OOS DL -> Hard-disable DL (XGB 100%)
 🔹 SIDEWAY : Thiếu data OOS -> Hard-disable DL, 100% XGB.
 🔹 TRENDING: Thiếu data OOS -> Hard-disable DL, 100% XGB.

✅ ĐÃ VÁ XONG SCHEMA ENSEMBLE! Quyền lực giờ đây được phân bổ sòng phẳng bằng thực lực OOS.


In [15]:
import json
import hashlib
from datetime import datetime

print("\n" + "="*60)
print("📦 ĐÓNG GÓI ARTIFACT MANIFEST (CHUẨN MLOPS PORTABLE)")
print("="*60)

# Tạo Hash cho Features để Bot Live kiểm tra tính toàn vẹn
feature_string = "".join(feature_columns_v8)
feat_hash = hashlib.md5(feature_string.encode('utf-8')).hexdigest()

# 🚨 CHỈ DÙNG TÊN FILE TRỰC TIẾP (RELATIVE PATH), KHÔNG DÙNG ĐƯỜNG DẪN DÀI
manifest = {
    "train_time_utc": datetime.utcnow().isoformat(),
    "feature_hash": feat_hash,
    "expert_models_file": "expert_models_v8.pkl",       # Thêm file Não bộ Tầng 1
    "ensemble_weights_file": "ensemble_weights_v8.pkl", 
    "meta_model_file": "xgb_v8_meta_model.pkl",         # File Tầng 2
    "meta_features_file": "xgb_v8_meta_features.pkl",   # Cột Tầng 2
    "calibration_file": "model_calibrations.json",      # File Gates & Bootstrap
    "required_features_count": len(feature_columns_v8),
    "author": "Quant_Director"
}

with open("artifact_manifest.json", "w") as f:
    json.dump(manifest, f, indent=4)

print("✅ Đã xuất xưởng artifact_manifest.json với Relative Paths!")


📦 ĐÓNG GÓI ARTIFACT MANIFEST (CHUẨN MLOPS PORTABLE)
✅ Đã xuất xưởng artifact_manifest.json với Relative Paths!


### KHỞI CHẠY BOT

In [ ]:
import traceback
import time
import schedule
import pandas as pd
import numpy as np
import csv

# ========================================================
# 🧠 KHỞI TẠO BỘ NHỚ LƯU TRỮ (ORDERBOOK MEMORY & CẤU HÌNH)
# ========================================================
memory = {}  
ORDER_BOOK_DEPTH = 20  

# ========================================================
# 🤖 LÕI THỰC THI CHÍNH CỦA BOT (CHẠY 1 LẦN MỖI CHU KỲ)
# ========================================================
def run_cross_sectional_bot():
    try:
        from datetime import datetime
        utc_now_str = datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')
    except:
        utc_now_str = "NOW"
        
    print(f"\n[{utc_now_str}] 🌍 KHỞI ĐỘNG PHA 1: QUÉT ALPHA TOÀN THỊ TRƯỜNG...")
    scan_results = []
    
    # ---------------------------------------------------------
    # 🛒 PHA 1: ĐI CHỢ KHẢO GIÁ & GHI SỔ V2
    # ---------------------------------------------------------
    for sym in TARGET_SYMBOLS:
        try:
            print(f"  -> Đang cào & nhào nặn features cho {sym}...")
            
            # 1. TẢI DATA VÀ NẶN FEATURE
            klines = client.futures_historical_klines(symbol=sym, interval='1h', start_str="10 days ago UTC")
            if len(klines) < 2: continue
            
            df_raw = pd.DataFrame(klines, columns=[
                "Open time", "Open", "High", "Low", "Close", "Volume", "Close time", 
                "Quote Asset", "Trades", "Taker Buy Base", "Taker Buy Quote", "Ignore"
            ])
            df_feat = QuantFeatureEngineer.run_pipeline(df_raw, asset_sentiment=0.0, symbol=sym)
            df_feat['symbol'] = sym
            df_feat = df_feat.dropna().reset_index(drop=True)
            
            if len(df_feat) == 0: continue
                
            live_row = df_feat.iloc[-1].to_dict()
            df_features = df_feat.copy()
            market_regime = int(live_row.get("Market_Regime", 0))

            # 2. TẦNG 1: DỰ BÁO XGBOOST
            w_sleep = 1.0 if market_regime == 0 else 0.0
            w_sideway = 1.0 if market_regime == 1 else 0.0
            w_trend = 1.0 if market_regime == 2 else 0.0
            
            exp_0 = expert_models.get(0) or expert_models.get("0")
            exp_1 = expert_models.get(1) or expert_models.get("1")
            exp_2 = expert_models.get(2) or expert_models.get("2")
            
            p_l_0 = hybrid_ensemble_predict(exp_0["long"], dl_model_long, df_features, df_features, 0, "LONG")[0] if exp_0 else 0.0
            p_l_1 = hybrid_ensemble_predict(exp_1["long"], dl_model_long, df_features, df_features, 1, "LONG")[0] if exp_1 else 0.0
            p_l_2 = hybrid_ensemble_predict(exp_2["long"], dl_model_long, df_features, df_features, 2, "LONG")[0] if exp_2 else 0.0

            p_s_0 = hybrid_ensemble_predict(exp_0["short"], dl_model_short, df_features, df_features, 0, "SHORT")[0] if exp_0 else 0.0
            p_s_1 = hybrid_ensemble_predict(exp_1["short"], dl_model_short, df_features, df_features, 1, "SHORT")[0] if exp_1 else 0.0
            p_s_2 = hybrid_ensemble_predict(exp_2["short"], dl_model_short, df_features, df_features, 2, "SHORT")[0] if exp_2 else 0.0

            raw_long_uncal = (w_sleep * p_l_0) + (w_sideway * p_l_1) + (w_trend * p_l_2)
            raw_short_uncal = (w_sleep * p_s_0) + (w_sideway * p_s_1) + (w_trend * p_s_2)
            
            # 3. GÓI PROBA VÀ MANIFEST
            p_l_all = [p_l_0, p_l_1, p_l_2]
            p_s_all = [p_s_0, p_s_1, p_s_2]
            
            try:
                model_ver = manifest.get("train_time_utc", "unknown_ver")
                feat_hash = manifest.get("feature_hash", "unknown_hash")
            except Exception:
                model_ver, feat_hash = "unknown_ver", "unknown_hash"
            
            calibs = LIVE_CALIBRATION.get(str(market_regime), LIVE_CALIBRATION.get("base"))
            long_calib = calibs["LONG"]
            short_calib = calibs["SHORT"]
            
            calib_proba_l = calibrate_xgboost_probability(raw_long_uncal, long_calib["A"], long_calib["B"])
            calib_proba_s = calibrate_xgboost_probability(raw_short_uncal, short_calib["A"], short_calib["B"])

            # =======================================================
            # 4. KỶ LUẬT THÉP TẦNG 1 (GATES CẬP NHẬT STRICTION)
            # =======================================================
            # Yêu cầu: PF_shrunk > 1.05, bootstrap_ci_lower > 1.0, min_trades >= 50, proba >= Thresh
            
            # --- KIỂM CHỨNG LONG ---
            status_l, reject_stage_l, reason_l = "PASSED_T1", "NONE", ""
            if long_calib["min_trades"] < 50:
                status_l, reject_stage_l, reason_l = "REJECTED", "T1_NODATA", f"Trades ({long_calib['min_trades']}) < 50"
            elif long_calib["bootstrap_ci_lower"] <= 1.0:
                status_l, reject_stage_l, reason_l = "REJECTED", "T1_BOOTSTRAP", f"CI_Lower ({long_calib['bootstrap_ci_lower']:.2f}) <= 1.0"
            elif long_calib["PF_shrunk"] <= 1.05:
                status_l, reject_stage_l, reason_l = "REJECTED", "T1_PF", f"Low PF ({long_calib['PF_shrunk']:.2f})"
            elif calib_proba_l < long_calib["Threshold"]:
                status_l, reject_stage_l, reason_l = "REJECTED", "T1_PROBA", f"Proba < Thresh ({calib_proba_l:.3f})"
            
            # --- KIỂM CHỨNG SHORT ---
            status_s, reject_stage_s, reason_s = "PASSED_T1", "NONE", ""
            if short_calib["min_trades"] < 50:
                status_s, reject_stage_s, reason_s = "REJECTED", "T1_NODATA", f"Trades ({short_calib['min_trades']}) < 50"
            elif short_calib["bootstrap_ci_lower"] <= 1.0:
                status_s, reject_stage_s, reason_s = "REJECTED", "T1_BOOTSTRAP", f"CI_Lower ({short_calib['bootstrap_ci_lower']:.2f}) <= 1.0"
            elif short_calib["PF_shrunk"] <= 1.05:
                status_s, reject_stage_s, reason_s = "REJECTED", "T1_PF", f"Low PF ({short_calib['PF_shrunk']:.2f})"
            elif calib_proba_s < short_calib["Threshold"]:
                status_s, reject_stage_s, reason_s = "REJECTED", "T1_PROBA", f"Proba < Thresh ({calib_proba_s:.3f})"

            # =======================================================
            # 5. TẦNG 2: NẶN META-FEATURES VÀ DỰ BÁO LỢI NHUẬN (EV)
            # =======================================================
            ev_adj_l, unc_l = 0.0, 0.0
            ev_adj_s, unc_s = 0.0, 0.0
            
            # (Đảm bảo Giám đốc đã nạp ev_model và uncertainty_model ở Boot Sequence nhé)
            
            # Nếu LONG lọt qua Tầng 1 -> Kích hoạt AI Vệ Sĩ
            if status_l == "PASSED_T1" and 'ev_model' in globals():
                # Nặn đặc trưng Tầng 2 theo đúng chuẩn
                meta_input_l = {
                    "pred_proba": calib_proba_l,
                    "Market_Regime": market_regime,
                    "Taker_Imbalance": df_features.iloc[-1].get("Taker_Imbalance", 0.0),
                    "Sentiment_Score": df_features.iloc[-1].get("Sentiment_Score", 0.5),
                    "Volatility_24": df_features.iloc[-1].get("Volatility_24", 0.02),
                    "Volume_Z": df_features.iloc[-1].get("Volume_Z", 0.0),
                    "Trend_Strength": df_features.iloc[-1].get("Trend_Strength", 0.0)
                }
                # (Biến meta_feature_cols phải được nạp từ file xgb_v8_meta_features.pkl)
                df_meta_l = pd.DataFrame([meta_input_l])[meta_feature_cols]
                
                raw_ev_l = ev_model.predict(df_meta_l)[0]
                unc_l = uncertainty_model.predict(df_meta_l)[0]
                ev_adj_l = raw_ev_l - (0.5 * unc_l) # Trừ hao rủi ro (Risk-Penalty)
                
                if ev_adj_l > 0.0:
                    status_l = "APPROVED_T2"
                    reason_l = "EV Positive"
                else:
                    status_l = "REJECTED"
                    reject_stage_l = "T2_EV"
                    reason_l = f"Negative EV ({ev_adj_l:.4f})"

            # Nếu SHORT lọt qua Tầng 1 -> Kích hoạt AI Vệ Sĩ
            if status_s == "PASSED_T1" and 'ev_model' in globals():
                meta_input_s = {
                    "pred_proba": calib_proba_s,
                    "Market_Regime": market_regime,
                    "Taker_Imbalance": df_features.iloc[-1].get("Taker_Imbalance", 0.0),
                    "Sentiment_Score": df_features.iloc[-1].get("Sentiment_Score", 0.5),
                    "Volatility_24": df_features.iloc[-1].get("Volatility_24", 0.02),
                    "Volume_Z": df_features.iloc[-1].get("Volume_Z", 0.0),
                    "Trend_Strength": df_features.iloc[-1].get("Trend_Strength", 0.0)
                }
                df_meta_s = pd.DataFrame([meta_input_s])[meta_feature_cols]
                
                raw_ev_s = ev_model.predict(df_meta_s)[0]
                unc_s = uncertainty_model.predict(df_meta_s)[0]
                ev_adj_s = raw_ev_s - (0.5 * unc_s)
                
                if ev_adj_s > 0.0:
                    status_s = "APPROVED_T2"
                    reason_s = "EV Positive"
                else:
                    status_s = "REJECTED"
                    reject_stage_s = "T2_EV"
                    reason_s = f"Negative EV ({ev_adj_s:.4f})"

            # =======================================================
            # 6. GHI SỔ CÁI BÓNG ĐÊM V2 ĐÃ CÓ EV
            # =======================================================
            ShadowLedger.log_candidate(
                symbol=sym, side="LONG", regime=market_regime, 
                raw_proba=raw_long_uncal, calib_proba=calib_proba_l, threshold=long_calib["Threshold"],
                p_l_all=p_l_all, p_s_all=p_s_all, 
                ev=ev_adj_l, unc=unc_l, 
                status=status_l, reject_stage=reject_stage_l, reject_reason=reason_l,
                model_ver=model_ver, feat_hash=feat_hash
            )
            
            ShadowLedger.log_candidate(
                symbol=sym, side="SHORT", regime=market_regime, 
                raw_proba=raw_short_uncal, calib_proba=calib_proba_s, threshold=short_calib["Threshold"],
                p_l_all=p_l_all, p_s_all=p_s_all, 
                ev=ev_adj_s, unc=unc_s, 
                status=status_s, reject_stage=reject_stage_s, reject_reason=reason_s,
                model_ver=model_ver, feat_hash=feat_hash
            )

        except Exception as e:
            print(f" ⚠️ Lỗi quét {sym}: {e}")
            continue

    # ---------------------------------------------------------
    # ⚔️ PHA 2: TẠM TẮT TRONG SHADOW MODE ĐỂ THU DATA NHANH
    # ---------------------------------------------------------
    print(f"\n[{utc_now_str}] ⚖️ Đã lưu toàn bộ nhận định vào Shadow Ledger V2.")
    print(f"[{utc_now_str}] ✅ Đã xong 1 nhịp quét Cross-Sectional. Bot nghỉ ngơi chờ chu kỳ tới!")

# ========================================================
# 🚀 CÔNG TẮC KHỞI ĐỘNG HỆ THỐNG (SHADOW MODE V2)
# ========================================================
if __name__ == "__main__":
    print("\n" + "="*65)
    print("🚀 HỆ ĐIỀU HÀNH: CROSS-SECTIONAL ALPHA (SHADOW MODE) KÍCH HOẠT!")
    print("="*65)
    
    # 1. Chạy True E2E Smoke Test (Bản V2 Tối thượng)
    is_safe = run_true_e2e_smoke_test()

    if not is_safe:
        print("🧯 Bot đã bị vô hiệu hóa an toàn do không vượt qua True E2E Smoke Test.")
        import sys
        sys.exit("Dừng chạy code. Vui lòng check log lỗi bên trên.")
    else:
        # 2. Phát súng quét thị trường đầu tiên ngay khi bật
        print("\n⚡ KÍCH HOẠT QUÉT THỊ TRƯỜNG LẦN ĐẦU TIÊN...")
        run_cross_sectional_bot() 
    
    # 3. Lập lịch tự động quét (Chỉ gọi hàm Bot V2)
    schedule.every().hour.at(":00").do(run_cross_sectional_bot)
    
    # Lập lịch Lò Rèn tự động luyện lại não bộ (Mỗi ngày 1 lần vào 01:00 AM)
    schedule.every().day.at("01:00").do(run_batch_retrain_cycle)
    
    print("\n💤 Hệ thống đang vào trạng thái chờ lịch trình. Bấm Ctrl+C để tắt an toàn.")
    while True:
        try:
            schedule.run_pending()
            time.sleep(1) 
        except KeyboardInterrupt:
            print("\n🛑 GIÁM ĐỐC ĐÃ RA LỆNH DỪNG BOT! Cỗ máy tắt an toàn.")
            break
        except Exception as e:
            print(f"\n❌ LỖI HỆ TRỌNG TRONG VÒNG LẶP AUTO-PILOT: {e}\n{traceback.format_exc()}")
            print("🔄 Sẽ thử lại vòng lặp sau 10 giây...")
            time.sleep(10)


🚀 HỆ ĐIỀU HÀNH: CROSS-SECTIONAL ALPHA (SHADOW MODE) KÍCH HOẠT!

🔥 KHỞI ĐỘNG TRUE E2E SMOKE TEST: KIỂM TOÁN TOÀN DIỆN MẠCH MÁU
  -> [1/6] Đang khởi tạo Mock Data...
  -> [2/6] Đang ép khuôn SchemaContract...
  -> [2.5/6] Đang kiểm toán Live Inference Contract (Dự báo thực tế)...
     ✅ Vượt qua Live Inference! Models hợp lệ, Features khớp 100%.
  -> [3/6] Đang đọc cấu hình Ensemble & Calibration...
  -> [4/6] Đang mô phỏng nặn Data Tầng 2...
  -> [5/6] Đang test Sổ cái Shadow Ledger V2...

✅ [SMOKE TEST PASSED] Mạch máu thông suốt. Não bộ XGBoost và Sổ cái V2 hoạt động 100%!

⚡ KÍCH HOẠT QUÉT THỊ TRƯỜNG LẦN ĐẦU TIÊN...

[2026-05-07 07:52:08] 🌍 KHỞI ĐỘNG PHA 1: QUÉT ALPHA TOÀN THỊ TRƯỜNG...
  -> Đang cào & nhào nặn features cho BTCUSDT...
  -> Đang cào & nhào nặn features cho ETHUSDT...
  -> Đang cào & nhào nặn features cho SOLUSDT...
  -> Đang cào & nhào nặn features cho BNBUSDT...
  -> Đang cào & nhào nặn features cho XRPUSDT...
  -> Đang cào & nhào nặn features cho DOGEUSDT...
  -> Đ